# Beyond 'Stressed' or 'Not': Wearable-Derived Stress-Physiology Phenotypes

Companion notebook reproducing the four-step analysis reported in the manuscript:

1. Trait-level stress–physiology associations (Kruskal–Wallis)
2. Unsupervised cohort-level phenotyping (PCA + k-means)
3. Linear mixed-effects modeling of within-person associations
4. Dynamic stress–physiology association modeling (PCA + GMM + UMAP)

Set `INPUT_DIR` in the setup cell to point to your local copies.


## Setup


In [ ]:
# Core
import time
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

# Statistics
import scipy.stats as stats
from scipy.stats import circmean, circstd
import statsmodels.formula.api as smf

# ML
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, silhouette_score, silhouette_samples

# Visualization
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.ticker as ticker
from matplotlib.ticker import MaxNLocator, MultipleLocator
from matplotlib.patches import Wedge, FancyBboxPatch, Patch
from matplotlib.colors import ListedColormap, BoundaryNorm
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
from matplotlib.patches import Patch

import warnings
warnings.filterwarnings('ignore')

# Google Colab authentication and Drive mounting
from google.colab import auth, drive
from google.colab import files

auth.authenticate_user()
drive.mount('/content/drive/')

# ----------------------------------------------------------------------
# Paths -- point INPUT_DIR at the directory containing the LifeSnaps
# parquet/CSV inputs (see Methods); OUTPUT_DIR is where figures are saved.
# ----------------------------------------------------------------------
INPUT_DIR = '/data/lifesnaps/processed/PostProcess/PostDataset'

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## Section 1 — Trait-level stress–physiology associations

Categorize participants by mean S-STAI and compare wearable-derived metrics with Kruskal–Wallis tests (Methods).


In [ ]:
df_anxiety = pd.read_csv(f'{INPUT_DIR}/stai.csv')
df_anxiety.rename(columns={'user_id': 'Global_Deidentified'}, inplace=True)

In [ ]:
# Build df_anxiety_filtered with category column (used downstream).
df_anxiety_filtered = df_anxiety.dropna(subset=['stai_stress'])

def get_color(val):
    if 20 <= val <= 45:
        return 'green'
    elif 45 < val <= 50:
        return 'yellow'
    else:
        return 'red'

df_anxiety_filtered['category'] = df_anxiety_filtered['stai_stress'].apply(get_color)

In [ ]:
Sleep_df = pd.read_parquet(f'{INPUT_DIR}/Sleep_features.parquet')
df_SRI = pd.read_csv(f'{INPUT_DIR}/SRI_focused.csv')
df_demo = pd.read_parquet(f'{INPUT_DIR}/fitbit_Profile.parquet')
VO2Max_df = pd.read_parquet(f'{INPUT_DIR}/VO2Max.parquet')
BR_df = pd.read_parquet(f'{INPUT_DIR}/BR.parquet')
SpO2_df = pd.read_parquet(f'{INPUT_DIR}/SpO2.parquet')
Temp_df = pd.read_parquet(f'{INPUT_DIR}/Temperature.parquet')
Steps_df = pd.read_parquet(f'{INPUT_DIR}/Steps.parquet')
RestingHR_df = pd.read_parquet(f'{INPUT_DIR}/RHR.parquet')
NightHR_Sleep_df = pd.read_parquet(f'{INPUT_DIR}/SleepHR_features.parquet')
HRV_df = pd.read_parquet(f'{INPUT_DIR}/HRV_features.parquet')
Sleep_df["Date"] = pd.to_datetime(Sleep_df["Date"]).dt.strftime('%m-%d-%Y')


In [ ]:
# 1. Data Preparation (same as before)
df_anxiety_min_2_samples = df_anxiety_filtered.groupby('Global_Deidentified').filter(lambda x: len(x) >= 2)
median_scores_df = df_anxiety_min_2_samples.groupby('Global_Deidentified')['stai_stress'].median().reset_index()
median_scores_df = median_scores_df.sort_values('stai_stress').reset_index(drop=True)
median_scores_df.columns = ['Global_Deidentified', 'median_stai_stress']
num_digits = len(str(len(median_scores_df)))
id_mapping = {row['Global_Deidentified']: f'ID_{str(index+1).zfill(num_digits)}' for index, row in median_scores_df.iterrows()}
df_plot = df_anxiety_min_2_samples.copy()
df_plot['ID'] = df_plot['Global_Deidentified'].map(id_mapping)
df_plot['category'] = df_plot['stai_stress'].apply(get_color)
label_mapping = {'green': 'Below Average', 'gold': 'Average', 'red': 'Above Average'}
df_plot['stress_level'] = df_plot['category'].map(label_mapping)

# 2. Setup the plot
new_order = list(id_mapping.values())
fig, ax = plt.subplots(figsize=(14, 6))
new_palette = {'Below Average': 'green', 'Average': 'gold', 'Above Average': 'red'}

# 3. Draw the stripplot
sns.stripplot(data=df_plot, x='ID', y='stai_stress', hue='stress_level', palette=new_palette,
              order=new_order, dodge=False, jitter=True, alpha=0.7, ax=ax)

# 4. Add colored background boxes and annotations (same as before)
green_count = (median_scores_df['median_stai_stress'] <= 45).sum()
gold_count = ((median_scores_df['median_stai_stress'] > 45) & (median_scores_df['median_stai_stress'] <= 50)).sum()
red_count = (median_scores_df['median_stai_stress'] > 50).sum()
green_end_pos = green_count - 0.5
gold_end_pos = green_end_pos + gold_count
ax.axvspan(-0.5, green_end_pos, color='green', alpha=0.1, zorder=-1)
ax.axvspan(green_end_pos, gold_end_pos, color='gold', alpha=0.1, zorder=-1)
ax.axvspan(gold_end_pos, len(new_order) - 0.5, color='red', alpha=0.1, zorder=-1)
y_position = 60
green_mid_pos = (green_count - 1) / 2
gold_mid_pos = green_count + (gold_count - 1) / 2
red_mid_pos = green_count + gold_count + (red_count - 1) / 2
if green_count > 0: ax.text(green_mid_pos, y_position, f"n = {green_count}", ha='center', fontsize=12, fontweight='bold')
if gold_count > 0: ax.text(gold_mid_pos, y_position, f"n = {gold_count}", ha='center', fontsize=12, fontweight='bold')
if red_count > 0: ax.text(red_mid_pos, y_position, f"n = {red_count}", ha='center', fontsize=12, fontweight='bold')

# 5. Final Formatting
ax.axhline(45, color='black', linestyle='--', linewidth=1)
ax.axhline(50, color='black', linestyle='--', linewidth=1)
plt.xticks(rotation=90)
ax.set_xlabel('Participant ID (Ranked by Median Stress)', fontsize=14)
ax.set_ylabel('STAI Stress Score', fontsize=14)
ax.set_title(f'STAI Stress Scores by Participant n={(len(df_plot['Global_Deidentified'].unique()))}', fontsize=16)
ax.set_xlim(-0.5, len(new_order) - 0.5)

first_legend = ax.legend(title='Individual Score', bbox_to_anchor=(0.99, 0.2), loc='lower right')

legend_elements = [
    Patch(facecolor='green', alpha=0.1, edgecolor='black', label='Below Average'),
    Patch(facecolor='gold', alpha=0.1, edgecolor='black', label='Average'),
    Patch(facecolor='red', alpha=0.1, edgecolor='black', label='Above Average')
]

ax.add_artist(first_legend)
ax.legend(handles=legend_elements, title='Group Category by Median', bbox_to_anchor=(0.99, 0), loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# Merge all features

excluded_ids_per_feature = []

vo2max_stats_per_user = VO2Max_df.groupby('Global_Deidentified')['Value'].agg(['mean', 'count']).reset_index()
# (Optional but recommended) Rename the 'Value' column to be more descriptive.
avg_vo2max_per_user = vo2max_stats_per_user.rename(columns={'mean': 'Avg_VO2Max'})
excluded_ids = set(
    avg_vo2max_per_user.loc[
        avg_vo2max_per_user['count'] <= 29, 'Global_Deidentified'
    ]
)
excluded_ids_per_feature.append(excluded_ids)
avg_vo2max_per_user = avg_vo2max_per_user[avg_vo2max_per_user['count'] > 29]

avg_deep_sleep_br_per_user = BR_df.groupby('Global_Deidentified')['deep_sleep_breathing_rate'].agg(['mean', 'count']).reset_index()

avg_deep_sleep_br_per_user = avg_deep_sleep_br_per_user.rename(columns={'mean': 'Avg_deep_sleep_BR'})

excluded_ids = set(
    avg_deep_sleep_br_per_user.loc[
        avg_deep_sleep_br_per_user['count'] <= 29, 'Global_Deidentified'
    ]
)

excluded_ids_per_feature.append(excluded_ids)
avg_deep_sleep_br_per_user = avg_deep_sleep_br_per_user[avg_deep_sleep_br_per_user['count'] > 29]

avg_SpO2_per_user = SpO2_df.groupby('Global_Deidentified')['Value'].agg(['mean', 'count']).reset_index()
avg_SpO2_per_user = avg_SpO2_per_user.rename(columns={'mean': 'Avg_SpO2'})

excluded_ids = set(
    avg_SpO2_per_user.loc[
        avg_SpO2_per_user['count'] <= 29, 'Global_Deidentified'
    ]
)

excluded_ids_per_feature.append(excluded_ids)
avg_SpO2_per_user = avg_SpO2_per_user[avg_SpO2_per_user['count'] > 29]

avg_Temp_per_user = Temp_df.groupby('Global_Deidentified')['nightly_temperature'].agg(['mean', 'count']).reset_index()

avg_Temp_per_user = avg_Temp_per_user.rename(columns={'mean': 'Avg_Temperature'})

excluded_ids = set(
    avg_Temp_per_user.loc[
        avg_Temp_per_user['count'] <= 29, 'Global_Deidentified'
    ]
)

excluded_ids_per_feature.append(excluded_ids)
avg_Temp_per_user = avg_Temp_per_user[avg_Temp_per_user['count'] > 29]

avg_Steps_per_user = Steps_df.groupby('Global_Deidentified')['Total_Number_of_Steps'].agg(['mean', 'count']).reset_index()
avg_Steps_per_user = avg_Steps_per_user.rename(columns={'mean': 'Avg_Steps_per_day'})

excluded_ids = set(
    avg_Steps_per_user.loc[
        avg_Steps_per_user['count'] <= 29, 'Global_Deidentified'
    ]
)

excluded_ids_per_feature.append(excluded_ids)
avg_Steps_per_user = avg_Steps_per_user[avg_Steps_per_user['count'] > 29]

avg_RHR_per_user = RestingHR_df.groupby('Global_Deidentified')['Resting Heart Rate'].agg(['mean', 'count']).reset_index()
avg_RHR_per_user = avg_RHR_per_user.rename(columns={'mean': 'Avg_RHR'})

excluded_ids = set(
    avg_RHR_per_user.loc[
        avg_RHR_per_user['count'] <= 29, 'Global_Deidentified'
    ]
)

excluded_ids_per_feature.append(excluded_ids)
avg_RHR_per_user = avg_RHR_per_user[avg_RHR_per_user['count'] > 29]

avg_deep_sleep_hr_per_user = NightHR_Sleep_df.groupby('Global_Deidentified')['deep_mean_hr'].agg(['mean', 'count']).reset_index()
avg_deep_sleep_hr_per_user = avg_deep_sleep_hr_per_user.rename(columns={'mean': 'Avg_deep_sleep_HR'})

excluded_ids = set(
    avg_deep_sleep_hr_per_user.loc[
        avg_deep_sleep_hr_per_user['count'] <= 29, 'Global_Deidentified'
    ]
)

excluded_ids_per_feature.append(excluded_ids)
avg_deep_sleep_hr_per_user = avg_deep_sleep_hr_per_user[avg_deep_sleep_hr_per_user['count'] > 29]

avg_deep_sleep_hrv_per_user = HRV_df.groupby('Global_Deidentified')['deep_RMSSD_mean'].agg(['mean', 'count']).reset_index()
avg_deep_sleep_hrv_per_user = avg_deep_sleep_hrv_per_user.rename(columns={'mean': 'Avg_deep_sleep_RMSSD'})

excluded_ids = set(
    avg_deep_sleep_hrv_per_user.loc[
        avg_deep_sleep_hrv_per_user['count'] <= 29, 'Global_Deidentified'
    ]
)

excluded_ids_per_feature.append(excluded_ids)
avg_deep_sleep_hrv_per_user = avg_deep_sleep_hrv_per_user[avg_deep_sleep_hrv_per_user['count'] > 29]

avg_deep_sleep_lf_hf_per_user = HRV_df.groupby('Global_Deidentified')['deep_LF_HF_Ratio_mean'].agg(['mean', 'count']).reset_index()
avg_deep_sleep_lf_hf_per_user = avg_deep_sleep_lf_hf_per_user.rename(columns={'mean': 'Avg_deep_sleep_LF/HF'})

excluded_ids = set(
    avg_deep_sleep_lf_hf_per_user.loc[
        avg_deep_sleep_lf_hf_per_user['count'] <= 29, 'Global_Deidentified'
    ]
)

excluded_ids_per_feature.append(excluded_ids)
avg_deep_sleep_lf_hf_per_user = avg_deep_sleep_lf_hf_per_user[avg_deep_sleep_lf_hf_per_user['count'] > 29]

avg_sleep_per_user = Sleep_df.groupby('Global_Deidentified').agg({
    'Time_asleep': ['mean', 'count'],
    'deep_sleep': 'mean',
    'rem_sleep': 'mean',
    'light_sleep': 'mean',
    'Awake_count': 'mean'
}).reset_index()


avg_sleep_per_user.columns = ['_'.join(col).strip('_') for col in avg_sleep_per_user.columns.values]
excluded_ids = set(
    avg_sleep_per_user.loc[
        avg_sleep_per_user['Time_asleep_count'] <= 29, 'Global_Deidentified'
    ]
)
excluded_ids_per_feature.append(excluded_ids)
avg_sleep_per_user = avg_sleep_per_user[avg_sleep_per_user['Time_asleep_count'] > 29]

df_SRI['Avg_SRI'] = df_SRI['SRI']

excluded_ids = set(
    df_SRI.loc[
        df_SRI['NumNights'] <= 29, 'Global_Deidentified'
    ]
)

excluded_ids_per_feature.append(excluded_ids)
df_SRI = df_SRI[df_SRI['NumNights']>29]

avg_sleep_per_user = avg_sleep_per_user.merge(
    df_SRI,
    on='Global_Deidentified',
    how='inner'
)

excluded_in_all = sorted(set.intersection(*excluded_ids_per_feature)) # This ID didn't have anyway 2 STAI scores

from functools import reduce

merged_features_df = []

dfs_to_merge = [
    avg_vo2max_per_user,
    avg_deep_sleep_br_per_user,
    avg_SpO2_per_user,
    avg_Temp_per_user,
    avg_Steps_per_user,
    avg_RHR_per_user,
    avg_deep_sleep_hr_per_user,
    avg_deep_sleep_hrv_per_user,
    avg_deep_sleep_lf_hf_per_user,
    avg_sleep_per_user,
    #df_SRI
]

cleaned_dfs = []
for df in dfs_to_merge:
    # Select the ID column and any column that represents an average value
    cols_to_keep = [col for col in df.columns if col == 'Global_Deidentified' or '_mean' in col or 'Avg_' in col]
    cleaned_dfs.append(df[cols_to_keep])


merged_features_df = reduce(
    lambda left, right: pd.merge(
        left, right, on='Global_Deidentified', how='outer'
    ),
    cleaned_dfs # Use the new, cleaned list
)

id_map_df = df_plot[['Global_Deidentified', 'ID']].drop_duplicates()

final_df = pd.merge(
    merged_features_df,
    id_map_df,
    on='Global_Deidentified',
    how='left'
)

# extract numeric part from existing IDs
num_existing = (
    final_df['ID']
    .dropna()
    .astype(str)
    .str.extract(r'ID_(\d+)')[0]
    .dropna()
    .astype(int)
)

start = num_existing.max() + 1 if not num_existing.empty else 1

# find missing IDs
missing_mask = final_df['ID'].isna()

# assign next IDs only to missing rows
final_df.loc[missing_mask, 'ID'] = [
    f"ID_{i}" for i in range(start, start + missing_mask.sum())
]

# We want to remove those that don't have at least two surveys
final_df_cleaned = final_df.loc[~missing_mask].copy()

ids_final = set(final_df_cleaned['Global_Deidentified'])
ids_plot  = set(df_plot['Global_Deidentified'])
ids_both = ids_final & ids_plot
ids_only_final = ids_final - ids_plot
ids_only_plot  = ids_plot - ids_final

def assign_stress_group(median_score):
    if median_score <= 45:
        return 'Below Average'
    elif 45 < median_score <= 50:
        return 'Average'
    else:
        return 'Above Average'

median_scores_df['Stress_Group'] = median_scores_df['median_stai_stress'].apply(assign_stress_group)

# Create a small, clean DataFrame for mapping groups to participants
group_map_df = median_scores_df[['Global_Deidentified', 'Stress_Group']]

final_df_with_groups = pd.merge(
    final_df_cleaned,
    group_map_df,
    on='Global_Deidentified',
    how='left'
)


In [ ]:
filtered_sleep_df = Sleep_df.groupby('Global_Deidentified').filter(lambda x: len(x) >= 30)

filtered_sleep_df['ID'] = filtered_sleep_df['Global_Deidentified'].map(id_mapping)

# -------------------------------
# Build per-participant missingness from wearable days
# -------------------------------
def summarize_wearable_missingness(df_wearable, id_col="ID", wearable_date_col="Date"):
    d = df_wearable[[id_col, wearable_date_col]].dropna().copy()
    d[wearable_date_col] = pd.to_datetime(d[wearable_date_col], errors="coerce")
    d = d.dropna(subset=[wearable_date_col])
    d["day"] = d[wearable_date_col].dt.floor("D")

    out = (
        d.groupby(id_col)["day"]
         .agg(study_start="min", study_end="max", observed_days="nunique")
         .reset_index()
    )
    out["total_days_in_study"] = (out["study_end"] - out["study_start"]).dt.days + 1
    out["missing_days"] = out["total_days_in_study"] - out["observed_days"]

    # safety
    out.loc[out["missing_days"] < 0, "missing_days"] = 0

    return out[[id_col, "missing_days"]]

df = final_df_with_groups.copy()

wear_missing = summarize_wearable_missingness(
    df_wearable=filtered_sleep_df,
    id_col="ID",
    wearable_date_col="Date"
)

df = df.merge(wear_missing, on="ID", how="left")


base_exclude = ["Unnamed: 0", "Global_Deidentified", "ID", "Stress_Group", "median_stai_stress"]
metrics_to_test = [c for c in df.columns if c not in base_exclude]

# Make sure missingness columns are included (even if you later reorder)
missingness_metrics = ["total_days_in_study", "observed_days", "missing_days", "missing_pct"]
for m in missingness_metrics:
    if m in df.columns and m not in metrics_to_test:
        metrics_to_test.append(m)

# -------------------------------
# Group-level summary + Kruskal-Wallis per metric
# -------------------------------
table_data = []
group_labels = ["Below Average", "Average", "Above Average"]

for metric in metrics_to_test:
    df_metric = df.dropna(subset=[metric, "Stress_Group"]).copy()

    group_below = df_metric[df_metric["Stress_Group"] == "Below Average"][metric]
    group_avg   = df_metric[df_metric["Stress_Group"] == "Average"][metric]
    group_above = df_metric[df_metric["Stress_Group"] == "Above Average"][metric]
    groups_for_test = [group_below, group_avg, group_above]

    row_data = {"Metric": metric}

    for gname, gdata in zip(group_labels, groups_for_test):
        n = len(gdata)
        if n > 0:
            median = gdata.median()
            q1 = gdata.quantile(0.25)
            q3 = gdata.quantile(0.75)
            mean = gdata.mean()
            std = gdata.std()
            row_data[f"{gname} (n)"] = n

            # tighter formatting for missing_pct
            if metric == "missing_pct":
                row_data[f"{gname} [Median [IQR]]"] = f"{median:.3f} [{q1:.3f} - {q3:.3f}]"
                row_data[f"{gname} [Mean (std)]"]   = f"{mean:.3f} ({std:.3f})"
            else:
                row_data[f"{gname} [Median [IQR]]"] = f"{median:.2f} [{q1:.2f} - {q3:.2f}]"
                row_data[f"{gname} [Mean (std)]"]   = f"{mean:.2f} ({std:.2f})"
        else:
            row_data[f"{gname} (n)"] = 0
            row_data[f"{gname} [Median [IQR]]"] = "N/A"
            row_data[f"{gname} [Mean (std)]"] = "N/A"

    # Kruskal-Wallis (same rule you used: each group needs >=10)
    if all(len(g) > 9 for g in groups_for_test):
        h_stat, p_value = stats.kruskal(*groups_for_test)
        row_data["H-statistic"] = f"{h_stat:.2f}"
        row_data["P-value"] = f"{p_value:.3f}"
    else:
        row_data["H-statistic"] = "N/A"
        row_data["P-value"] = "N/A"

    table_data.append(row_data)

summary_table = pd.DataFrame(table_data).set_index("Metric")

metric_order = [
    "missing_days",
]

# Keep only rows that exist (avoid KeyError if any metric name differs)
metric_order_present = [m for m in metric_order if m in summary_table.index]
summary_table = summary_table.reindex(metric_order_present + [m for m in summary_table.index if m not in metric_order_present])
summary_table

## Section 2 — Unsupervised cohort-level phenotyping
as described at: Lautman, Z. D., Snyder, M. & Stanford University. 10.25740/YZ160DB0686. Preprint at https://doi.org/10.25740/YZ160DB0686 (2027).

PCA + k-means applied separately to sleep features (n = 35) and physio-behavioral features (n = 27).

### 2.1 — Sleep phenotyping (Figure 4)


In [ ]:
# one  participant (ID31) was excluded for extreme outlier values across
# multiple sleep metrics (s.d. of sleep onset and duration ≈3 h; PC1 ≈ −8, vs. ≳−3 for all others)
# affecting only the sleep analysis
exclude_ids = ["621e2f3967b776a240c654db"]

ids_to_work_with = final_df_with_groups['Global_Deidentified'].unique()

filtered_sleep_df = filtered_sleep_df[
    filtered_sleep_df['Global_Deidentified'].isin(ids_to_work_with)
    ]

filtered_sleep_df = filtered_sleep_df[~
    filtered_sleep_df['Global_Deidentified'].isin(exclude_ids)
    ]

# List of columns to convert to hr and round
time_columns = ['Time_in_bed', 'Time_to_fall_asleep', 'Time_asleep',
                'light_sleep', 'deep_sleep', 'rem_sleep',
                'Time_to_deep_sleep', 'Time_awake']

Sleep_df_radians = filtered_sleep_df[['Global_Deidentified','Date','Bedtime_start','Bedtime_end']].copy()
Sleep_df_radians_filtered = Sleep_df_radians.groupby('Global_Deidentified').filter(lambda x: len(x) >= 30)


def time_to_radians(time):
    """
    Convert a time value to radians for circular calculations.
    Args:
        time: Time in string, Timestamp, or datetime format.
    Returns:
        float: Time converted to radians.
    """
    if pd.isna(time):
        return np.nan
    if isinstance(time, (pd.Timestamp, datetime)):
        time = time.time()  # Extracts time part
    elif isinstance(time, str):
        try:
            time = datetime.strptime(time, '%H:%M').time()
        except ValueError:
            return np.nan
    else:
        return np.nan
    seconds = time.hour * 3600 + time.minute * 60 + time.second
    radians = (seconds / 86400) * 2 * np.pi
    return radians

def calculate_circular_stats(Sleep_df, start_col, end_col):
    """
    Calculate circular mean and std for bedtime start and end for each participant.
    Args:
        Sleep_df (pd.DataFrame): DataFrame with ID, bedtime start, and end columns.
        start_col (str): Column name for bedtime start.
        end_col (str): Column name for bedtime end.
    Returns:
        pd.DataFrame: DataFrame with circular mean and std for start and end.
    """
    results = []
    for col in [start_col, end_col]:
        # Convert times to radians
        Sleep_df[f'{col}_radians'] = Sleep_df[col].apply(time_to_radians)

        # Group by ID and calculate circular mean and std
        stats = (
            Sleep_df.dropna(subset=[f'{col}_radians'])
            .groupby('Global_Deidentified')[f'{col}_radians']
            .agg(mean_radians=lambda x: circmean(x, high=2 * np.pi, low=0),
                 std_radians=lambda x: circstd(x, high=2 * np.pi, low=0))
            .reset_index()
        )

        # Convert radians back to hours
        stats[f'{col}_mean_hours'] = (stats['mean_radians'] / (2 * np.pi)) * 24 % 24
        stats[f'{col}_std_hours'] = (stats['std_radians'] / (2 * np.pi)) * 24

        # Drop intermediate columns
        stats = stats.drop(columns=['mean_radians', 'std_radians'])
        results.append(stats)

    # Merge stats for start and end
    merged_stats = results[0].merge(results[1], on='Global_Deidentified', suffixes=('_start', '_end'))
    return merged_stats

circular_stats_df = calculate_circular_stats(Sleep_df_radians_filtered, 'Bedtime_start', 'Bedtime_end')

ID_focus1 = '621e32e667b776a2406d2f1c' #blue
ID_focus2 = '621e326767b776a24012e179' #Pink
filtered_ID_sleep_df = Sleep_df[Sleep_df['Global_Deidentified'] == ID_focus1]
filtered_ID_sleep_df = filtered_ID_sleep_df[['Global_Deidentified','Date','Bedtime_start','Bedtime_end','Time_asleep']]
filtered_ID_sleep_df['Time_asleep'] = filtered_ID_sleep_df['Time_asleep']
filtered_ID_sleep_df['Date'] = pd.to_datetime(filtered_ID_sleep_df['Date'], errors='coerce')

filtered_ID_sleep_df2 = Sleep_df[Sleep_df['Global_Deidentified'] == ID_focus2]
filtered_ID_sleep_df2 = filtered_ID_sleep_df2[['Global_Deidentified','Date','Bedtime_start','Bedtime_end','Time_asleep']]
filtered_ID_sleep_df2['Time_asleep'] = filtered_ID_sleep_df2['Time_asleep']
filtered_ID_sleep_df2['Date'] = pd.to_datetime(filtered_ID_sleep_df2['Date'], errors='coerce')

# --- 2. Compute Aggregated Sleep Duration ---
agg_sleep = (filtered_sleep_df
             .groupby('Global_Deidentified')['Time_asleep']
             .agg(['mean', 'std','count'])
             .reset_index()
             .rename(columns={'mean': 'Mean_Time', 'std': 'Std_Time'}))

# --- 2. Compute Aggregated Sleep Stage Percentages ---
df = filtered_sleep_df.copy()
df['perc_light'] = (df['Time_asleep'] - (df['deep_sleep'] + df['rem_sleep']))  / df['Time_asleep'] * 100
df['perc_deep']  = df['deep_sleep']  / df['Time_asleep'] * 100
df['perc_rem']   = df['rem_sleep']   / df['Time_asleep'] * 100
agg_stage = df.groupby('Global_Deidentified')[['perc_light', 'perc_deep', 'perc_rem']].mean().reset_index()

merged_df_sleep = circular_stats_df.merge(agg_sleep, on='Global_Deidentified', how='inner')
merged_df_sleep = merged_df_sleep.merge(agg_stage, on='Global_Deidentified', how='inner')
merged_df_sleep = merged_df_sleep.merge(df_SRI, on='Global_Deidentified', how='inner')

# Remove the 'Unnamed: 0' column if it exists
if 'Unnamed: 0' in merged_df_sleep.columns:
    merged_df_sleep = merged_df_sleep.drop(columns=['Unnamed: 0'])

def time_to_radians(time):
    """
    Convert a time value to radians for circular calculations.
    Args:
        time: Time in string, Timestamp, or datetime format.
    Returns:
        float: Time converted to radians.
    """
    if pd.isna(time):
        return np.nan
    if isinstance(time, (pd.Timestamp, datetime)):
        time = time.time()  # Extracts time part
    elif isinstance(time, str):
        try:
            time = datetime.strptime(time, '%H:%M').time()
        except ValueError:
            return np.nan
    else:
        return np.nan
    seconds = time.hour * 3600 + time.minute * 60 + time.second
    radians = (seconds / 86400) * 2 * np.pi
    return radians

def calculate_circular_stats(Sleep_df, start_col, end_col):
    """
    Calculate circular mean and std for bedtime start and end for each participant.
    Args:
        Sleep_df (pd.DataFrame): DataFrame with ID, bedtime start, and end columns.
        start_col (str): Column name for bedtime start.
        end_col (str): Column name for bedtime end.
    Returns:
        pd.DataFrame: DataFrame with circular mean and std for start and end.
    """
    for col in [start_col, end_col]:
        # Convert times to radians
        Sleep_df[f'{col}_radians_hh:mm'] = (Sleep_df[col].apply(time_to_radians) / (2 * np.pi)) * 24 % 24
        Sleep_df[f'{col}_radians'] = Sleep_df[col].apply(time_to_radians)

    return Sleep_df

circular_stats_df1 = calculate_circular_stats(filtered_ID_sleep_df, 'Bedtime_start', 'Bedtime_end')
circular_stats_df1 = circular_stats_df1.reset_index(drop=True).copy()

circular_stats_df2 = calculate_circular_stats(filtered_ID_sleep_df2, 'Bedtime_start', 'Bedtime_end')
circular_stats_df2 = circular_stats_df2.reset_index(drop=True).copy()

def rolling_circ_mean(x):
    return circular_mean(x)

# Define the function to map Time_asleep to colors with transparency
def map_time_asleep_to_color(time_asleep,alpha = 0.5):
    if time_asleep < 5:
        return mcolors.to_rgba('#d73027', alpha)
    elif 5 <= time_asleep < 7:
        return mcolors.to_rgba('#fee08b', alpha)
    elif 7 <= time_asleep < 8:
        return mcolors.to_rgba('#006837', alpha)
    elif 8 <= time_asleep < 9:
        return mcolors.to_rgba('#fee08b', alpha)
    else:
        return mcolors.to_rgba('#d73027', alpha)

def plot_segmented_line(ax, angles, radii, onset_Time_asleep, label=''):
    """
    Draw a polar line (day by day) where each segment's color is set by std_hrs[i+1].
      angles[i], radii[i] = day i
      angles[i+1], radii[i+1] = day i+1
    We only set the label on the first segment to avoid repeated legend entries.
    """

    # Convert the list to a pandas Series
    onset_Time_asleep_series = pd.Series(onset_Time_asleep)

    # Map to window = 1, essentially keep values
    rolling_value = onset_Time_asleep_series.rolling(window=1).mean()
    for i in range(len(angles) - 1):
        seg_theta = [angles[i], angles[i+1]]
        seg_r = [radii[i], radii[i+1]]
        # color from rolling_value
        seg_color = map_time_asleep_to_color(rolling_value.iloc[i+1])

        # Only set label on the first segment
        use_label = label if i == 0 else None
        ax.plot(seg_theta, seg_r,
        color='black',
        linewidth=5,
        zorder=4)
        ax.plot(seg_theta, seg_r, color='white', linewidth=3, label=use_label, zorder=5)
        ax.plot(seg_theta, seg_r, color=seg_color, linewidth=3, label=use_label, zorder=6)

def circular_mean(angles_radians: np.ndarray) -> float:
    """
    Compute the circular mean of a 1D array of angles (in radians).
    Returns an angle in [0, 2*pi).
    """
    sin_sum = np.sum(np.sin(angles_radians))
    cos_sum = np.sum(np.cos(angles_radians))
    mean_angle = np.arctan2(sin_sum, cos_sum)
    if mean_angle < 0:
        mean_angle += 2*np.pi
    return mean_angle

def circular_std(angles_radians: np.ndarray) -> float:
    """
    Compute the circular standard deviation (in radians).
      R = sqrt((sum cos)^2 + (sum sin)^2) / n
      circ_std = sqrt(-2 * ln(R)).
    """
    n = len(angles_radians)
    if n == 0:
        return np.nan
    sin_sum = np.sum(np.sin(angles_radians))
    cos_sum = np.sum(np.cos(angles_radians))
    R = np.sqrt(sin_sum**2 + cos_sum**2) / n
    if R <= 0:
        return np.nan
    return np.sqrt(-2 * np.log(R))

In [ ]:
# Pairwise Pearson correlation of sleep metrics (Figure S4)
df_num = merged_df_sleep[[
    'Bedtime_start_std_hours',
    'Bedtime_end_std_hours', 'Mean_Time', 'Std_Time',
    'perc_rem', 'SRI']].select_dtypes(include='number')

df_num.columns = [
    'Mean Sleep Onset SD (h)',
    'Mean Sleep Wakeup SD (h)',
    'Mean Sleep Duration (h)',
    'Sleep Duration SD (h)',
    'REM %',
    'Sleep Regularity Index'
]

corr = df_num.corr(method='pearson')

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask,
    vmin=-1, vmax=1, center=0,
    cmap='coolwarm', square=True, linewidths=0.5,
    annot=True, fmt=".2f", cbar_kws={'shrink': 0.8})
plt.title('Pairwise Pearson Correlation')
plt.tight_layout()
plt.show()

In [ ]:
def plot_kmeans_pca_with_stress_color(
    merged_df_sleep, median_scores_df, cluster_markers, axs, ax_position,
    scatter_size=100,
    grid_linewidth=0.8,
    grid_alpha=0.7,
    fontsize = 20
):
    """
    Generates a K-Means/PCA plot where subjects are colored by their median stress level.
    """
    # --------------------
    # Data Preparation (No changes here)
    # --------------------

    features = merged_df_sleep[['Mean_Time', 'Std_Time', 'perc_rem', 'SRI', 'Bedtime_start_std_hours','Bedtime_end_std_hours']]
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)
    kmeans = KMeans(n_clusters=3, random_state=42)
    merged_df_sleep['Cluster'] = kmeans.fit_predict(scaled_features)
    pca = PCA(n_components=2)
    pca_components = pca.fit_transform(scaled_features)
    pca_fit = pca.fit(scaled_features)
    merged_df_sleep['PC1'] = pca_components[:, 0] * -1
    merged_df_sleep['PC2'] = pca_components[:, 1]

    labels = merged_df_sleep['Cluster'].values

    silhouette_per_point = silhouette_samples(features, labels)
    silhouette_avg = silhouette_per_point.mean()
    silhouette_min = silhouette_per_point.min()
    silhouette_max = silhouette_per_point.max()
    silhouette_med = np.median(silhouette_per_point)

    print(f"Average silhouette: {silhouette_avg:.3f}")
    print(f"Silhouette range: {silhouette_min:.3f} to {silhouette_max:.3f}")
    print(f"Silhouette median: {silhouette_med:.3f}")

    evr = pca.explained_variance_ratio_

    print(f"Explained variance PC1: {evr[0]:.3f}")
    print(f"Explained variance PC2: {evr[1]:.3f}")
    print(f"Total explained by first 2 PCs: {evr.sum():.3f}")

    # Map each subject to a marker based on its cluster for later emphasis
    id_to_marker = {row['Global_Deidentified']: cluster_markers[row['Cluster']]
                    for _, row in merged_df_sleep.iterrows()}

    #  Merge with median scores to link sleep data to anxiety levels ---
    plot_df = pd.merge(
        merged_df_sleep,
        median_scores_df,
        on='Global_Deidentified',
        how='left'  # Use a 'left' merge to keep all sleep subjects
    )
    # Apply the color function to create a new column for highlighting
    plot_df['stress_color'] = plot_df['median_stai_stress'].apply(get_color)

    # --------------------
    # Plotting Clusters
    # --------------------
    # 1. Plot ALL subjects in grayscale first to serve as a background
    cluster_shades = {0: '#AAAAAA', 1: '#666666', 2: '#222222'}
    for cluster, marker in cluster_markers.items():
        cluster_data = plot_df[plot_df['Cluster'] == cluster]
        shade = cluster_shades[cluster]
        edge = 'black' if cluster == 0 else 'none'
        axs[ax_position].scatter(
            cluster_data['PC1'], cluster_data['PC2'],
            marker=marker, s=scatter_size, facecolors=shade,
            edgecolors=edge, linewidths=1, zorder=1
        )

    # 2. Plot ONLY the subjects with stress data on TOP, using their stress color
    highlight_data = plot_df.dropna(subset=['stress_color'])
    for cluster, marker in cluster_markers.items():
        # Further filter by cluster to use the correct marker
        cluster_highlight_data = highlight_data[highlight_data['Cluster'] == cluster]
        if not cluster_highlight_data.empty:
            axs[ax_position].scatter(
                cluster_highlight_data['PC1'],
                cluster_highlight_data['PC2'],
                marker=marker,
                s=scatter_size,
                facecolors=cluster_highlight_data['stress_color'], # Use the color from the column
                edgecolors='black',
                linewidths=0.5,
                zorder=5  # Higher zorder ensures these points are on top
            )

    cluster_data_A = merged_df_sleep[merged_df_sleep['Global_Deidentified'] == ID_focus1]
    cluster_data_B = merged_df_sleep[merged_df_sleep['Global_Deidentified'] == ID_focus2]

    subject_data_a = median_scores_df[median_scores_df['Global_Deidentified'] == ID_focus1]
    stress_score_a = subject_data_a['median_stai_stress'].iloc[0]
    color_a = get_color(stress_score_a)

    subject_data_b = median_scores_df[median_scores_df['Global_Deidentified'] == ID_focus2]
    stress_score_b = subject_data_b['median_stai_stress'].iloc[0]
    color_b = get_color(stress_score_b)

    # Highlight focused subjects with distinct colors
    axs[ax_position].scatter(cluster_data_A['PC1'], cluster_data_A['PC2'],
                             marker=id_to_marker[ID_focus1],
                             s=scatter_size,
                             alpha=1, color = color_a,
                             edgecolors = 'blue',linewidths=3, zorder=5)
    axs[ax_position].scatter(cluster_data_B['PC1'], cluster_data_B['PC2'],
                             marker=id_to_marker[ID_focus2],
                             s=scatter_size, color = color_b,
                             alpha=1,edgecolors = 'blue',linewidths=3, zorder=5)

    # --------------------
    # Plotting PC Loadings (Arrows) - No changes here
    # --------------------
    loadings = pca_fit.components_[:2, :]
    loadings[0, :] *= -1
    feature_names = features.columns
    scores = plot_df[['PC1', 'PC2']].values
    max_val = np.max(np.abs(scores))
    arrow_scale = 0.13 * max_val
    x_origin, y_origin = -3, -2.8
    for i, feature in enumerate(feature_names):
        arrow_x = loadings[0, i] * arrow_scale
        arrow_y = loadings[1, i] * arrow_scale
        axs[ax_position].arrow(x_origin, y_origin, arrow_x, arrow_y,
                               head_width=0.02*max_val, head_length=0.02*max_val,
                               fc='white', ec='black', zorder=6)

    # --------------------
    # Final Touches and Legend
    # --------------------
    axs[ax_position].grid(True, linestyle='--', alpha=grid_alpha, linewidth=grid_linewidth)
    axs[ax_position].tick_params(axis='both', which='major', labelsize=20)

    edgecolor = plt.rcParams['legend.edgecolor']
    bbox = FancyBboxPatch((-3.5, -3), 0.8, 0.8, fc='white', ec=edgecolor, lw=0.8, zorder=5, alpha=0.8)
    axs[ax_position].add_patch(bbox)

    # Create a new legend for the stress colors ---
    handles = []
    for i,m in enumerate(cluster_markers.values()):
        handles.append( plt.Line2D([], [], linestyle='None', marker=m,
                                  markersize=12, markeredgecolor='black',
                                  markerfacecolor='black', label=f'Cluster {i+1}') )
    axs[ax_position].legend(handles=handles, title='Subject',loc='lower right')


    expl_var = pca_fit.explained_variance_ratio_

    x_label = f"PC1 ({expl_var[0]*100:.1f}% explained)"
    y_label = f"PC2 ({expl_var[1]*100:.1f}% explained)"

    return merged_df_sleep, scaler, kmeans, pca, pca_fit, id_to_marker, scaled_features

#### Sleep Duration and s.d.


In [ ]:
def plot_sleep_duration_vs_std(merged_df_sleep, median_scores_df, id_to_marker, ID_focus1, ID_focus2, cluster_markers, axs, ax_position,
                               scatter_size=100,
                               grid_linewidth=0.8,
                               grid_alpha = 0.7):
    """
    Plots Yearly Mean Sleep Duration vs. Standard Deviation with color-coded population benchmarks.

    Parameters:
      merged_df_sleep (DataFrame): Data containing sleep duration statistics (columns: 'Mean_Time', 'Std_Time', 'Cluster').
      id_to_marker (dict): Mapping from subject identifiers to marker styles.
      ID_focus1, ID_focus2: Identifiers for subjects to emphasize.
      cluster_markers (dict): Mapping of cluster indices to marker styles.
      axs (array-like): Array of matplotlib axes.
      ax_position (int): Index position in axs to plot on.
      scatter_size (int): Marker size used for scatter plots (default: 40).
      grid_linewidth (float): Width of grid lines (default: 0.8).

    Returns:
      DataFrame: Aggregated DataFrame (here using merged_df_sleep).
    """
    # --------------------
    # SECTION 1: Set Axes Limits and Background Color Spans
    # --------------------
    x_min = np.floor(merged_df_sleep['Mean_Time'].min())
    x_max = np.ceil(merged_df_sleep['Mean_Time'].max())
    y_min = np.floor(merged_df_sleep['Std_Time'].min())
    y_max = np.ceil(merged_df_sleep['Std_Time'].max())

    ax = axs[ax_position]
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    # Vertical (x-axis) spans
    ax.axvspan(x_min, 5, facecolor='#d73027', alpha=0.3, zorder=0)  # Red
    ax.axvspan(5, 7, facecolor='#fee08b', alpha=0.3, zorder=0)         # Orange
    ax.axvspan(7, 8, facecolor='#006837', alpha=0.3, zorder=0)         # Green
    ax.axvspan(8, 9, facecolor='#fee08b', alpha=0.3, zorder=0)         # Orange
    ax.axvspan(9, x_max, facecolor='#d73027', alpha=0.3, zorder=0)      # Red

    # Horizontal (y-axis) spans
    ax.axhspan(y_min, 1, facecolor='#006837', alpha=0.3, zorder=1)      # Green
    ax.axhspan(1, 2, facecolor='#fee08b', alpha=0.3, zorder=1)           # Orange
    ax.axhspan(2, y_max, facecolor='#d73027', alpha=0.3, zorder=1)       # Red

    # --------------------
    # SECTION 2: Plot Data Points for All Subjects
    # --------------------
    cluster_shades = {0: '#AAAAAA', 1: '#666666', 2: '#222222'}
    for id_, row in merged_df_sleep.iterrows():
        marker = id_to_marker.get(row['Global_Deidentified'], 'o')
        shade = cluster_shades[row['Cluster']]
        edge = 'black'
        subject_data = median_scores_df[median_scores_df['Global_Deidentified'] == row['Global_Deidentified']]
        stress_score = subject_data['median_stai_stress'].iloc[0]
        color = get_color(stress_score)
        ax.scatter(row['Mean_Time'], row['Std_Time'], color=color,edgecolors=edge,
            linewidths=1,   marker=marker,
                   zorder=3, s=scatter_size)

    # --------------------
    # SECTION 3: Highlight Focused Subjects
    # --------------------
    agg_sleep_A = merged_df_sleep[merged_df_sleep['Global_Deidentified'] == ID_focus1]
    agg_sleep_B = merged_df_sleep[merged_df_sleep['Global_Deidentified'] == ID_focus2]

    subject_data_a = median_scores_df[median_scores_df['Global_Deidentified'] == ID_focus1]
    stress_score_a = subject_data_a['median_stai_stress'].iloc[0]
    color_a = get_color(stress_score_a)

    subject_data_b = median_scores_df[median_scores_df['Global_Deidentified'] == ID_focus2]
    stress_score_b = subject_data_b['median_stai_stress'].iloc[0]
    color_b = get_color(stress_score_b)

    ax.scatter(agg_sleep_A['Mean_Time'], agg_sleep_A['Std_Time'],
               marker=id_to_marker[ID_focus1], s=scatter_size, alpha=1, edgecolors = 'blue',linewidths=3,
               facecolors=color_a, zorder=3)
    ax.scatter(agg_sleep_B['Mean_Time'], agg_sleep_B['Std_Time'],
               marker=id_to_marker[ID_focus2], s=scatter_size, alpha=1,
               facecolors=color_b,edgecolors = 'blue',linewidths=3, zorder=3)

    # --------------------
    # SECTION 4: Final Aesthetic Touches
    # --------------------
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(True, which='both', linestyle='--', linewidth=grid_linewidth, alpha = grid_alpha)
   # ax.set_xlabel('Mean Sleep Duration (h)', fontsize=20)
   # ax.set_ylabel('Sleep Duration s.d. (h)', fontsize=20)
    ax.tick_params(axis='both', which='major', labelsize=20)

    return merged_df_sleep

#### Sleep Onset and s.d.


In [ ]:
def plot_circular_sleep_stats(df_clock,median_scores_df, id_to_marker, ID_focus1, ID_focus2, axs, ax_position,
                              scatter_size=100,
                              grid_linewidth=0.8,
                              grid_alpha = 0.7):
    """
    Creates a circular (polar) plot of sleep statistics.

    Parameters:
      df_clock (DataFrame): Data containing sleep statistics including bedtime start times.
      id_to_marker (dict): Mapping from subject identifiers to marker styles.
      ID_focus1, ID_focus2: Identifiers for subjects to emphasize on the plot.
      axs (array-like): Array of matplotlib axes.
      ax_position (int): Index of the axis to plot on.
      scatter_size (int): Marker size for scatter points (default: 40).
      grid_linewidth (float): Line width for grid lines (default: 0.8).

    Returns:
      DataFrame: The modified dataframe with additional computed columns.
    """
    # --------------------
    # SECTION 1: Data Preparation
    # --------------------
    # Compute the maximum bedtime standard hours and normalize radial distance.
    max_radial_distance = df_clock['Bedtime_start_std_hours'].max()
    df_clock['Radial_distance_start_std'] = df_clock['Bedtime_start_std_hours'] / max_radial_distance

    # --------------------
    # SECTION 2: Setting up the Polar Plot
    # --------------------
    ax = axs[ax_position]
    ax.set_theta_direction(-1)
    ax.set_theta_offset(np.pi / 2)
    ax.set_thetamin(-90)
    ax.set_thetamax(90)
    ax.set_yticklabels([])

    # Adjust x-ticks for the top half circle (from -90 to 90 degrees)
    ax.set_xticks(np.linspace(-np.pi/2, np.pi/2, 13))
    ax.set_xticklabels([f'{i:02d}' for i in range(18, 24)] + [f'{i:02d}' for i in range(7)], fontsize=20)
    ax.tick_params(axis='x', pad=5)

    # --------------------
    # SECTION 3: Radial Ticks and Circular Guides
    # --------------------
    num_radial_ticks = round(max_radial_distance) + 1
    radial_ticks = [i / max_radial_distance for i in range(num_radial_ticks + 1)]
    ax.set_yticks(radial_ticks)
    ax.set_yticklabels([''] * (num_radial_ticks + 1))

    # --------------------
    # SECTION 4: Background Fill for Theta Ranges
    # --------------------
    theta = np.linspace(-np.pi/2, np.pi/2, 100)
    ax.fill(theta, np.ones_like(theta) * radial_ticks[-1], color='#d73027', alpha=0.5, label="_nolegend_", zorder=0)  # Red
    ax.fill(theta, np.ones_like(theta) * radial_ticks[2], color='white', label="_nolegend_", zorder=0)  # Orange
    ax.fill(theta, np.ones_like(theta) * radial_ticks[2], color='#fee08b', alpha=0.5, label="_nolegend_", zorder=0)  # Orange
    ax.fill(theta, np.ones_like(theta) * radial_ticks[1], color='white', label="_nolegend_", zorder=0)  # Green
    ax.fill(theta, np.ones_like(theta) * radial_ticks[1], color='#006837', alpha=0.5, label="_nolegend_", zorder=0)  # Green

    # --------------------
    # SECTION 5: Plotting Data Points
    # --------------------
    cluster_shades = {0: '#AAAAAA', 1: '#666666', 2: '#222222'}

    for id_, row in df_clock.iterrows():
        marker = id_to_marker.get(row['Global_Deidentified'], 'o')
        theta_val = row['Bedtime_start_mean_hours'] * (2 * np.pi / 24)
        subject_data = median_scores_df[median_scores_df['Global_Deidentified'] == row['Global_Deidentified']]
        stress_score = subject_data['median_stai_stress'].iloc[0]
        color = get_color(stress_score)

        if row['Global_Deidentified'] == ID_focus1:
            # Adjust the radial distance slightly if it reaches the maximum (to avoid edge overlap)
            temp_r = 0.98 if row['Radial_distance_start_std'] == 1 else row['Radial_distance_start_std']
            ax.scatter(theta_val, temp_r, label='Mean Sleep Onset', color=color,
                       marker=id_to_marker[ID_focus1], s=scatter_size, alpha=1,linewidths=3,edgecolors='blue', zorder=2)
        elif row['Global_Deidentified'] == ID_focus2:
            ax.scatter(theta_val, row['Radial_distance_start_std'], label='Mean Sleep Onset', color=color,
                       marker=id_to_marker[ID_focus2], s=scatter_size,edgecolors='blue',linewidths=3, alpha=1, zorder=2)
        else:
            ax.scatter(theta_val, row['Radial_distance_start_std'], label='Mean Sleep Onset',
                       color=color, marker=marker, s=scatter_size,edgecolors='black',linewidths=1, alpha=1, zorder=2)

    # --------------------
    # SECTION 6: Adjusting Plot Position and Limits & Add grid
    # --------------------
    box = ax.get_position()
    ax.set_position([box.x0 - 0.03, box.y0 - 0.115, box.width * 1.3, box.height * 1.3])
    ax.set_ylim(0, 1)
    ax.grid(True, color='grey', linestyle='--', linewidth=grid_linewidth, alpha = grid_alpha)

    return df_clock

#### Sleep Regularity Index


In [ ]:
def plot_sri_with_stress_color(
    df_SRI, median_scores_df, id_to_marker, axs, ax_position,
    scatter_size=100,
    grid_linewidth=0.8,
    grid_alpha=0.7
):
    """
    Plots a boxplot and swarmplot of SRI, coloring subjects by their median stress level.
    """
    ax = axs[ax_position]

    # Merge with median scores to link SRI to anxiety levels ---
    plot_df = pd.merge(df_SRI, median_scores_df, on='Global_Deidentified', how='left')

    # --------------------
    # SECTION 1: Base Plot (No changes)
    # --------------------
    sns.boxplot(data=plot_df['SRI'], orient='h', fliersize=0, width=0.5, ax=ax,
                boxprops=dict(facecolor='none', edgecolor='black'),
                whiskerprops=dict(color='black'), medianprops=dict(color='black'))
    swarm_plot = sns.swarmplot(data=plot_df['SRI'], color='black', alpha=0,
                               orient='h', ax=ax, size=15)
    x_positions = swarm_plot.collections[0].get_offsets()[:, 0]
    y_positions = swarm_plot.collections[0].get_offsets()[:, 1]

    # --------------------
    # SECTION 2: Overplotting with Conditional Coloring
    # --------------------
    cluster_shades = {'o': '#AAAAAA', '^': '#666666', 's': '#222222'}

    # --- CHANGE: Loop now uses the merged 'plot_df' and conditional logic ---
    for i, (x, y) in enumerate(zip(x_positions, y_positions)):
        # Get data for the current subject from the merged DataFrame
        subject_data = plot_df.iloc[i]
        id_ = subject_data['Global_Deidentified']
        stress_score = subject_data['median_stai_stress']
        marker = id_to_marker.get(id_, 'o')
        color = get_color(stress_score)

        # If the subject has a stress score, color them accordingly
        if pd.notna(stress_score):
          if id_ == ID_focus1:
              ax.scatter(x=x, y=y, color=color, marker=id_to_marker[ID_focus1],
                        s=scatter_size, zorder=5, alpha=1, edgecolors='blue',linewidths=3)
          elif id_ == ID_focus2:
              ax.scatter(x=x, y=y, color=color, marker=id_to_marker[ID_focus2],
                        s=scatter_size, zorder=5,edgecolors='blue',linewidths=3, alpha=1)
          else:
              ax.scatter(x=x, y=y, color=color, marker=marker,
                        s=scatter_size, zorder=5, edgecolors='black',
                        linewidths=1, alpha=1)

    # --------------------
    # SECTION 3: Axes Formatting (No changes)
    # --------------------
    ax.set_xlabel(' ', fontsize=20)
    ax.tick_params(axis='x', labelsize=20)
    ax.set_xlim(np.floor(min(plot_df['SRI'])), 100)
    sns.despine(ax=ax)
    ax.fill_betweenx(y=[-0.5, 0.5], x1=0, x2=60, color='#d73027', alpha=0.5, zorder=0)
    ax.fill_betweenx(y=[-0.5, 0.5], x1=60, x2=80, color='#fee08b', alpha=0.5, zorder=0)
    ax.fill_betweenx(y=[-0.5, 0.5], x1=80, x2=100, color='#006837', alpha=0.5, zorder=0)
    ax.grid(True, linestyle='--', alpha=1, linewidth=grid_linewidth)
    ax.set_ylim(-0.3, 0.3)

    return df_SRI

#### Participant A & B


In [ ]:
def plot_Individual_circular_sleep_stats(df_clock1, df_clock2,color_a,color_b, label1, label2, id_to_marker,
                                           ID_Focus1, ID_Focus2, axs, ax_position,
                                           scatter_size=80,
                                           grid_linewidth=0.5,
                                           grid_alpha = 0.7):
    """
    Creates an individual circular plot comparing sleep statistics for two datasets,
    representing two subjects or conditions. The plot includes scatter markers, segmented
    lines based on a 7-day rolling circular mean, and a colorbar for additional annotation.

    Parameters:
      df_clock1 (DataFrame): Data for the first subject/condition. Must include columns
                             'Bedtime_start_radians_hh:mm', 'Date', and 'Time_asleep'.
      df_clock2 (DataFrame): Data for the second subject/condition with the same structure as df_clock1.
      label1 (str): Label for df_clock1.
      label2 (str): Label for df_clock2.
      id_to_marker (dict): Mapping from subject identifiers to marker styles.
      ID_Focus1: Identifier to emphasize from df_clock1.
      ID_Focus2: Identifier to emphasize from df_clock2.
      axs (array-like): Array of matplotlib axes.
      ax_position (int): Index in axs where the plot will be drawn.
      scatter_size (int): Marker size for scatter plots (default: 20).
      grid_linewidth (float): Grid line width for the polar plot (default: 0.5).

    Returns:
      tuple: (processed df_clock1, processed df_clock2)

    Notes:
      - This function requires that external functions `rolling_circ_mean` and `plot_segmented_line`
        are defined.
      - The variable `fig` must be defined globally (or passed in via another mechanism) for the colorbar.
    """
    # --------------------
    # SECTION 1: Data Preparation
    # --------------------
    def prepare_data(df_clock):
        max_radial_distance = len(df_clock)
        df_clock['Bedtime_start_radians_length'] = df_clock.index / max_radial_distance
        df_clock['start_angle'] = df_clock['Bedtime_start_radians_hh:mm'] * (2 * np.pi / 24)
        df_clock['start_angle_14d_mean'] = df_clock['start_angle'].rolling(window=7, min_periods=7)\
                                              .apply(rolling_circ_mean, raw=True)
        return df_clock

    df_clock1 = prepare_data(df_clock1)
    df_clock2 = prepare_data(df_clock2)
    max_radial_distance = min(len(df_clock1), len(df_clock2))

    # --------------------
    # SECTION 2: Polar Plot Setup
    # --------------------
    ax = axs[ax_position]
    ax.grid(True, color='black', linestyle='--', linewidth=grid_linewidth, alpha = grid_alpha)
    ax.spines['polar'].set_edgecolor('black')
    ax.spines['polar'].set_linestyle((0, (3, 20)))
    ax.set_theta_direction(-1)
    ax.set_theta_offset(np.pi / 2)
    ax.set_yticklabels([])
    ax.set_xticks(np.linspace(0, 2 * np.pi, 24, endpoint=False))
    ax.set_xticklabels([f'{i:02d}' for i in range(24)], fontsize=20)
    ax.tick_params(axis='x', pad=5)
    ax.set_ylim(0, 1)

    # --------------------
    # SECTION 3: Radial Ticks and Circular Guides
    # --------------------
    num_radial_ticks = len(df_clock1['Date'].dt.to_period('M').unique())
    radial_ticks = np.linspace(0, 1, num_radial_ticks)
    radial_labels = [f'{df_clock1["Date"][round(max(0, i * max_radial_distance - 1))].strftime("%b")}'
                     for i in radial_ticks]
    ax.set_yticks(radial_ticks)
    ax.set_yticklabels([''] * len(radial_labels))

    # Add circular guide lines with varying linestyles
    linestyles = [':', '--', (0, (5, 10))]
    colors = ['white', 'white', 'white']
    for i, r in enumerate(radial_ticks[1:]):
        circle = plt.Circle((0, 0), r, transform=ax.transData._b, color=colors[i % len(colors)],
                            fill=False, linestyle=linestyles[i % len(linestyles)])
        ax.add_artist(circle)

    # --------------------
    # SECTION 4: Plotting Data Points and Segmented Lines
    # --------------------
    # Plot the first dataset (df_clock1)
    ax.scatter(df_clock1['Bedtime_start_radians_hh:mm'] * (2 * np.pi / 24),
               df_clock1['Bedtime_start_radians_length'],
               label=f'{label1} Sleep Onset',
               color=color_a, s=scatter_size, alpha=0.8, edgecolor = 'black', linewidth=1,
               marker=id_to_marker[ID_Focus1])
    plot_segmented_line(ax,
                        df_clock1['start_angle_14d_mean'].values,
                        df_clock1['Bedtime_start_radians_length'].values,
                        df_clock1['Time_asleep'].values,
                        label=f'{label1} Onset (7d Mean, colored by std in hours)')

    # Plot the second dataset (df_clock2)
    ax.scatter(df_clock2['Bedtime_start_radians_hh:mm'] * (2 * np.pi / 24),
               df_clock2['Bedtime_start_radians_length'],
               label=f'{label2} Sleep Onset',
               color=color_b, s=scatter_size, alpha=0.8, edgecolor = 'black', linewidth=1,
               marker=id_to_marker[ID_Focus2])
    plot_segmented_line(ax,
                        df_clock2['start_angle_14d_mean'].values,
                        df_clock2['Bedtime_start_radians_length'].values,
                        df_clock2['Time_asleep'].values,
                        label=f'{label2} Onset (7d Mean, colored by std in hours)')

    # --------------------
    # SECTION 5: Adjusting Subplot Position and Adding a Colorbar
    # --------------------
    box = ax.get_position()
    ax.set_position([box.x0, box.y0, box.width, box.height])
    # Note: 'fig' must be defined outside this function.
    box = ax.get_position()
    cbar_ax = fig.add_axes([box.x1 + 0.02, box.y0, 0.003, box.height])

    cmap_colors = [
        mcolors.to_rgba('#d73027', alpha=0.5),  # Red
        mcolors.to_rgba('#fee08b', alpha=0.5),  # Soft Yellow
        mcolors.to_rgba('#006837', alpha=0.5)   # Green

    ]
    boundaries = [1, 2, 3, 4]
    cmap = ListedColormap(cmap_colors)
    norm = BoundaryNorm(boundaries, cmap.N)
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                        cax=cbar_ax, ax=ax, boundaries=boundaries,
                        spacing='uniform', extend='neither', ticks=[])

    return df_clock1, df_clock2

#### Sleep phenotyping figure (Figure 4)


In [ ]:
# Since here the PC1 is on the other direction, we are flipping it to match
# the previous paper - see the * -1 in the PCA plot function

# Define markers for each cluster
cluster_markers = {2: 'o', 0: '^', 1: 's'}

fig, axs = plt.subplots(2, 3, figsize=(32, 16))

# a - Kmeans and PCA
#merged_df_sleep, scaler, kmeans, pca, pca_fit, id_to_marker, scaled_features = plot_kmeans_pca_sleep_features(merged_df_sleep, cluster_markers, ID_focus1, ID_focus2, axs, (0, 0))
merged_df_sleep, scaler, kmeans, pca, pca_fit, id_to_marker, scaled_features = plot_kmeans_pca_with_stress_color(merged_df_sleep, median_scores_df, cluster_markers, axs, (0, 0))

# b - Sleep duration and s.d.
agg_sleep = plot_sleep_duration_vs_std(merged_df_sleep,median_scores_df, id_to_marker, ID_focus1, ID_focus2, cluster_markers, axs, (0, 2))

# c - Sleep onset and s.d.
axs[1, 0].set_frame_on(False)
axs[1, 0].set_xticks([])
axs[1, 0].set_yticks([])
axs[1, 0] = fig.add_subplot(2, 3, 4, projection='polar')
new_circular_stats_df = circular_stats_df.merge(merged_df_sleep[['Global_Deidentified', 'Cluster']],on='Global_Deidentified', how='left')
new_circular_stats_df = new_circular_stats_df.dropna(subset=["Cluster"])
df_clock = plot_circular_sleep_stats(new_circular_stats_df,median_scores_df, id_to_marker, ID_focus1, ID_focus2, axs, (1, 0))

# d - prepare for REM
axs[1,1].set_frame_on(False)
axs[1,1].set_xticks([])
axs[1,1].set_yticks([])

# e - Sleep Regularity Index
#merged_df_sleep = plot_sleep_regularity_index(merged_df_sleep, id_to_marker, ID_focus1, ID_focus2, )
merged_df_sleep = plot_sri_with_stress_color(merged_df_sleep, median_scores_df, id_to_marker, axs, (0, 1))

# f - Participant A & B
subject_data_a = median_scores_df[median_scores_df['Global_Deidentified'] == ID_focus1]
stress_score_a = subject_data_a['median_stai_stress'].iloc[0]
color_a = get_color(stress_score_a)

subject_data_b = median_scores_df[median_scores_df['Global_Deidentified'] == ID_focus2]
stress_score_b = subject_data_b['median_stai_stress'].iloc[0]
color_b = get_color(stress_score_b)
axs[1, 2].set_frame_on(False)
axs[1, 2].set_xticks([])
axs[1, 2].set_yticks([])
axs[1, 2] = fig.add_subplot(2, 3, 6, projection='polar')
df_clock1, df_clock2 = plot_Individual_circular_sleep_stats(circular_stats_df1, circular_stats_df2, color_a, color_b, 'Participant A', 'Participant B', id_to_marker, ID_focus1, ID_focus2, axs, (1, 2))

#plt.savefig(f'{OUTPUT_DIR}/Sleep.png', dpi=600,transparent=True)
#plt.show()

len(merged_df_sleep['Global_Deidentified'].unique())


#### Plot REM


In [ ]:
import math

x = 52

def floor_ceil(x):
  # floor to nearest 10
  floor10 = (x // 10) * 10        # → 50

  # ceil to nearest 10
  ceil10  = math.ceil(x / 10) * 10  # → 60

  return floor10, ceil10

min_light, _ = floor_ceil(merged_df_sleep['perc_light'].min())
_ , max_light = floor_ceil(merged_df_sleep['perc_light'].max())
min_deep, _ = floor_ceil(merged_df_sleep['perc_deep'].min())
_, max_deep = floor_ceil(merged_df_sleep['perc_deep'].max())
min_rem, _ = floor_ceil(merged_df_sleep['perc_rem'].min())
_, max_rem = floor_ceil(merged_df_sleep['perc_rem'].max())

# --- 3. Define Custom Axis Styling Function ---
def makeAxis(title, tickangle, min_val):
    return {
        'title': {'text': title, 'font': {'size': 1}},
        'tickangle': tickangle,
        'tickfont': {'size': 20, 'color': 'black'},  # Set tick labels to black},
        'ticksuffix': '%',  # Adds '%' to all tick labels
        #'ticklen': 8,
        'showline': True,
        'showgrid': True,
        'linecolor': 'black',  # Set tick marks to black
        'gridcolor': 'black',  # Set grid lines to black
        'min': min_val,
    }

# --- 4. Create the Ternary Plot ---
# Coordinate order:
#    a-axis: Deep Sleep (%)
#    b-axis: REM Sleep (%)
#    c-axis: Light Sleep (%)
fig = go.Figure()

# Define REM sleep boundaries (20% and 25%)
rem_min, rem_max = 0.2, 0.25  # REM sleep between 20% and 25%
t = [0.0, 1.0, 1.0, 0.0]  # Deep Sleep varies from 0% to 100%
l = [rem_min, rem_min, rem_max, rem_max]  # Fixed REM sleep between 20%-25%
r = [1 - (t[i] + l[i]) for i in range(len(t))]  # Light Sleep (ensuring sum=1)
fig.add_scatterternary(a=t, b=l, c=r, name="shaded area",
                        mode='lines', fill="toself", showlegend=False, fillcolor='rgba(0,104,55,0.5)', line=dict(width=0))

# Define REM sleep boundaries
rem_min, rem_max = 0.15, 0.2  # REM sleep between 15% and 20%
t = [0.0, 1.0, 1.0, 0.0]  # Deep Sleep varies from 0% to 100%
l = [rem_min, rem_min, rem_max, rem_max]  # Fixed REM sleep between 15%-20%
r = [1 - (t[i] + l[i]) for i in range(len(t))]  # Light Sleep (ensuring sum=1)
fig.add_scatterternary(a=t, b=l, c=r, name="shaded area",
                        mode='lines', fill="toself", showlegend=False, fillcolor='rgba(254,224,139,0.5)', line=dict(width=0))

# Define REM sleep boundaries
rem_min, rem_max = 0, 0.15  # REM sleep between 0% and 15%
t = [0.0, 1.0, 1.0, 0.0]  # Deep Sleep varies from 0% to 100%
l = [rem_min, rem_min, rem_max, rem_max]  # Fixed REM sleep between 0%-15%
r = [1 - (t[i] + l[i]) for i in range(len(t))]  # Light Sleep (ensuring sum=1)
fig.add_scatterternary(a=t, b=l, c=r, name="shaded area",
                        mode='lines', fill="toself", showlegend=False, fillcolor='rgba(215,48,39,0.5)', line=dict(width=0))

# Plot each point with the assigned marker
cluster_shades = {'o': '#AAAAAA', '^': '#666666', 's': '#222222'}

for i, row in agg_stage.iterrows():
    subject_data = median_scores_df[median_scores_df['Global_Deidentified'] == row['Global_Deidentified']]
    stress_score = subject_data['median_stai_stress'].iloc[0]
    color = get_color(stress_score)
    marker = id_to_marker.get(row['Global_Deidentified'], 'circle')  # Default marker is 'circle'
    if marker == 'o':
        marker = 'circle'
        edge = 'black'
    if marker == '^':
        marker = 'triangle-up'
        edge = None
    if marker == 's':
        marker = 'square'
        edge = None
    marker_color = id_to_marker.get(row['Global_Deidentified'], 'o')
    if marker == 'circle':
      fig.add_trace(go.Scatterternary(
        mode='markers',
        a=[row['perc_deep']],
        b=[row['perc_rem']],
        c=[row['perc_light']],
        text=row['Global_Deidentified'],
      marker=dict(
          symbol=marker,
          color=color,
          size=60,
          line=dict(color='black', width=2)  # ← black edge
      ),
      name=row['Global_Deidentified'],
      showlegend=False
    ))
    else:
        fig.add_trace(go.Scatterternary(
        mode='markers',
        a=[row['perc_deep']],
        b=[row['perc_rem']],
        c=[row['perc_light']],
        text=row['Global_Deidentified'],
      marker=dict(
          symbol=marker,
          color=color,
          size=60,
          line=dict(color='black', width=2)  # ← black edge
      ),
      name=row['Global_Deidentified'],
      showlegend=False
    ))

for i, row in agg_stage.iterrows():
    subject_data = median_scores_df[median_scores_df['Global_Deidentified'] == row['Global_Deidentified']]
    stress_score = subject_data['median_stai_stress'].iloc[0]
    color = get_color(stress_score)
    marker = id_to_marker.get(row['Global_Deidentified'], 'circle')  # Default marker is 'circle'
    if marker == 'o':
        marker = 'circle'
    if marker == '^':
        marker = 'triangle-up'
    if marker == 's':
        marker = 'square'
    if row['Global_Deidentified'] == ID_focus1:
      fig.add_trace(go.Scatterternary(
      mode='markers',
      a=[row['perc_deep']],
      b=[row['perc_rem']],
      c=[row['perc_light']],
      text=row['Global_Deidentified'],
      marker=dict(
          symbol=marker,
          color=color,
          size=60,
          line=dict(color='blue', width=8)  # ← black edge
      ),
      name=row['Global_Deidentified'],
      showlegend=False   # Remove individual IDs legend
    ))
    elif row['Global_Deidentified'] == ID_focus2:
      fig.add_trace(go.Scatterternary(
      mode='markers',
      a=[row['perc_deep']],
      b=[row['perc_rem']],
      c=[row['perc_light']],
      text=row['Global_Deidentified'],
      marker=dict(
          symbol=marker,
          color=color,
          size=60,
          line=dict(color='blue', width=8)  # ← black edge
      ),
      name=row['Global_Deidentified'],
      showlegend=False   # Remove individual IDs legend
    ))

# Combine *everything* in one .update_layout(...) call
fig.update_layout(
    width=12*96,
    height=12*96,
    margin=dict(l=200, r=200, t=200, b=300),
    ternary=dict(
        sum=100,
        aaxis={
            **makeAxis('', 0, min_val=10),
            'dtick': 5,
            'ticksuffix': '%',
            'tickfont': {'size': 20, 'color': 'black'}
        },
        baxis={
            **makeAxis('', 60, min_val=10),
            'dtick': 5,
            'ticksuffix': '%',
            'tickfont': {'size': 20, 'color': 'black'}
        },
        caxis={
            **makeAxis('', -60, min_val=45),
            'dtick': 5,
            'ticksuffix': '%',
            'tickfont': {'size': 20, 'color': 'black'}
        }
    ),
    showlegend=False
)

fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',  # Makes the overall background transparent
)

fig.update_layout(
    font=dict(size=25),  # Global font size setting
    ternary=dict(
        aaxis={'tickfont': {'size': 120}},
        baxis={'tickfont': {'size': 120}},
        caxis={'tickfont': {'size': 120}}
    )
)

#fig.write_image(f'{OUTPUT_DIR}/Sleep_REM.png', width=3000, height=3000, scale=1)  # Equivalent to ~600 DPI
fig.show()


### 2.2 — Physio-behavioral phenotyping (Figure 5)


In [ ]:

physio_df = final_df_with_groups[['ID','Global_Deidentified', 'Stress_Group', 'Avg_VO2Max', 'Avg_deep_sleep_BR', 'Avg_SpO2',
       'Avg_Temperature', 'Avg_Steps_per_day', 'Avg_RHR', 'Avg_deep_sleep_HR',
       'Avg_deep_sleep_RMSSD', 'Avg_deep_sleep_LF/HF']]

nan_report = physio_df.isna().sum().to_frame('n_missing')
nan_report['pct_missing'] = nan_report['n_missing'] / len(physio_df) * 100

# 1. Rank columns by how many NaNs they have (worst first)
nan_per_col = physio_df.isna().sum().sort_values(ascending=False)
cols_sorted_by_nan = nan_per_col.index.tolist()

results = []  # (num_cols_kept, num_subjects_complete, cols_dropped)

# 2. For k columns dropped (0 .. all), compute how many complete subjects remain
for k in range(len(cols_sorted_by_nan) + 1):
    cols_to_drop = cols_sorted_by_nan[:k]
    cols_keep = [c for c in physio_df.columns if c not in cols_to_drop]

    n_subjects_complete = physio_df[cols_keep].dropna(axis=0, how='any').shape[0]
    results.append((len(cols_keep), n_subjects_complete, cols_to_drop))

# 3. Put results in a DataFrame for inspection
tradeoff_df = pd.DataFrame(results, columns=['n_cols_kept', 'n_subjects_complete', 'cols_dropped'])

# 4. Plot the trade-off curve
plt.figure(figsize=(8, 5))
plt.plot(tradeoff_df['n_cols_kept'], tradeoff_df['n_subjects_complete'], marker='o')
plt.xlabel('Number of columns kept')
plt.ylabel('Number of subjects with no NaNs')
plt.title('Trade-off: columns kept vs complete subjects')
plt.grid(True)
plt.tight_layout()
plt.show()

best_row = tradeoff_df.iloc[1]
cols_to_drop = best_row['cols_dropped']
physio_df_clean = physio_df.drop(columns=cols_to_drop).dropna(axis=0, how='any')

len(physio_df_clean)

In [ ]:
# 1) Map STAI to categories
def get_color(val):
    if val == 'Below Average':
        return 'green'
    elif val == 'Average':
        return 'yellow'
    else:  # Above average
        return 'red'

def plot_kmeans_pca_with_stress_color_2(
    physio_df_clean, cluster_markers, axs, ax_position,
    scatter_size=100,
    grid_linewidth=0.8,
    grid_alpha=0.7,
    fontsize = 20
):
    """
    Generates a K-Means/PCA plot where subjects are colored by their median stress level.
    """
    # --------------------
    # Data Preparation (No changes here)
    # --------------------

    features = physio_df_clean[['Avg_VO2Max',
            'Avg_Temperature', 'Avg_Steps_per_day', 'Avg_RHR',
          'Avg_deep_sleep_HR', 'Avg_deep_sleep_RMSSD', 'Avg_deep_sleep_LF/HF']]

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)

    # K-Means Clustering
    kmeans = KMeans(n_clusters=3, random_state=42)
    physio_df_clean['Cluster'] = kmeans.fit_predict(scaled_features)

    # PCA Transformation
    pca = PCA(n_components=2)
    pca_components = pca.fit_transform(scaled_features)
    pca_fit = pca.fit(scaled_features)
    physio_df_clean['PC1'] = pca_components[:, 0] * -1
    physio_df_clean['PC2'] = pca_components[:, 1]

    labels = physio_df_clean['Cluster'].values

    silhouette_per_point = silhouette_samples(features, labels)
    silhouette_avg = silhouette_per_point.mean()
    silhouette_min = silhouette_per_point.min()
    silhouette_max = silhouette_per_point.max()
    silhouette_med = np.median(silhouette_per_point)

    print(f"Average silhouette: {silhouette_avg:.3f}")
    print(f"Silhouette range: {silhouette_min:.3f} to {silhouette_max:.3f}")
    print(f"Silhouette median: {silhouette_med:.3f}")

    evr = pca.explained_variance_ratio_

    print(f"Explained variance PC1: {evr[0]:.3f}")
    print(f"Explained variance PC2: {evr[1]:.3f}")
    print(f"Total explained by first 2 PCs: {evr.sum():.3f}")

    # Map each subject to a marker based on its cluster for later emphasis
    id_to_marker = {row['Global_Deidentified']: cluster_markers[row['Cluster']]
                    for _, row in physio_df_clean.iterrows()}

    # Apply the color function to create a new column for highlighting
    physio_df_clean['stress_color'] = physio_df_clean['Stress_Group'].apply(get_color)

    # --------------------
    # Plotting Clusters
    # --------------------
    # 1. Plot ALL subjects in grayscale first to serve as a background
    cluster_shades = {0: '#AAAAAA', 1: '#666666', 2: '#222222'}
    for cluster, marker in cluster_markers.items():
        cluster_data = physio_df_clean[physio_df_clean['Cluster'] == cluster]
        shade = cluster_shades[cluster]
        edge = 'black' if cluster == 0 else 'none'
        axs[ax_position].scatter(
            cluster_data['PC1'], cluster_data['PC2'],
            marker=marker, s=scatter_size, facecolors=shade,
            edgecolors=edge, linewidths=1, zorder=1
        )

    # 2. Plot ONLY the subjects with stress data on TOP, using their stress color
    highlight_data = physio_df_clean.dropna(subset=['stress_color'])
    for cluster, marker in cluster_markers.items():
        # Further filter by cluster to use the correct marker
        cluster_highlight_data = highlight_data[highlight_data['Cluster'] == cluster]
        if not cluster_highlight_data.empty:
            axs[ax_position].scatter(
                cluster_highlight_data['PC1'],
                cluster_highlight_data['PC2'],
                marker=marker,
                s=scatter_size,
                facecolors=cluster_highlight_data['stress_color'], # Use the color from the column
                edgecolors='black',
                linewidths=0.5,
                zorder=5  # Higher zorder ensures these points are on top
            )

    # --------------------
    # Plotting PC Loadings (Arrows) - No changes here
    # --------------------
    loadings = pca_fit.components_[:2, :]
    loadings[0, :] *= -1
    feature_names = features.columns
    scores = physio_df_clean[['PC1', 'PC2']].values
    max_val = np.max(np.abs(scores))
    arrow_scale = 0.13 * max_val
    x_origin, y_origin = -3, -2.8
    for i, feature in enumerate(feature_names):
        arrow_x = loadings[0, i] * arrow_scale
        arrow_y = loadings[1, i] * arrow_scale
        axs[ax_position].arrow(x_origin, y_origin, arrow_x, arrow_y,
                               head_width=0.02*max_val, head_length=0.02*max_val,
                               fc='white', ec='black', zorder=6)

    # --------------------
    # Final Touches and Legend
    # --------------------
    axs[ax_position].grid(True, linestyle='--', alpha=grid_alpha, linewidth=grid_linewidth)
    axs[ax_position].tick_params(axis='both', which='major', labelsize=20)

    edgecolor = plt.rcParams['legend.edgecolor']
    bbox = FancyBboxPatch((-3.5, -3), 0.8, 0.8, fc='white', ec=edgecolor, lw=0.8, zorder=5, alpha=0.8)
    axs[ax_position].add_patch(bbox)

    # --- CHANGE: Create a new legend for the stress colors ---
    handles = []
    for i,m in enumerate(cluster_markers.values()):
        handles.append( plt.Line2D([], [], linestyle='None', marker=m,
                                  markersize=12, markeredgecolor='black',
                                  markerfacecolor='black', label=f'Cluster {i+1}') )
    axs[ax_position].legend(handles=handles, title='Subject',loc='lower right')

    expl_var = pca_fit.explained_variance_ratio_

    x_label = f"PC1 ({expl_var[0]*100:.1f}% explained)"
    y_label = f"PC2 ({expl_var[1]*100:.1f}% explained)"

    axs[ax_position].set_xlabel(x_label,fontsize=fontsize)
    axs[ax_position].set_ylabel(y_label,fontsize=fontsize)

    #axs[ax_position].set_xlim(-4, 3)

    # The function can still return the original data if needed, but the primary output is the plot
    return physio_df_clean, scaler, kmeans, pca, pca_fit, id_to_marker, scaled_features

#### HRV RMSSD / LF-HF


In [ ]:
def plot_HRV_ratio(physio_df_clean, axs, ax_position,
                                   input_ids = [],
                                   scatter_size=100,
                                   grid_linewidth=0.8,
                                   grid_alpha = 0.7,
                         fontsize =20):

    ax = axs[ax_position]
    x = physio_df_clean['Avg_deep_sleep_LF/HF']
    y = physio_df_clean['Avg_deep_sleep_RMSSD']
    markers = physio_df_clean['Cluster'].map(cluster_markers)  # example column

    for m in markers.unique():
        mask = markers == m
        ax.scatter(
            x[mask], y[mask],
            s=scatter_size, alpha=1,
            color=physio_df_clean.loc[mask, 'Stress_Group'].apply(get_color),
            marker=m, label=m, edgecolor = 'black'
        )

    ax.tick_params(axis='both', which='major', labelsize=fontsize)

    ax.set_xlabel('Deep Sleep Average LF/HF [-]', fontsize=fontsize)
    ax.set_ylabel('Deep Sleep Average RMSSD [ms]', fontsize=fontsize)

    ax.grid(which='major', linewidth=grid_linewidth, alpha=grid_alpha)
    ax.grid(which='minor', linewidth=grid_linewidth/2, alpha=grid_alpha/2)
    ax.minorticks_on()

    # spacings — adjust freely
    ax.xaxis.set_major_locator(plt.MultipleLocator(1))
    ax.yaxis.set_major_locator(plt.MultipleLocator(20))
    ax.xaxis.set_minor_locator(plt.MultipleLocator(0.5))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(5))

    # manual legend
    handles = []
    for i,m in enumerate(cluster_markers.values()):
        handles.append( plt.Line2D([], [], linestyle='None', marker=m,
                                  markersize=12, markeredgecolor='black',
                                  markerfacecolor='black', label=f'Cluster {i+1}') )
    ax.legend(handles=handles, title='Subject')

    plt.tight_layout()
    return

#### HRV RMSSD / HR


In [ ]:
def plot_HRV_ratio_HR(physio_df_clean, axs, ax_position,
                                   input_ids = [],
                                   scatter_size=100,
                                   grid_linewidth=0.8,
                                   grid_alpha = 0.7,
                         fontsize =20):

    ax = axs[ax_position]
    x = physio_df_clean['Avg_deep_sleep_HR']
    y = physio_df_clean['Avg_deep_sleep_RMSSD']
    markers = physio_df_clean['Cluster'].map(cluster_markers)  # example column

    for m in markers.unique():
        mask = markers == m
        ax.scatter(
            x[mask], y[mask],
            s=scatter_size, alpha=1,
            color=physio_df_clean.loc[mask, 'Stress_Group'].apply(get_color),
            marker=m, label=m, edgecolor = 'black'
        )

    ax.tick_params(axis='both', which='major', labelsize=fontsize)

    ax.set_xlabel('Deep Sleep Average Heart Rate [bpm]', fontsize=fontsize)
    ax.set_ylabel('Deep Sleep Average RMSSD [ms]', fontsize=fontsize)

    ax.grid(which='major', linewidth=grid_linewidth, alpha=grid_alpha)
    ax.grid(which='minor', linewidth=grid_linewidth/2, alpha=grid_alpha/2)
    ax.minorticks_on()

    # spacings — adjust freely
    ax.xaxis.set_major_locator(plt.MultipleLocator(5))
    ax.yaxis.set_major_locator(plt.MultipleLocator(20))
    ax.xaxis.set_minor_locator(plt.MultipleLocator(1))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(5))

    # manual legend
    handles = []
    for i,m in enumerate(cluster_markers.values()):
        handles.append( plt.Line2D([], [], linestyle='None', marker=m,
                                  markersize=12, markeredgecolor='black',
                                  markerfacecolor='black', label=f'Cluster {i+1}') )
    ax.legend(handles=handles, title='Subject')

    plt.tight_layout()
    return

#### Steps / Vo2Max


In [ ]:
def plot_Steps_VO2MAx(physio_df_clean, axs, ax_position,
                                   input_ids = [],
                                   scatter_size=100,
                                   grid_linewidth=0.8,
                                   grid_alpha = 0.7,
                         fontsize =20):

    ax = axs[ax_position]
    x = physio_df_clean['Avg_VO2Max']
    y = physio_df_clean['Avg_Steps_per_day']
    markers = physio_df_clean['Cluster'].map(cluster_markers)  # example column

    for m in markers.unique():
        mask = markers == m
        ax.scatter(
            x[mask], y[mask],
            s=scatter_size, alpha=1,
            color=physio_df_clean.loc[mask, 'Stress_Group'].apply(get_color),
            marker=m, label=m, edgecolor = 'black'
        )

    ax.tick_params(axis='both', which='major', labelsize=fontsize)

    ax.set_xlabel('Average VO2Max [mL/kg/min]', fontsize=fontsize)
    ax.set_ylabel('Average Steps per Day', fontsize=fontsize)

    ax.grid(which='major', linewidth=grid_linewidth, alpha=grid_alpha)
    ax.grid(which='minor', linewidth=grid_linewidth/2, alpha=grid_alpha/2)
    ax.minorticks_on()

    # spacings — adjust freely
    ax.xaxis.set_major_locator(plt.MultipleLocator(5))
    ax.yaxis.set_major_locator(plt.MultipleLocator(2000))
    ax.xaxis.set_minor_locator(plt.MultipleLocator(1))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(1000))

    # manual legend
    handles = []
    for i,m in enumerate(cluster_markers.values()):
        handles.append( plt.Line2D([], [], linestyle='None', marker=m,
                                  markersize=12, markeredgecolor='black',
                                  markerfacecolor='black', label=f'Cluster {i+1}') )
    ax.legend(handles=handles, title='Subject')

    plt.tight_layout()
    return

#### Physio-behavioral figure (Figure 5)


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(24, 16))

physio_df_clean_cluster, scaler, kmeans, pca, pca_fit, id_to_marker, scaled_features = plot_kmeans_pca_with_stress_color_2(physio_df_clean, cluster_markers, axs, (0, 0))
plot_Steps_VO2MAx(physio_df_clean, axs, (0, 1))
plot_HRV_ratio(physio_df_clean, axs, (1, 0))
plot_HRV_ratio_HR(physio_df_clean, axs, (1, 1))
#plt.savefig(f'{OUTPUT_DIR}/physio.png', dpi=600,transparent=True)
#plt.show()

len(physio_df_clean)

In [ ]:
df_demo = pd.read_parquet(f'{INPUT_DIR}/fitbit_Profile.parquet')

df_gender = df_demo[['id','bmi','age','gender']]
df_gender = df_gender.rename(columns={'id': 'Global_Deidentified'})

df_gender_physio = physio_df_clean.merge(df_gender,on='Global_Deidentified',how='left')
len(df_gender_physio)# Imports
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA# Cluster markers (LOCKED)
cluster_markers = {0: 'o', 1: '^', 2: 's'}# Gender color map (canonical)
GENDER_COLORS = {
    'male':   '#1f77b4',   # blue
    'female': '#e377c2'    # pink
}
DEFAULT_COLOR = '#BBBBBB'# Normalize gender ONCE
def normalize_gender(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({
            'm': 'male',
            'f': 'female',
            'nan': np.nan,
            'none': np.nan
        })
    )# PCA + KMeans plot
def plot_kmeans_pca_gender(
    df, axs, ax_position,
    scatter_size=100, grid_linewidth=0.8, grid_alpha=0.7, fontsize=20
):

    df = df.copy()
    df['gender_norm'] = normalize_gender(df['gender'])

    features = df[['Avg_VO2Max',
                   'Avg_Temperature',
                   'Avg_Steps_per_day',
                   'Avg_RHR',
                   'Avg_deep_sleep_HR',
                   'Avg_deep_sleep_RMSSD',
                   'Avg_deep_sleep_LF/HF']]

    X = StandardScaler().fit_transform(features)

    df['Cluster'] = KMeans(n_clusters=3, random_state=42).fit_predict(X)

    pcs = PCA(n_components=2).fit_transform(X)
    df['PC1'] = -pcs[:, 0]
    df['PC2'] =  pcs[:, 1]

    ax = axs[ax_position]

    # ---- background (cluster geometry)
    cluster_shades = {0:'#AAAAAA', 1:'#666666', 2:'#222222'}
    for c, m in cluster_markers.items():
        d = df[df['Cluster'] == c]
        ax.scatter(
            d['PC1'], d['PC2'],
            marker=m, s=scatter_size,
            facecolor=cluster_shades[c],
            edgecolor='none', zorder=1
        )

    # ---- foreground (gender color, FIXED)
    for c, m in cluster_markers.items():
        d = df[df['Cluster'] == c]
        colors = d['gender_norm'].map(GENDER_COLORS).fillna(DEFAULT_COLOR)

        ax.scatter(
            d['PC1'], d['PC2'],
            marker=m, s=scatter_size,
            facecolor=colors,
            edgecolor='black', linewidth=0.6, zorder=5
        )

    ax.grid(True, linestyle='--', linewidth=grid_linewidth, alpha=grid_alpha)
    ax.tick_params(axis='both', labelsize=fontsize)

    return df# Generic scatter (gender colored)
def scatter_by_cluster_gender(
    df, xcol, ycol, ax,
    scatter_size=100, fontsize=20,
    grid_linewidth=0.8, grid_alpha=0.7
):

    for c, m in cluster_markers.items():
        d = df[df['Cluster'] == c]
        colors = d['gender_norm'].map(GENDER_COLORS).fillna(DEFAULT_COLOR)

        ax.scatter(
            d[xcol], d[ycol],
            marker=m, s=scatter_size,
            facecolor=colors,
            edgecolor='black'
        )

    ax.tick_params(axis='both', labelsize=fontsize)
    ax.grid(True, linewidth=grid_linewidth, alpha=grid_alpha)
    ax.minorticks_on()
fig, axs = plt.subplots(2, 2, figsize=(24, 16))

df_gender_physio = plot_kmeans_pca_gender(
    df_gender_physio, axs, (0, 0)
)

scatter_by_cluster_gender(
    df_gender_physio,
    'Avg_VO2Max', 'Avg_Steps_per_day',
    axs[0,1]
)
axs[0,1].set_xlabel('Average VO2Max [mL/kg/min]', fontsize=20)
axs[0,1].set_ylabel('Average Steps per Day', fontsize=20)

scatter_by_cluster_gender(
    df_gender_physio,
    'Avg_deep_sleep_LF/HF', 'Avg_deep_sleep_RMSSD',
    axs[1,0]
)
axs[1,0].set_xlabel('Deep Sleep Average LF/HF [-]', fontsize=20)
axs[1,0].set_ylabel('Deep Sleep Average RMSSD [ms]', fontsize=20)

scatter_by_cluster_gender(
    df_gender_physio,
    'Avg_deep_sleep_HR', 'Avg_deep_sleep_RMSSD',
    axs[1,1]
)
axs[1,1].set_xlabel('Deep Sleep Average Heart Rate [bpm]', fontsize=20)
axs[1,1].set_ylabel('Deep Sleep Average RMSSD [ms]', fontsize=20)

plt.tight_layout()

## Section 3 — Linear mixed-effects modeling

Within-person association between S-STAI and wearable-derived physiology (z-scored to each participant's low-stress baseline; ±5-day window). See Methods Eq. 1.


In [ ]:
Sleep_df = pd.read_parquet(f'{INPUT_DIR}/Sleep_features.parquet')
df_SRI = pd.read_csv(f'{INPUT_DIR}/SRI_focused.csv')
VO2Max_df = pd.read_parquet(f'{INPUT_DIR}/VO2Max.parquet')
BR_df = pd.read_parquet(f'{INPUT_DIR}/BR.parquet')
SpO2_df = pd.read_parquet(f'{INPUT_DIR}/SpO2.parquet')
Temp_df = pd.read_parquet(f'{INPUT_DIR}/Temperature.parquet')
Steps_df = pd.read_parquet(f'{INPUT_DIR}/Steps.parquet')
RestingHR_df = pd.read_parquet(f'{INPUT_DIR}/RHR.parquet')
Temp_df = pd.read_parquet(f'{INPUT_DIR}/Temperature.parquet')
NightHR_Sleep_df = pd.read_parquet(f'{INPUT_DIR}/SleepHR_features.parquet')
HRV_df = pd.read_parquet(f'{INPUT_DIR}/HRV_features.parquet')
Sleep_df["Date"] = pd.to_datetime(Sleep_df["Date"]).dt.strftime('%m-%d-%Y')
SpO2_df["Date"] = pd.to_datetime(SpO2_df["datetime"]).dt.strftime('%m-%d-%Y')
Sleep_df['perc_deep']  = Sleep_df['deep_sleep']  / Sleep_df['Time_asleep'] * 100
Sleep_df['perc_rem']   = Sleep_df['rem_sleep']   / Sleep_df['Time_asleep'] * 100
Sleep_df['perc_light'] = (Sleep_df['Time_asleep'] - (Sleep_df['deep_sleep'] + Sleep_df['rem_sleep']))  / Sleep_df['Time_asleep'] * 100
df_anxiety_GP = df_anxiety_filtered[['Global_Deidentified','submitdate','stai_stress','category']]
df_anxiety_GP["Date"] = pd.to_datetime(df_anxiety_GP["submitdate"]).dt.strftime('%m-%d-%Y')
Temp_df["Date"] = pd.to_datetime(Temp_df["sleep_start"]).dt.strftime('%m-%d-%Y')
RestingHR_df = RestingHR_df.rename(columns={'Resting Heart Rate': 'RHR'})
SpO2_df = SpO2_df.rename(columns={'Value': 'SpO2'})

def get_color(val):
    if 20 <= val <= 45:
        return 'green'
    elif 45 < val <= 50:
        return 'yellow'
    else:
        return 'red'
# 1. Ensure datetime columns
df_anxiety_min_2_samples['submitdate'] = pd.to_datetime(df_anxiety_min_2_samples['submitdate'])
df_anxiety_min_2_samples['Date'] = df_anxiety_min_2_samples['submitdate'].dt.date
df_anxiety_min_2_samples['Date'] = pd.to_datetime(df_anxiety_min_2_samples['Date'])

Sleep_df['Date'] = pd.to_datetime(Sleep_df['Date'], errors='coerce')
NightHR_Sleep_df['Date'] = pd.to_datetime(NightHR_Sleep_df['Date'], errors='coerce')
HRV_df['Date'] = pd.to_datetime(HRV_df['Date'], errors='coerce')
SpO2_df['Date'] = pd.to_datetime(SpO2_df['Date'], errors='coerce')
BR_df['Date'] = pd.to_datetime(BR_df['Date'], errors='coerce')
Steps_df['Date'] = pd.to_datetime(Steps_df['Date'], errors='coerce')
RestingHR_df['Date'] = pd.to_datetime(RestingHR_df['Date'], errors='coerce')
Temp_df['Date'] = pd.to_datetime(Temp_df['Date'], errors='coerce')
# 2. Select columns we need for each dataset
sleep_keep = Sleep_df[['Global_Deidentified','Date','Time_asleep','light_sleep','deep_sleep','rem_sleep','Awake_count']]
hr_keep = NightHR_Sleep_df[['Global_Deidentified','Date','deep_mean_hr']]
hrv_keep = HRV_df[['Global_Deidentified','Date','deep_RMSSD_mean','deep_LF_HF_Ratio_mean']]
spo2_keep = SpO2_df[['Global_Deidentified','Date','SpO2']]
br_keep = BR_df[['Global_Deidentified','Date','deep_sleep_breathing_rate']]
steps_keep = Steps_df[['Global_Deidentified','Date','Total_Number_of_Steps']]
anxiety_keep = df_anxiety_min_2_samples[['Global_Deidentified','Date','stai_stress']]
Temp_keep = Temp_df[['Global_Deidentified','Date','nightly_temperature']]
# 3. Merge all datasets across subjects
dfs = [
    sleep_keep,
    hr_keep,
    hrv_keep,
    spo2_keep,
    br_keep,
    steps_keep,
    RestingHR_df,
    anxiety_keep,
    Temp_keep
]

full_df = reduce(
    lambda l, r: pd.merge(l, r, on=['Global_Deidentified', 'Date'], how='outer'),
    dfs
)
# 4. Final processing
full_df = full_df.sort_values(['Global_Deidentified','Date'])
full_df['stress_color'] = full_df['stai_stress'].apply(get_color)


### 3.1 — LMM results (Table 2)


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

signals_to_run = [
    'deep_mean_hr',
    'deep_RMSSD_mean',
    'deep_LF_HF_Ratio_mean',
    'Time_asleep',
    'deep_sleep_breathing_rate',
    'Total_Number_of_Steps',
    'RHR',
    'light_sleep',
    'deep_sleep',
    'rem_sleep',
    'Awake_count',
    'nightly_temperature',
    'SpO2'
]

results_rows = []
all_ids = ids_to_work_with

window_days = 5
baseline_window_days = 5
use_zscore = True

for metric in signals_to_run:

    window_col = f"{metric}_avg_window_{window_days}d"
    z_col = f"{window_col}_z"

    rows = []

    for sid in all_ids:

        subj_stress = (
            df_anxiety_min_2_samples
            .loc[df_anxiety_min_2_samples['Global_Deidentified'] == sid,
                 ['Date', 'stai_stress']]
            .sort_values('Date')
        )

        subj_data = (
            full_df
            .loc[full_df['Global_Deidentified'] == sid,
                 ['Date', metric]]
            .dropna()
        )

        if subj_stress.empty or subj_data.empty:
            continue

        # --- baseline = lowest stress ---
        idx_min = subj_stress['stai_stress'].idxmin()
        baseline_date = subj_stress.loc[idx_min, 'Date']

        base_win = subj_data[
            (subj_data['Date'] >= baseline_date - pd.Timedelta(days=baseline_window_days)) &
            (subj_data['Date'] <= baseline_date + pd.Timedelta(days=baseline_window_days))
        ]

        if base_win.empty or base_win[metric].std() == 0 or len(base_win) < 3:
            continue

        baseline_mean = base_win[metric].mean()
        baseline_std  = base_win[metric].std()

        for _, row in subj_stress.iterrows():

            stress_date = row['Date']
            stress_score = row['stai_stress']

            if stress_date == baseline_date:
                continue

            win = subj_data[
                (subj_data['Date'] >= stress_date - pd.Timedelta(days=window_days)) &
                (subj_data['Date'] <= stress_date + pd.Timedelta(days=window_days))
            ]

            if win.empty:
                continue

            avg_val = win[metric].mean()
            z_val = (avg_val - baseline_mean) / baseline_std

            rows.append({
                "Global_Deidentified": sid,
                "stai_stress": stress_score,
                z_col: z_val
            })

    # -------------------------------------------------
    # Assemble + enforce ≥2 observations per participant
    # -------------------------------------------------
    window_df = pd.DataFrame(rows).dropna()

    if window_df.empty:
        continue

    counts = window_df.groupby("Global_Deidentified").size()
    valid_ids = counts[counts >= 2].index

    window_df = window_df[window_df["Global_Deidentified"].isin(valid_ids)]

    if window_df.empty:
        continue

    # -------------------------------------------------
    # LMM
    # -------------------------------------------------
    try:
        model = smf.mixedlm(
            f"{z_col} ~ stai_stress",
            data=window_df,
            groups=window_df["Global_Deidentified"],
            re_formula="~stai_stress"
        )
        res = model.fit(method="lbfgs")

    except Exception:
        model = smf.mixedlm(
            f"{z_col} ~ stai_stress",
            data=window_df,
            groups=window_df["Global_Deidentified"],
            re_formula="1"
        )
        res = model.fit(method="lbfgs")

    # -------------------------------------------------
    # Extract results
    # -------------------------------------------------
    beta = res.params.get("stai_stress", np.nan)
    se   = res.bse.get("stai_stress", np.nan)

    if "stai_stress" in res.conf_int().index:
        ci_l, ci_u = res.conf_int().loc["stai_stress"].values
    else:
        ci_l, ci_u = np.nan, np.nan

    pval = res.pvalues.get("stai_stress", np.nan)

    results_rows.append({
        "Metric": metric,
        "Participants_n": window_df["Global_Deidentified"].nunique(),
        "Observations_n": len(window_df),
        "Beta": beta,
        "CI_lower": ci_l,
        "CI_upper": ci_u,
        "p_value": pval
    })

# -------------------------------------------------
# Build final table
# -------------------------------------------------
df_lmm_5d = pd.DataFrame(results_rows)

metric_order = [
    'Total_Number_of_Steps',
    'RHR',
    'deep_mean_hr',
    'deep_RMSSD_mean',
    'deep_LF_HF_Ratio_mean',
    'deep_sleep_breathing_rate',
    'SpO2',
    'nightly_temperature',
    'Time_asleep',
    'deep_sleep',
    'rem_sleep',
    'light_sleep',
    'Awake_count',
]

df_lmm_5d["Metric"] = pd.Categorical(
    df_lmm_5d["Metric"],
    categories=metric_order,
    ordered=True
)

df_lmm_5d = df_lmm_5d.sort_values("Metric").reset_index(drop=True)

df_lmm_5d["Beta (95% CI)"] = (
    df_lmm_5d["Beta"].round(3).astype(str) + " [" +
    df_lmm_5d["CI_lower"].round(3).astype(str) + ", " +
    df_lmm_5d["CI_upper"].round(3).astype(str) + "]"
)

df_lmm_5d["p_value"] = df_lmm_5d["p_value"].round(3)

df_lmm_5d = df_lmm_5d[
    ["Metric", "Participants_n", "Observations_n", "Beta (95% CI)", "p_value"]
]

df_lmm_5d

In [ ]:
# Running on all possible +- days

import statsmodels.formula.api as smf

signals_to_run = [
    'deep_mean_hr',
    'deep_RMSSD_mean',
    'deep_LF_HF_Ratio_mean',
    'Time_asleep',
    'deep_sleep_breathing_rate',
    'Total_Number_of_Steps',
    'RHR',
    'light_sleep','deep_sleep','rem_sleep','Awake_count','nightly_temperature','SpO2'
]

baseline_window_days = 5
window_days_list = [0,1,2,3,4,5]
use_zscore = True

all_ids = ids_to_work_with

# Ensure datetime for anxiety surveys
df_anxiety_min_2_samples['Date'] = (
    pd.to_datetime(df_anxiety_min_2_samples['submitdate']).dt.date
)
df_anxiety_min_2_samples['Date'] = pd.to_datetime(df_anxiety_min_2_samples['Date'])

for window_days in window_days_list:
  print(f"Running Window size: {window_days}")
  # ================================================================
  # LOOP OVER ALL SIGNALS
  # ================================================================
  for metric in signals_to_run:

      print("\n" + "="*80)
      print(f"Running metric: {metric}")
      print("="*80)

      window_col = f"{metric}_avg_window_{window_days}d"
      z_col = f"{window_col}_z"

      # ------------------------------------------
      # Build rows for this metric
      # ------------------------------------------
      rows = []

      for sid in all_ids:

          # SUBJECT-STRESS DATA
          subj_stress = df_anxiety_min_2_samples[
              df_anxiety_min_2_samples['Global_Deidentified'] == sid
          ].sort_values('Date')[['Date', 'stai_stress']]

          if subj_stress.empty:
              continue

          # SUBJECT-METRIC DATA
          subj_data = full_df[
              full_df['Global_Deidentified'] == sid
          ][['Date', metric]].dropna()

          if subj_data.empty:
              continue

          # -----------------------------------------------------
          # BASELINE WINDOW (LOWEST-STRESS)
          # -----------------------------------------------------
          baseline_mean = np.nan
          baseline_std = np.nan
          baseline_date = None

          if use_zscore:

              idx_min = subj_stress['stai_stress'].idxmin()
              baseline_date = subj_stress.loc[idx_min, 'Date']

              start_b = baseline_date - pd.Timedelta(days=baseline_window_days)
              end_b   = baseline_date + pd.Timedelta(days=baseline_window_days)

              base_win = subj_data[
                  (subj_data['Date'] >= start_b) &
                  (subj_data['Date'] <= end_b)
              ]

              if not base_win.empty and len(base_win) > 2:
                  baseline_mean = base_win[metric].mean()
                  baseline_std  = base_win[metric].std()

                  if baseline_std == 0:
                      baseline_std = np.nan  # avoid divide-by-zero

          # -----------------------------------------------------
          # WINDOW AVERAGING FOR EACH STRESS DATE
          # -----------------------------------------------------
          for _, row in subj_stress.iterrows():

              stress_date = row["Date"]
              stress_score = row["stai_stress"]

              start = stress_date - pd.Timedelta(days=window_days)
              end   = stress_date + pd.Timedelta(days=window_days)

              win = subj_data[
                  (subj_data['Date'] >= start) &
                  (subj_data['Date'] <= end)
              ]

              avg_val = win[metric].mean() if len(win) > 0 else np.nan

              out = {
                  "Global_Deidentified": sid,
                  "stress_date": stress_date,
                  "stai_stress": stress_score,
                  window_col: avg_val,
                  "is_baseline": (stress_date == baseline_date)
              }

              if use_zscore:
                  if not np.isnan(baseline_mean) and not np.isnan(baseline_std):
                      out[z_col] = (avg_val - baseline_mean) / baseline_std
                  else:
                      out[z_col] = np.nan

              rows.append(out)

      # ------------------------------------------
      # Assemble window_df for this metric
      # ------------------------------------------
      window_df = pd.DataFrame(rows)
      window_df = window_df.sort_values(["Global_Deidentified", "stress_date"])

      # Remove baseline rows if z-scored
      if use_zscore:
          window_df = window_df[window_df["is_baseline"] == False].reset_index(drop=True)

      # ------------------------------------------
      # STATS: PARTICIPANTS AND SAMPLES
      # ------------------------------------------
      n_participants = window_df["Global_Deidentified"].nunique()
      #print(f"\nParticipants in model: {n_participants}")

      counts = window_df.groupby("Global_Deidentified").size()
      print("\nPoints per participant:")
      #print(counts)

      print("\nSummary:")
      #print(counts.describe())

      # ------------------------------------------
      # LMM: RANDOM INTERCEPT + SLOPE
      # ------------------------------------------
      print("\nFitting LMM...")

      clean_df = window_df[[z_col, "stai_stress", "Global_Deidentified"]].dropna()

      # -------------------------------------------------
      # KEEP ONLY PARTICIPANTS WITH ≥2 WEARABLE OBSERVATIONS
      # -------------------------------------------------
      counts = clean_df.groupby("Global_Deidentified").size()
      valid_ids = counts[counts >= 2].index

      clean_df = clean_df[clean_df["Global_Deidentified"].isin(valid_ids)]

      print("\nClean df shape:", clean_df.shape)

      formula = f"{z_col} ~ stai_stress"

      try:
          model_rs = smf.mixedlm(
              formula,
              data=clean_df,
              groups=clean_df["Global_Deidentified"],
              re_formula="~stai_stress"
          )
          res = model_rs.fit(method="lbfgs")
          print(res.summary())
      except Exception as e:
          print("Random slope model failed, switching to random intercept only.")
          model_ri = smf.mixedlm(
              formula,
              data=clean_df,
              groups=clean_df["Global_Deidentified"],
              re_formula="1"
          )
          res = model_ri.fit(method="lbfgs")
          print(res.summary())

## Section 4 — Dynamic stress–physiology association modeling

Unsupervised analysis of stress-centered wearable episodes: PCA on the 131 × 6 z-matrix, then GMM model selection by BIC, then UMAP for visualization.

### 4.1 — Build the stress-centered feature matrix


In [ ]:
Sleep_df = pd.read_parquet(f'{INPUT_DIR}/Sleep_features.parquet')
df_SRI = pd.read_csv(f'{INPUT_DIR}/SRI_focused.csv')
VO2Max_df = pd.read_parquet(f'{INPUT_DIR}/VO2Max.parquet')
BR_df = pd.read_parquet(f'{INPUT_DIR}/BR.parquet')
SpO2_df = pd.read_parquet(f'{INPUT_DIR}/SpO2.parquet')
Steps_df = pd.read_parquet(f'{INPUT_DIR}/Steps.parquet')
RestingHR_df = pd.read_parquet(f'{INPUT_DIR}/RHR.parquet')
NightHR_Sleep_df = pd.read_parquet(f'{INPUT_DIR}/SleepHR_features.parquet')
HRV_df = pd.read_parquet(f'{INPUT_DIR}/HRV_features.parquet')
Sleep_df["Date"] = pd.to_datetime(Sleep_df["Date"]).dt.strftime('%m-%d-%Y')
SpO2_df["Date"] = pd.to_datetime(SpO2_df["datetime"]).dt.strftime('%m-%d-%Y')
Sleep_df['perc_deep']  = Sleep_df['deep_sleep']  / Sleep_df['Time_asleep'] * 100
Sleep_df['perc_rem']   = Sleep_df['rem_sleep']   / Sleep_df['Time_asleep'] * 100
Sleep_df['perc_light'] = (Sleep_df['Time_asleep'] - (Sleep_df['deep_sleep'] + Sleep_df['rem_sleep']))  / Sleep_df['Time_asleep'] * 100
df_anxiety_GP = df_anxiety_filtered[['Global_Deidentified','submitdate','stai_stress','category']]
df_anxiety_GP["Date"] = pd.to_datetime(df_anxiety_GP["submitdate"]).dt.strftime('%m-%d-%Y')
RestingHR_df = RestingHR_df.rename(columns={'Resting Heart Rate': 'RHR'})

def get_color(val):
    if 20 <= val <= 45:
        return 'green'
    elif 45 < val <= 50:
        return 'yellow'
    else:
        return 'red'
# 1. Ensure datetime columns
df_anxiety_min_2_samples['submitdate'] = pd.to_datetime(df_anxiety_min_2_samples['submitdate'])
df_anxiety_min_2_samples['Date'] = df_anxiety_min_2_samples['submitdate'].dt.date
df_anxiety_min_2_samples['Date'] = pd.to_datetime(df_anxiety_min_2_samples['Date'])

Sleep_df['Date'] = pd.to_datetime(Sleep_df['Date'], errors='coerce')
NightHR_Sleep_df['Date'] = pd.to_datetime(NightHR_Sleep_df['Date'], errors='coerce')
HRV_df['Date'] = pd.to_datetime(HRV_df['Date'], errors='coerce')
SpO2_df['Date'] = pd.to_datetime(SpO2_df['Date'], errors='coerce')
BR_df['Date'] = pd.to_datetime(BR_df['Date'], errors='coerce')
Steps_df['Date'] = pd.to_datetime(Steps_df['Date'], errors='coerce')
RestingHR_df['Date'] = pd.to_datetime(RestingHR_df['Date'], errors='coerce')
# 2. Select columns we need for each dataset
sleep_keep = Sleep_df[['Global_Deidentified','Date','Time_asleep','light_sleep','deep_sleep','rem_sleep','Awake_count']]
hr_keep = NightHR_Sleep_df[['Global_Deidentified','Date','deep_mean_hr']]
hrv_keep = HRV_df[['Global_Deidentified','Date','deep_RMSSD_mean','deep_LF_HF_Ratio_mean']]
spo2_keep = SpO2_df[['Global_Deidentified','Date','Value']]
br_keep = BR_df[['Global_Deidentified','Date','deep_sleep_breathing_rate']]
steps_keep = Steps_df[['Global_Deidentified','Date','Total_Number_of_Steps']]
anxiety_keep = df_anxiety_min_2_samples[['Global_Deidentified','Date','stai_stress']]
# 3. Merge all datasets across subjects
dfs = [
    sleep_keep,
    hr_keep,
    hrv_keep,
    spo2_keep,
    br_keep,
    steps_keep,
    RestingHR_df,
    anxiety_keep
]

full_df = reduce(
    lambda l, r: pd.merge(l, r, on=['Global_Deidentified', 'Date'], how='outer'),
    dfs
)
# 4. Final processing
full_df = full_df.sort_values(['Global_Deidentified','Date'])
full_df['stress_color'] = full_df['stai_stress'].apply(get_color)

# This is the master dataset


In [ ]:
# --------------------------------
# Define feature order (grouped)
# --------------------------------
ordered_cols = [
    # --- Sleep duration / architecture ---
    'Time_asleep',
    'light_sleep',
    'deep_sleep',
    'rem_sleep',
    'Awake_count',

    # --- Cardiac / autonomic (deep sleep) ---
    'deep_mean_hr',
     'RHR',
    'deep_RMSSD_mean',
    'deep_LF_HF_Ratio_mean',
    'deep_sleep_breathing_rate',

    # --- Activity / baseline ---
    'Total_Number_of_Steps',
]

# --------------------------------
# Publication-ready labels
# --------------------------------
label_map = {
    'Time_asleep': 'Total Sleep Duration',
    'light_sleep': 'Light Sleep Duration',
    'deep_sleep': 'Deep Sleep Duration',
    'rem_sleep': 'REM Sleep Duration',
    'Awake_count': 'Wake Times After Sleep Onset (count)',
    'deep_mean_hr': 'Deep Sleep Heart Rate',
    'deep_RMSSD_mean': 'Deep Sleep RMSSD',
    'deep_LF_HF_Ratio_mean': 'Deep Sleep LF/HF',
    'deep_sleep_breathing_rate': 'Deep Sleep Breathing Rate',
    'Total_Number_of_Steps': 'Daily Steps',
    'RHR': 'Resting Heart Rate'
}

# --------------------------------
# Subset + order numeric data
# --------------------------------
df_num = (
    full_df[ordered_cols]
    .select_dtypes(include='number')
)

# --------------------------------
# Correlation (ordered)
# --------------------------------
corr = df_num.corr(method='pearson')
corr = corr.loc[ordered_cols, ordered_cols]
corr = corr.rename(index=label_map, columns=label_map)

# --------------------------------
# Plot
# --------------------------------
plt.figure(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,
    vmin=-1, vmax=1, center=0,
    cmap='coolwarm',
    square=True,
    linewidths=0.5,
    annot=True, fmt=".2f",
    cbar_kws={'shrink': 0.8}
)

plt.title('Pairwise Pearson Correlations of Wearable-Derived Features')
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------
# PARAMETERS
# ----------------------------------------
# signals = [
#     'deep_mean_hr',
#     'deep_RMSSD_mean',
#     'deep_LF_HF_Ratio_mean',
#     'Time_asleep',
#     'Value',  # SpO2
#     'deep_sleep_breathing_rate',
#     'Total_Number_of_Steps'
# ]

all_ids = ids_to_work_with

signals = [
    'deep_mean_hr',
    'deep_RMSSD_mean',
    'deep_LF_HF_Ratio_mean',
    'Time_asleep',
    'deep_sleep_breathing_rate',
    'Total_Number_of_Steps',
]

baseline_window_days = 5
window_days = 5
use_zscore = True

# Ensure datetime
df_anxiety_min_2_samples['Date'] = pd.to_datetime(
    pd.to_datetime(df_anxiety_min_2_samples['submitdate']).dt.date
)

# ----------------------------------------
# BUILD BIG RESULTS LIST
# ----------------------------------------
rows = []
baseline_rows = []

for sid in all_ids:

    subj_stress = df_anxiety_min_2_samples[
        df_anxiety_min_2_samples['Global_Deidentified'] == sid
    ].sort_values('Date')[['Date', 'stai_stress']]

    if subj_stress.empty:
        continue

    # subject-level baseline per metric
    baseline = {}   # metric -> (mean, std, baseline_date)

    for metric in signals:

        subj_data = full_df[
            full_df['Global_Deidentified'] == sid
        ][['Date', metric]].dropna()

        if subj_data.empty:
            continue

        # lowest-STAI baseline
        idx_min = subj_stress['stai_stress'].idxmin()
        baseline_date = subj_stress.loc[idx_min, 'Date']

        start_b = baseline_date - pd.Timedelta(days=baseline_window_days)
        end_b   = baseline_date + pd.Timedelta(days=baseline_window_days)

        base_win = subj_data[
            (subj_data['Date'] >= start_b) &
            (subj_data['Date'] <= end_b)
        ]

        if len(base_win) == 0:
            baseline_mean = np.nan
            baseline_std  = np.nan
        if len(base_win) < 3:
          print(f'found small window for {sid} in metric {metric}')
        else:
            baseline_mean = base_win[metric].mean()
            baseline_std  = base_win[metric].std()
            if baseline_std == 0:
                baseline_std = np.nan

        baseline[metric] = (baseline_mean, baseline_std, baseline_date)

    # ----------------------------------------
    # For every stress date, compute all metrics
    # ----------------------------------------
    for _, row_s in subj_stress.iterrows():

        stress_date = row_s['Date']
        stress_score = row_s['stai_stress']

        out = {
            'Global_Deidentified': sid,
            'stress_date': stress_date,
            'stai_stress': stress_score
        }

        is_baseline_overall = False

        for metric in signals:

            # subject metric data
            subj_data = full_df[
                full_df['Global_Deidentified'] == sid
            ][['Date', metric]].dropna()

            if subj_data.empty:
                out[f'{metric}_z'] = np.nan
                continue

            # window
            start = stress_date - pd.Timedelta(days=window_days)
            end   = stress_date + pd.Timedelta(days=window_days)

            win = subj_data[
                (subj_data['Date'] >= start) &
                (subj_data['Date'] <= end)
            ]

             # LOGIC CHECK: Comparison window needs at least 1 measurement
            if len(win) >0 and len(win) < 2:
                print(f'ID {sid} has less than 3 days window for STAI on {stress_date}')

            avg_val = win[metric].mean() if len(win) > 0 else np.nan
            out[f'{metric}_avg'] = avg_val

            # z-score
            baseline_mean, baseline_std, baseline_date = baseline[metric]

            if stress_date == baseline_date:
                is_baseline_overall = True
                baseline_rows.append([sid,metric,baseline_mean,baseline_std])

            if use_zscore and not np.isnan(baseline_mean) and not np.isnan(baseline_std):
                out[f'{metric}_z'] = (avg_val - baseline_mean) / baseline_std
            else:
                out[f'{metric}_z'] = np.nan

        out['is_baseline'] = is_baseline_overall
        rows.append(out)

# ----------------------------------------
# Build final dataset
# ----------------------------------------
big_df = pd.DataFrame(rows)
big_df = big_df.sort_values(['Global_Deidentified', 'stress_date'])

baseline_df = pd.DataFrame(baseline_rows, columns=['id', 'metric', 'mean','std'])

# remove baseline rows
big_df = big_df[big_df['is_baseline'] == False].reset_index(drop=True)


### 4.2 — Primary pipeline: PCA → GMM → UMAP


In [ ]:
import umap
from sklearn.metrics import silhouette_score, silhouette_samples

# Stress-centered z-features going into the dynamic pipeline.
# SpO2 is excluded -- limited availability across participants.
z_cols = [
    'deep_mean_hr_z',
    'deep_RMSSD_mean_z',
    'deep_LF_HF_Ratio_mean_z',
    'Time_asleep_z',
    'deep_sleep_breathing_rate_z',
    'Total_Number_of_Steps_z',
]

# Drop rows with any NaN in z-features
X = big_df[z_cols].copy()
mask = X.notna().all(axis=1)
X = X[mask]
big_df_clust = big_df.loc[mask].reset_index(drop=True)

print("Shape before clustering:", X.shape)

# Standardize across rows (features already per-subject z, but scale across metrics)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# 2. PCA for dimensionality reduction (for clustering + visualization)
pca = PCA(n_components=min(6, X_scaled.shape[1]))
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_.cumsum()
print("Cumulative variance explained by PCA components:", explained)

# We already determined k=2 is stable (see next cell)
n_pcs = np.searchsorted(explained, 0.80) + 1
n_pcs = max(2, n_pcs)
X_clust = X_pca[:, :n_pcs]
print(f"Using first {n_pcs} PCs for clustering.")

# 2. Fit Final GMM with Fixed k=2
best_k = 2
best_gmm = GaussianMixture(
    n_components=best_k,
    covariance_type='full',
    random_state=42,
)
best_gmm.fit(X_clust)

# 3. Get Cluster Assignments and Probabilities
labels = best_gmm.predict(X_clust)
probs = best_gmm.predict_proba(X_clust)

# 4. Map back to DataFrame
big_df_clust['cluster'] = labels
for k in range(best_k):
    # This adds 'cluster_prob_0' and 'cluster_prob_1'
    big_df_clust[f'cluster_prob_{k}'] = probs[:, k]

# 5. Validation Metrics for the fixed k=2
print(f"\nResults for fixed k={best_k}:")
print("Cluster counts:")
print(big_df_clust['cluster'].value_counts().sort_index())

# Calculate silhouette statistics
sil_samples = silhouette_samples(X_clust, labels)
sil_median = np.median(sil_samples)
sil_q1 = np.percentile(sil_samples, 25)
sil_q3 = np.percentile(sil_samples, 75)

print(f"Median Silhouette: {sil_median:.3f}")
print(f"Silhouette IQR: ({sil_q1:.3f}, {sil_q3:.3f})")

# UMAP embedding for Figure 6b-d
reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=RANDOM_SEED)
X_umap = reducer.fit_transform(X_clust)


### Cluster stability

Stability checks for the two-component GMM solution: full BIC curve over k = 1..6, bootstrap ARI, leave-one-subject-out ARI, and multi-seed initialization. Run after the primary PCA → GMM pipeline.


In [ ]:
# Full BIC curve over k = 1..6
K_RANGE_STABILITY = range(1, 7)
bic_full = {}
gmm_full = {}

for k in K_RANGE_STABILITY:
    gmm_k = GaussianMixture(n_components=k, covariance_type='full',
                            random_state=RANDOM_SEED).fit(X_clust)
    bic_full[k] = gmm_k.bic(X_clust)
    gmm_full[k] = gmm_k

bic_df = pd.DataFrame({'k': list(bic_full.keys()), 'BIC': list(bic_full.values())})
print(bic_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(bic_df['k'], bic_df['BIC'], 'o-', linewidth=2, markersize=8)
ax.set_xlabel('Number of GMM components (k)')
ax.set_ylabel('Bayesian Information Criterion (BIC)')
ax.set_title(f'GMM model selection on PCA-reduced stress episodes (n = {X_clust.shape[0]})')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- AIC alongside BIC (addition) ---
aic_full = {}
for k in K_RANGE_STABILITY:
    aic_full[k] = gmm_full[k].aic(X_clust)

aic_df = pd.DataFrame({'k': list(aic_full.keys()), 'AIC': list(aic_full.values())})

# Merge BIC and AIC into one table for reporting
model_selection_df = bic_df.merge(aic_df, on='k')
print("Model selection criteria (BIC and AIC):")
print(model_selection_df.to_string(index=False))
print(f"\nBIC-preferred k: {model_selection_df.loc[model_selection_df['BIC'].idxmin(), 'k']}")
print(f"AIC-preferred k: {model_selection_df.loc[model_selection_df['AIC'].idxmin(), 'k']}")

# Plot BIC and AIC together (extends the existing BIC-only figure)
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(bic_df['k'], bic_df['BIC'], 'o-', linewidth=2, markersize=8,
        label='BIC', color='steelblue')
ax.plot(aic_df['k'], aic_df['AIC'], 's--', linewidth=2, markersize=8,
        label='AIC', color='darkorange')
ax.axvline(x=2, color='gray', linestyle=':', alpha=0.7, label='k=2 (reported)')
ax.set_xlabel('Number of GMM components (k)')
ax.set_ylabel('Information criterion')
ax.set_title(f'GMM model selection: BIC and AIC (n = {X_clust.shape[0]} stress episodes)')
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Bootstrap ARI stability across k
N_BOOTSTRAP = 200
rng_boot = np.random.default_rng(RANDOM_SEED)
bootstrap_aris = {k: [] for k in K_RANGE_STABILITY}

for k in K_RANGE_STABILITY:
    ref_labels_k = gmm_full[k].predict(X_clust)
    for b in range(N_BOOTSTRAP):
        idx = rng_boot.integers(0, X_clust.shape[0], size=X_clust.shape[0])
        try:
            gmm_boot = GaussianMixture(n_components=k, covariance_type='full',
                                       random_state=None).fit(X_clust[idx])
            labels_boot_full = gmm_boot.predict(X_clust)
            bootstrap_aris[k].append(adjusted_rand_score(ref_labels_k, labels_boot_full))
        except Exception:
            pass

ari_summary = pd.DataFrame([{
    'k': k,
    'median_ARI': float(np.median(bootstrap_aris[k])) if bootstrap_aris[k] else np.nan,
    'IQR_low':    float(np.percentile(bootstrap_aris[k], 25)) if bootstrap_aris[k] else np.nan,
    'IQR_high':   float(np.percentile(bootstrap_aris[k], 75)) if bootstrap_aris[k] else np.nan,
    'n_bootstrap': len(bootstrap_aris[k]),
} for k in K_RANGE_STABILITY])
print(ari_summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.5))
positions = list(K_RANGE_STABILITY)
data_for_box = [bootstrap_aris[k] for k in K_RANGE_STABILITY]
bp = ax.boxplot(data_for_box, positions=positions, widths=0.6,
                showfliers=False, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#cfe3f7')
    patch.set_edgecolor('#1f77b4')
ax.set_xlabel('Number of GMM components (k)')
ax.set_ylabel('Adjusted Rand Index\n(bootstrap vs reference labels)')
ax.set_title(f'Cluster stability across {N_BOOTSTRAP} bootstrap resamples')
ax.set_ylim(-0.1, 1.05)
ax.axhline(0, color='grey', linewidth=0.5)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Leave-one-subject-out stability at k = 2
LOSO_K = 2
ref_labels_full = big_df_clust['cluster'].values
subjects_arr    = big_df_clust['Global_Deidentified'].values
feature_matrix  = big_df_clust[z_cols].values

loso_rows = []
for held_out in np.unique(subjects_arr):
    mask_in = subjects_arr != held_out
    if int(mask_in.sum()) < 10:
        continue

    X_in_scaled = StandardScaler().fit_transform(feature_matrix[mask_in])

    pca_in = PCA(n_components=min(6, X_in_scaled.shape[1]),
                 random_state=RANDOM_SEED).fit(X_in_scaled)
    n_pcs_in = max(2, int(np.searchsorted(pca_in.explained_variance_ratio_.cumsum(), 0.80)) + 1)
    X_in_pca = pca_in.transform(X_in_scaled)[:, :n_pcs_in]

    gmm_in = GaussianMixture(n_components=LOSO_K, covariance_type='full',
                             random_state=RANDOM_SEED).fit(X_in_pca)
    ari_in = adjusted_rand_score(ref_labels_full[mask_in], gmm_in.predict(X_in_pca))

    loso_rows.append({'held_out': held_out, 'n_retained_episodes': int(mask_in.sum()),
                      'n_pcs': n_pcs_in, 'ARI_vs_full': ari_in})

loso_df = pd.DataFrame(loso_rows)
print(f"LOSO (k={LOSO_K}): {len(loso_df)} fits | "
      f"median ARI = {loso_df['ARI_vs_full'].median():.3f} "
      f"(IQR {loso_df['ARI_vs_full'].quantile(0.25):.3f}-{loso_df['ARI_vs_full'].quantile(0.75):.3f})")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(loso_df['ARI_vs_full'], bins=15, edgecolor='#1f77b4', facecolor='#cfe3f7')
ax.axvline(loso_df['ARI_vs_full'].median(), linestyle='--', color='red', linewidth=2,
           label=f"Median = {loso_df['ARI_vs_full'].median():.2f}")
ax.set_xlabel('ARI (LOSO labels vs full-cohort labels)')
ax.set_ylabel('Number of LOSO refits')
ax.set_title(f'Leave-one-subject-out stability (k = {LOSO_K})')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Multi-seed GMM robustness at k = 2
MULTI_SEED_N = 20

multi_seed_aris = []
for s in range(1, MULTI_SEED_N + 1):
    gmm_s = GaussianMixture(n_components=2, covariance_type='full',
                            random_state=s).fit(X_clust)
    multi_seed_aris.append({
        'seed': s,
        'ARI_vs_ref': adjusted_rand_score(ref_labels_full, gmm_s.predict(X_clust)),
    })

seed_df = pd.DataFrame(multi_seed_aris)
print(f"Multi-seed GMM (k=2, {MULTI_SEED_N} seeds): "
      f"median ARI = {seed_df['ARI_vs_ref'].median():.3f}, "
      f"min = {seed_df['ARI_vs_ref'].min():.3f}, max = {seed_df['ARI_vs_ref'].max():.3f}, "
      f"frac >= 0.9 = {(seed_df['ARI_vs_ref'] >= 0.9).mean():.0%}")


### 4.4 — Mode interpretation: participant structure, UMAP, radar plots


In [ ]:
# Figure S1

df_master = df_plot[['Global_Deidentified', 'type', 'submitdate',
       'stai_stress', 'stai_stress_category', 'category', 'ID',
       'stress_level']].copy()

# -------------------------------------------------
# Main plotting function
# -------------------------------------------------
def plot_wearable_vs_stress_availability(
    df_wearable,
    df_events,
    id_col='ID',
    wearable_date_col='Date',
    event_date_col='submitdate',
    stress_col='stai_stress',
    window_days=5,
    title='Wearable & Stress Availability'
):
    """
    Visualizes wearable data availability and stress-centered analysis windows.

    Participants are sorted by the NUMBER of wearable days available.

    - Gray horizontal bar: wearable coverage span (min–max)
    - Thin black ticks: individual wearable days
    - Semi-transparent colored bands: ±window_days stress windows
    - Colored dots: stress survey dates
    """

    # Defensive copies
    df_wearable = df_wearable.copy()
    df_events = df_events.copy()

    # Ensure datetime
    df_wearable[wearable_date_col] = pd.to_datetime(df_wearable[wearable_date_col])
    df_events[event_date_col] = pd.to_datetime(df_events[event_date_col])

    # -------------------------------------------------
    # Compute wearable availability stats per participant
    # -------------------------------------------------
    wearable_stats = (
        df_wearable
        .groupby(id_col)[wearable_date_col]
        .agg(
            start='min',
            end='max',
            n_days=lambda x: x.dt.date.nunique()
        )
        .sort_values('n_days', ascending=False)
    )

    ids = wearable_stats.index.tolist()

    # -------------------------------------------------
    # Plot
    # -------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(14, max(4, len(ids) * 0.28))
    )

    for y, uid in enumerate(ids):

        start_date = wearable_stats.loc[uid, 'start']
        wearable_end = wearable_stats.loc[uid, 'end']
        n_wearable_days = wearable_stats.loc[uid, 'n_days']

        # ---- Wearable coverage (background span)
        ax.hlines(
            y=y,
            xmin=0,
            xmax=(wearable_end - start_date).days,
            color='lightgray',
            linewidth=7,
            zorder=0
        )

        # ---- Wearable measurement days
        user_days = df_wearable.loc[
            df_wearable[id_col] == uid,
            wearable_date_col
        ]
        rel_days = (user_days - start_date).dt.days

        ax.vlines(
            rel_days,
            y - 0.35,
            y + 0.35,
            color='black',
            alpha=0.35,
            linewidth=0.8,
            zorder=1
        )

        # ---- Stress surveys and windows
        ev = df_events[df_events[id_col] == uid]

        for _, r in ev.iterrows():
            d = (r[event_date_col] - start_date).days
            c = get_color(r[stress_col])

            # Stress-centered window
            ax.hlines(
                y=y,
                xmin=d - window_days,
                xmax=d + window_days,
                color=c,
                alpha=0.25,
                linewidth=6,
                zorder=2
            )

            # Stress survey point
            ax.scatter(
                d,
                y,
                color=c,
                edgecolor='black',
                s=35,
                zorder=3
            )

    # -------------------------------------------------
    # Formatting
    # -------------------------------------------------
    ax.set_yticks(range(len(ids)))
    ax.set_yticklabels(ids, fontsize=8)
    ax.invert_yaxis()

    ax.set_xlabel('Days Since First Wearable Measurement')
    ax.set_ylabel('Participant ID')
    ax.set_title(title)

    ax.grid(axis='x', linestyle='--', alpha=0.4)
    ax.xaxis.set_minor_locator(MultipleLocator(1))
    ax.tick_params(axis='x', which='minor', length=3, color='gray')

    plt.tight_layout()
    plt.show()

mini_filtered_sleep_df = filtered_sleep_df[filtered_sleep_df['Global_Deidentified'].isin(big_df_clust['Global_Deidentified'].unique())]

plot_wearable_vs_stress_availability(
    df_wearable=mini_filtered_sleep_df,
    df_events=df_master,
    id_col='ID',
    wearable_date_col='Date',
    event_date_col='submitdate',
    stress_col='stai_stress',
    window_days=5,
    title=''
)

#### Back-transformation from z-scores to raw units


In [ ]:
# Back-transformation from z-scores to raw physiological units
signals_for_backtrans = [
    'deep_mean_hr',              # bpm
    'deep_RMSSD_mean',           # ms
    'Time_asleep',               # hours
    'Total_Number_of_Steps',     # steps
    'deep_LF_HF_Ratio_mean',     # unitless ratio
    'deep_sleep_breathing_rate', # breaths/min
]

back_rows = []
for metric in signals_for_backtrans:
    sub = baseline_df[baseline_df['metric'] == metric]
    sigmas = pd.to_numeric(sub['std'], errors='coerce').dropna()
    if len(sigmas) == 0:
        continue
    back_rows.append({
        'metric':            metric,
        'n_participants':    len(sigmas),
        'sigma_median':      float(np.median(sigmas)),
        'sigma_IQR_low':     float(np.percentile(sigmas, 25)),
        'sigma_IQR_high':    float(np.percentile(sigmas, 75)),
        'sigma_mean':        float(np.mean(sigmas)),
        'delta_raw_per_1z_median': float(np.median(sigmas)),   # since delta_raw = sigma * 1
        'delta_raw_per_1z_mean':   float(np.mean(sigmas)),
    })

back_df = pd.DataFrame(back_rows)
print("Back-transformation: raw-unit deviation implied by a 1 z-score change,")
print("computed as the per-participant baseline SD and summarized across participants.\n")
print(back_df.round(3).to_string(index=False))


#### UMAP 2D embedding (Figures 6b–d)


In [ ]:
from matplotlib.lines import Line2D

cluster_name_map = {
    0: "Recovery",
    1: "Activation",
}

cluster_color_map = {
    0: "royalblue",   # Recovery
    1: "crimson",     # Activation
}

fig, ax = plt.subplots(figsize=(6, 5))

# CHANGE scatter: use mapped colors instead of c=labels
scatter = ax.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=[cluster_color_map[k] for k in labels],
    s=20,
)

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
#ax.set_title(f"UMAP of stress responses (k={best_k})")

unique_clusters = np.sort(np.unique(labels))

handles = [
    Line2D(
        [], [], marker="o", linestyle="",
        color=cluster_color_map[k],
        label=cluster_name_map.get(k, f"Cluster {k}")
    )
    for k in unique_clusters
]

ax.legend(handles=handles), #title="Subtype")

plt.show()

stress_by_cluster = big_df_clust.groupby('cluster')['stai_stress'].describe()
print("\nSTAI stress by cluster:")
print(stress_by_cluster)

In [ ]:
# Reconstruct subject-level switching summary
switch_df = (
    big_df_clust
    .groupby(['Global_Deidentified', 'cluster'])
    .size()
    .unstack(fill_value=0)
)

# Ensure both cluster columns exist
for c in [0, 1]:
    if c not in switch_df.columns: switch_df[c] = 0

switch_df = switch_df.rename(columns={0:'cluster_0_count', 1:'cluster_1_count'})

# Assign phenotype
def assign_group(row):
    if row['cluster_0_count'] > 0 and row['cluster_1_count'] > 0:
        return "Switchers"
    elif row['cluster_0_count'] > 0:
        return "Recovery"
    else:
        return "Activation"

switch_df['switch_group'] = switch_df.apply(assign_group, axis=1)

# Map each event to its subject’s switching group
switch_map = switch_df['switch_group'].to_dict()
event_groups = big_df_clust['Global_Deidentified'].map(switch_map)

# Define raw strings to colors
color_map = {
    "Recovery": "royalblue",
    "Switchers": "green",
    "Activation": "crimson"
}

# Map the colors using the raw strings from your switch_group column
point_colors = event_groups.map(color_map)

# Check for NaNs to be safe
if point_colors.isnull().any():
    print("Warning: Some points could not be mapped to a color. Check subject IDs.")
    point_colors = point_colors.fillna('gray') # Fallback color

# UMAP plot
fig, ax = plt.subplots(figsize=(7,6))
ax.scatter(
    X_umap[:,0],
    X_umap[:,1],
    c=point_colors,
    s=35,
    alpha=0.85
)

# Re-construct Legend handles
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', label="Recovery (n=7)",
               markerfacecolor='royalblue', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label="Switchers (n=8)",
               markerfacecolor='green', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label="Activation (n=14)",
               markerfacecolor='crimson', markersize=10)
]

ax.legend(handles=legend_elements, frameon=False, fontsize=11)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

plt.tight_layout()
plt.show()

In [ ]:
# Per-participant UMAP (Figure 6c): one color/marker per participant
unique_ids = big_df_clust['Global_Deidentified'].unique()
subject_ids = big_df_clust['Global_Deidentified'].values

markers = ['o', 's', '^', 'D', 'v', '<', '>', 'P', 'X', 'h']
cmap = plt.get_cmap('tab20')
color_map  = {sid: cmap(i % 20)              for i, sid in enumerate(unique_ids)}
marker_map = {sid: markers[i % len(markers)] for i, sid in enumerate(unique_ids)}

fig, ax = plt.subplots(figsize=(7, 6))
for sid in unique_ids:
    mask = subject_ids == sid
    ax.scatter(
        X_umap[mask, 0], X_umap[mask, 1],
        color=color_map[sid], marker=marker_map[sid],
        s=35, alpha=0.9, edgecolors='k', linewidths=0.3,
    )
ax.set_xlabel("UMAP 1", fontsize=12)
ax.set_ylabel("UMAP 2", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
pretty_labels = [
    "Deep sleep HR",
    "Deep sleep HRV (RMSSD)",
    "Sleep duration",
    "Daily steps",
    "Deep sleep LF/HF",
    "Deep sleep BR",
]

labels = [
    'deep_mean_hr_z',
    'deep_RMSSD_mean_z',
    'Time_asleep_z',
    'Total_Number_of_Steps_z',
    'deep_LF_HF_Ratio_mean_z',
    'deep_sleep_breathing_rate_z',
]

cluster_means = big_df_clust.groupby('cluster')[labels].mean()

num_vars = len(labels)
angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
angles = np.concatenate([angles, [angles[0]]])

fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))

colors = {0: '#1f77b4', 1: '#d62728'}   # blue & red, colorblind-safe
names  = {0: 'Recovery', 1: 'Activation'}

# plot each cluster
for k in cluster_means.index:
    vals = cluster_means.loc[k].values
    vals = np.concatenate([vals, [vals[0]]])
    ax.plot(angles, vals, linewidth=2.5, label=names[k], color=colors[k])
    ax.fill(angles, vals, alpha=0.20, color=colors[k])

# pretty label replacement
ax.set_xticks(angles[:-1])
ax.set_xticklabels(pretty_labels, fontsize=11)

# dashed zero-line for visual anchor
ax.plot(angles, [0]*len(angles), '--', color='gray', linewidth=1.2, alpha=0.8)

# radial limits (tight!)
ax.set_ylim(-0.8, 0.8)
ax.set_yticks([-0.6, -0.3, 0, 0.3, 0.6])
ax.set_yticklabels(['-0.6', '-0.3', '0', '0.3', '0.6'], fontsize=8)
ax.set_rlabel_position(90)
ax.tick_params(axis='x', pad=32)

# clean title
# ax.set_title(
#     "Physiological Stress-Response Profiles (Within-Person Z-Scores)",
#     fontsize=14,
#     pad=24
# )

# legend below the plot
ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.18),
    frameon=False,
    fontsize=11
)

plt.subplots_adjust(top=0.90, bottom=0.28, left=0.10, right=0.90)
plt.show()

pretty_labels = [
    "Deep sleep HR",
    "Deep sleep HRV (RMSSD)",
    "Sleep duration",
    "Daily steps",
    "Deep sleep LF/HF",
    "Deep sleep BR",
]

labels = [
    "deep_mean_hr_z",
    "deep_RMSSD_mean_z",
    "Time_asleep_z",
    "Total_Number_of_Steps_z",
    "deep_LF_HF_Ratio_mean_z",
    "deep_sleep_breathing_rate_z",
]

# 1) Subject x cluster counts
per_subj_cluster_counts = (
    big_df_clust
    .groupby(["Global_Deidentified", "cluster"])
    .size()
    .unstack(fill_value=0)
    .rename(columns=lambda c: f"cluster_{c}_count")
)

needed_cols = {"cluster_0_count", "cluster_1_count"}
missing = needed_cols - set(per_subj_cluster_counts.columns)
if missing:
    raise ValueError(f"Expected clusters 0 and 1. Missing columns: {missing}. موجود: {list(per_subj_cluster_counts.columns)}")

only_cluster0 = (per_subj_cluster_counts["cluster_0_count"] > 0) & (per_subj_cluster_counts["cluster_1_count"] == 0)
only_cluster1 = (per_subj_cluster_counts["cluster_1_count"] > 0) & (per_subj_cluster_counts["cluster_0_count"] == 0)
switchers     = (per_subj_cluster_counts > 0).sum(axis=1) == 2


In [ ]:
# Per-participant mode expression: Recovery-only / Activation-only / Switcher
per_subj = (
    big_df_clust
    .groupby('Global_Deidentified')['cluster']
    .agg(n_episodes='size',
         n_activation=lambda s: int((s == 1).sum()),
         n_recovery=lambda s:   int((s == 0).sum()))
    .reset_index()
)
per_subj['phenotype'] = np.where(
    (per_subj['n_activation'] > 0) & (per_subj['n_recovery'] > 0), 'Switcher',
    np.where(per_subj['n_activation'] > 0, 'Activation-only', 'Recovery-only')
)

print("Per-participant phenotype counts:")
print(per_subj['phenotype'].value_counts())


### 4.5 — PCA loadings


In [ ]:
# Use first two PCs (same PCA used to build X_clust)
scores_2d = X_pca[:, :2]           # shape: (n_samples, 2)
loadings  = pca.components_[:2, :].T   # shape: (n_features, 2)

labels_plot = big_df_clust['cluster'].to_numpy()

fig, ax = plt.subplots(figsize=(7, 6))

# 1) scatter of subjects in PC space, colored by GMM cluster
scatter = ax.scatter(
    scores_2d[:, 0], scores_2d[:, 1],
    c=labels_plot, s=40, alpha=0.7
)

# 2) feature loadings as arrows
arrow_scale = 3.0  # tweak if arrows too small/large
for i, feat in enumerate(z_cols):
    x_load = loadings[i, 0] * arrow_scale
    y_load = loadings[i, 1] * arrow_scale
    ax.arrow(
        0, 0,
        x_load, y_load,
        head_width=0.08,
        head_length=0.12,
        length_includes_head=True,
        alpha=0.9
    )
    ax.text(
        x_load * 1.05,
        y_load * 1.05,
        feat,
        fontsize=9,
        ha='center', va='center'
    )

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title(f"PCA biplot with GMM clusters (k={best_k})")

handles, lab = scatter.legend_elements()
ax.legend(handles, lab, title="Cluster")

ax.axhline(0, color='grey', linewidth=0.5)
ax.axvline(0, color='grey', linewidth=0.5)

plt.tight_layout()
plt.show()

# ---- 1. Variance explained plot ----
plt.figure(figsize=(5, 3))
plt.plot(np.arange(1, len(pca.explained_variance_ratio_) + 1),
         pca.explained_variance_ratio_, "o-")
plt.xlabel("Principal Component")
plt.ylabel("Variance Explained")
plt.title("PCA Variance Explained per Component")
plt.tight_layout()
plt.show()

# ---- 2. Cumulative variance ----
plt.figure(figsize=(5, 3))
plt.plot(np.arange(1, len(explained) + 1), explained, "o-")
plt.axhline(0.80, color='red', linestyle='--', alpha=0.6)
plt.xlabel("Principal Component")
plt.ylabel("Cumulative Variance Explained")
plt.title("Cumulative PCA Variance Explained")
plt.tight_layout()
plt.show()

# ---- 3. Loadings (components) ----
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(pca.components_.shape[0])],
    index=z_cols  # your feature names
)

print("\nPCA Loadings (feature contributions to each PC):")
loadings = loadings.rename(index={
    'deep_mean_hr_z': 'Deep sleep HR',
    'deep_RMSSD_mean_z': 'Deep sleep HRV (RMSSD)',
    'deep_LF_HF_Ratio_mean_z': 'Deep sleep LF/HF',
    'Time_asleep_z': 'Sleep duration',
    'deep_sleep_breathing_rate_z': 'Deep sleep BR',
    'Total_Number_of_Steps_z': 'Daily steps',
})
# ---- 4. Heatmap of loadings ----
plt.figure(figsize=(7, 5))
sns.heatmap(loadings, annot=True, cmap="coolwarm", center=0)
plt.title("PCA Component Loadings")
plt.tight_layout()
plt.show()

### 4.5.5 — Overlap analysis


In [ ]:
from collections import Counter
import pandas as pd
import numpy as np

# ----------------------------------------
# Restrict to participants used in clustering
# ----------------------------------------
valid_ids = set(big_df_clust['Global_Deidentified'].unique())

overlap_stats = []

# Ensure datetime
big_df['stress_date'] = pd.to_datetime(big_df['stress_date'])

for sid, g in big_df.groupby('Global_Deidentified'):

    # Only include participants used in clustering
    if sid not in valid_ids:
        continue

    # Stress dates ACTUALLY USED in analysis
    stress_dates = g['stress_date'].unique()

    if len(stress_dates) == 0:
        continue

    all_days = []

    for d in stress_dates:
        win_days = pd.date_range(
            d - pd.Timedelta(days=window_days),
            d + pd.Timedelta(days=window_days),
            freq='D'
        )
        all_days.extend(win_days)

    day_counts = Counter(all_days)

    total_window_days = len(all_days)
    unique_days = len(day_counts)
    reused_days = sum(1 for c in day_counts.values() if c > 1)

    overlap_ratio = (
        1 - unique_days / total_window_days
        if total_window_days > 0 else np.nan
    )

    overlap_stats.append({
        'Global_Deidentified': sid,
        'n_stress_windows_used': len(stress_dates),
        'total_window_days': total_window_days,
        'unique_days': unique_days,
        'days_used_more_than_once': reused_days,
        'overlap_ratio': overlap_ratio
    })

overlap_df = pd.DataFrame(overlap_stats)

overlap_df

In [ ]:
def plot_overlap_summary(overlap_df):
    import numpy as np
    import matplotlib.pyplot as plt

    df = overlap_df.dropna(subset=["overlap_ratio", "n_stress_windows_used"])

    pct = df["overlap_ratio"].to_numpy() * 100
    events = df["n_stress_windows_used"].to_numpy()

    # Per-subject summaries
    med, q1, q3 = np.percentile(pct, [50, 25, 75])
    ev_med, ev_q1, ev_q3 = np.percentile(events, [50, 25, 75])

    # Pooled fraction of window-days unique to a single survey
    pct_unique = 100 * df["unique_days"].sum() / df["total_window_days"].sum()

    print(f"N subjects: {len(df)}")
    print(f"Overlap per subject:        median {med:.1f}% (IQR {q1:.1f}-{q3:.1f}%)")
    print(f"Stress events per subject:  median {ev_med:.0f} (IQR {ev_q1:.0f}-{ev_q3:.0f})")
    print(f"Window-days unique to a single survey (pooled): ~{pct_unique:.0f}%")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.hist(pct, bins=np.linspace(0, max(10, pct.max()), 16))
    ax1.set_title("Overlap across subjects")
    ax1.set_xlabel("Percent overlap of window-days (%)")
    ax1.set_ylabel("Number of subjects")
    ax1.text(
        0.02, 0.95,
        f"median={med:.1f}%\nIQR={q1:.1f}-{q3:.1f}%\n~{pct_unique:.0f}% window-days unique",
        transform=ax1.transAxes, va="top",
    )

    ax2.scatter(events, pct, alpha=0.7)
    ax2.set_title("Overlap vs. number of stress events")
    ax2.set_xlabel(f"Stress events (median {ev_med:.0f}, IQR {ev_q1:.0f}-{ev_q3:.0f})")
    ax2.set_ylabel("Percent overlap (%)")

    plt.tight_layout()
    plt.show()


plot_overlap_summary(overlap_df)

### Overlap sensitivity analysis: non-overlapping windows only

In [ ]:
# =========================================================================
# Overlap sensitivity analysis
#
# The primary analysis uses ±5-day windows centered on each stress survey,
# which can produce overlapping windows when surveys are close together.
# The overlap analysis (Cell 67) showed a median overlap of ~13.6% across
# participants, meaning ~82% of analyzed days were non-overlapping.
#
# This cell re-runs the primary PCA -> GMM pipeline after excluding
# overlapping days from each episode's metric averages. Specifically, for
# each participant, any calendar day that falls within the ±5-day window
# of MORE THAN ONE stress survey is excluded from the averaging for all
# affected episodes. Only days unique to a single episode are used.
#
# If the two-mode structure (Activation vs Recovery) is robust, the
# cluster assignments from this non-overlapping analysis should be
# highly concordant with the primary analysis (high ARI).
#
# Variables confirmed in scope: big_df, big_df_clust, full_df,
# df_anxiety_min_2_samples, signals, z_cols, baseline_lookup,
# window_days, RANDOM_SEED, X_clust, pca, scaler
# =========================================================================

from collections import defaultdict

# -----------------------------------------------------------------------
# Step 1: For each participant, identify which calendar days belong to
# more than one stress-survey window (i.e., overlapping days).
# -----------------------------------------------------------------------
# We'll build a dict: {(participant_id, stress_date): set of unique days}
# where unique days = days that fall only in THIS episode's window and
# no other episode's window for the same participant.

episode_unique_days = {}  # key: (sid, stress_date), value: set of unique dates

for sid, g in df_anxiety_min_2_samples.groupby('Global_Deidentified'):

    # Only process participants used in the primary clustering
    if sid not in set(big_df_clust['Global_Deidentified'].unique()):
        continue

    stress_dates = pd.to_datetime(g['Date'].dropna().unique())

    # For each survey date, compute its ±window_days date range
    windows = {}
    for d in stress_dates:
        win_days = set(pd.date_range(
            d - pd.Timedelta(days=window_days),
            d + pd.Timedelta(days=window_days),
            freq='D'
        ))
        windows[d] = win_days

    # Count how many windows each calendar day appears in
    day_count = defaultdict(int)
    for win_days in windows.values():
        for day in win_days:
            day_count[day] += 1

    # For each episode, keep only days that appear in exactly one window
    for d, win_days in windows.items():
        unique_days = {day for day in win_days if day_count[day] == 1}
        episode_unique_days[(sid, d)] = unique_days

print(f"Computed unique days for {len(episode_unique_days)} episodes.")

# -----------------------------------------------------------------------
# Step 2: Recompute per-episode metric averages using only unique days.
# Uses the same baseline z-scoring as the primary pipeline.
# -----------------------------------------------------------------------
nonoverlap_rows = []

for sid, g in df_anxiety_min_2_samples.groupby('Global_Deidentified'):

    if sid not in set(big_df_clust['Global_Deidentified'].unique()):
        continue

    subj_stress = g.copy()
    subj_stress['Date'] = pd.to_datetime(subj_stress['Date'])

    # Skip baseline survey (same logic as primary pipeline)
    min_idx = subj_stress['stai_stress'].idxmin()
    baseline_date = subj_stress.loc[min_idx, 'Date']

    for _, row_s in subj_stress.iterrows():
        stress_date = row_s['Date']
        stress_score = row_s['stai_stress']

        # Skip baseline
        if stress_date == baseline_date:
            continue

        unique_days = episode_unique_days.get((sid, stress_date), set())

        # Need at least 1 unique day to compute a meaningful average
        if len(unique_days) == 0:
            continue

        out = {
            'Global_Deidentified': sid,
            'stress_date': stress_date,
            'stai_stress': stress_score,
        }

        for metric, z_col in zip(signals, z_cols):
            info = baseline_lookup.get((sid, metric))
            if not info or pd.isna(info['std']) or info['std'] == 0:
                out[z_col] = np.nan
                continue

            # Get wearable data only on non-overlapping days for this episode
            subj_data = full_df[
                (full_df['Global_Deidentified'] == sid) &
                (full_df['Date'].isin(unique_days))
            ]

            raw_val = subj_data[metric].mean() if (
                metric in subj_data.columns and not subj_data.empty
            ) else np.nan

            out[z_col] = ((raw_val - info['mean']) / info['std']
                          if not pd.isna(raw_val) else np.nan)

        nonoverlap_rows.append(out)

nonoverlap_df = pd.DataFrame(nonoverlap_rows)
print(f"Non-overlapping dataset: {len(nonoverlap_df)} episodes from "
      f"{nonoverlap_df['Global_Deidentified'].nunique()} participants.")
print(f"Primary dataset had {len(big_df_clust)} episodes — "
      f"difference of {len(big_df_clust) - len(nonoverlap_df)} episodes "
      f"lost due to complete overlap.")

# -----------------------------------------------------------------------
# Step 3: Run the same PCA -> GMM pipeline on the non-overlapping data.
# -----------------------------------------------------------------------
# Drop rows with any NaN in z-features (same as primary pipeline)
X_no = nonoverlap_df[z_cols].dropna()
nonoverlap_clust = nonoverlap_df.loc[X_no.index].reset_index(drop=True)
X_no = X_no.reset_index(drop=True)

if len(X_no) < 10:
    print("WARNING: Too few non-overlapping episodes to run pipeline.")
else:
    # Standardize
    scaler_no = StandardScaler()
    X_no_scaled = scaler_no.fit_transform(X_no)

    # PCA (same number of components as primary pipeline)
    pca_no = PCA(n_components=min(6, X_no_scaled.shape[1]),
                 random_state=RANDOM_SEED)
    X_no_pca = pca_no.fit_transform(X_no_scaled)
    cum_var_no = pca_no.explained_variance_ratio_.cumsum()
    n_pcs_no = max(2, int(np.searchsorted(cum_var_no, 0.80)) + 1)
    X_no_clust = X_no_pca[:, :n_pcs_no]
    print(f"Non-overlapping PCA: using {n_pcs_no} PCs "
          f"({cum_var_no[n_pcs_no-1]:.2f} cumulative variance)")

    # GMM at k=2 (same as primary pipeline)
    gmm_no = GaussianMixture(n_components=2, covariance_type='full',
                              random_state=RANDOM_SEED).fit(X_no_clust)
    nonoverlap_clust['cluster_no'] = gmm_no.predict(X_no_clust)

    print("
Non-overlapping cluster counts:")
    print(nonoverlap_clust['cluster_no'].value_counts().sort_index())

    # -----------------------------------------------------------------------
    # Step 4: Compare cluster assignments to primary pipeline.
    # Match episodes that appear in both datasets and compute ARI.
    # -----------------------------------------------------------------------
    # Merge on participant + stress_date to align episodes
    primary_labels = big_df_clust[
        ['Global_Deidentified', 'stress_date', 'cluster']
    ].copy()
    primary_labels['stress_date'] = pd.to_datetime(primary_labels['stress_date'])

    nonoverlap_clust['stress_date'] = pd.to_datetime(nonoverlap_clust['stress_date'])

    merged_labels = pd.merge(
        primary_labels,
        nonoverlap_clust[['Global_Deidentified', 'stress_date', 'cluster_no']],
        on=['Global_Deidentified', 'stress_date'],
        how='inner'
    )

    n_matched = len(merged_labels)
    ari_overlap = adjusted_rand_score(
        merged_labels['cluster'], merged_labels['cluster_no']
    )

    print(f"
Overlap sensitivity analysis:")
    print(f"  Episodes in primary analysis:          {len(big_df_clust)}")
    print(f"  Episodes in non-overlapping analysis:  {len(nonoverlap_clust)}")
    print(f"  Episodes matched for ARI comparison:   {n_matched}")
    print(f"  ARI (primary vs non-overlapping):      {ari_overlap:.3f}")
    print()
    if ari_overlap >= 0.8:
        print("  -> High concordance: the two-mode structure is robust to")
        print("     exclusion of overlapping days.")
    elif ari_overlap >= 0.6:
        print("  -> Moderate concordance: the two-mode structure is largely")
        print("     stable but some sensitivity to overlapping days exists.")
    else:
        print("  -> Low concordance: cluster assignments are sensitive to")
        print("     overlapping days. Consider reporting this as a limitation.")

    # -----------------------------------------------------------------------
    # Step 5: Compare cluster-mean physiological profiles between
    # primary and non-overlapping analyses (radar plot comparison).
    # -----------------------------------------------------------------------
    primary_means   = big_df_clust.groupby('cluster')[z_cols].mean()
    nonoverlap_means = nonoverlap_clust.groupby('cluster_no')[z_cols].mean()

    print("
Primary cluster means (z-scores):")
    print(primary_means.round(3))
    print("
Non-overlapping cluster means (z-scores):")
    print(nonoverlap_means.round(3))

    # Centroid anti-correlation (same diagnostic as primary paper)
    if len(nonoverlap_means) == 2:
        c0 = nonoverlap_means.iloc[0].values
        c1 = nonoverlap_means.iloc[1].values
        r_no = np.corrcoef(c0, c1)[0, 1]
        print(f"
Non-overlapping centroid Pearson r: {r_no:.3f}")
        print(f"Primary centroid Pearson r:         -0.82 (reported in paper)")
        print("(More negative = more opposing profiles, consistent with primary result)")


### 4.6 — Mode-level characterization: S-STAI, box plots, effect sizes


In [ ]:
n_per_cluster = (
    big_df_clust
    .groupby('cluster')['stai_stress']
    .count()
    .to_dict()
)

plt.figure(figsize=(5, 5))
sns.boxplot(
    data=big_df_clust,
    x="cluster",
    y="stai_stress",
    palette={'0': "tab:blue", '1': "tab:red"},
    width=0.5,
    fliersize=3
)

sns.stripplot(
    data=big_df_clust,
    x="cluster",
    y="stai_stress",
    color="black",
    alpha=0.4,
    jitter=0.15,
    size=3
)

plt.xticks(
    [0, 1],
    [
        f"Recovery\nN={n_per_cluster.get(0, 0)}",
        f"Activation\nN={n_per_cluster.get(1, 0)}"
    ]
)

plt.xlabel("Cluster")
plt.ylabel("STAI Stress Score")
plt.tight_layout()
from matplotlib.ticker import AutoMinorLocator

ax = plt.gca()
ax.yaxis.set_minor_locator(AutoMinorLocator(5))  # 2 minor ticks between majors
ax.grid(which='major', axis='y', linestyle='-', alpha=0.3)
ax.grid(which='minor', axis='y', linestyle=':', alpha=0.2)

plt.tight_layout()
plt.show()

# BEFORE plotting: build a baseline group (per-subject minimum STAI)
baseline_rows = (
    big_df_clust
    .loc[big_df_clust.groupby('Global_Deidentified')['stai_stress'].idxmin(),
         ['stai_stress']]
    .copy()
)
baseline_rows['cluster'] = 'Baseline'

plot_df = pd.concat(
    [big_df_clust[['cluster', 'stai_stress']], baseline_rows],
    ignore_index=True
)

# --- Build baseline group (per-subject minimum STAI) ---
baseline_rows = (
    big_df_clust
    .loc[big_df_clust.groupby('Global_Deidentified')['stai_stress'].idxmin(),
         ['Global_Deidentified', 'stai_stress']]
    .copy()
)
baseline_rows['cluster'] = 'Baseline'

# Make sure cluster is string for everything
big_df_tmp = big_df_clust.copy()
big_df_tmp['cluster'] = big_df_tmp['cluster'].astype(str)

plot_df = pd.concat(
    [big_df_tmp[['cluster', 'stai_stress']], baseline_rows[['cluster', 'stai_stress']]],
    ignore_index=True
)

# --- Plot ---
plt.figure(figsize=(5, 5))

sns.boxplot(
    data=plot_df,
    x="cluster",
    y="stai_stress",
    order=['Baseline', '0', '1'],
    palette={'Baseline': '0.7', '0': "tab:blue", '1': "tab:red"},
    width=0.5,
    fliersize=3
)

sns.stripplot(
    data=plot_df,
    x="cluster",
    y="stai_stress",
    order=['Baseline', '0', '1'],
    color="black",
    alpha=0.4,
    jitter=0.15,
    size=3
)

ax = plt.gca()
ax.set_xticklabels(
    ["Baseline",
     "Recovery (Cluster 0)",
     "Activation (Cluster 1)"]
)
ax.set_xlabel("Group")
ax.set_ylabel("STAI Stress Score")

from matplotlib.ticker import AutoMinorLocator
ax.yaxis.set_minor_locator(AutoMinorLocator(5))
ax.grid(which='major', axis='y', linestyle='-', alpha=0.3)
ax.grid(which='minor', axis='y', linestyle=':', alpha=0.2)

plt.tight_layout()
plt.show()

### Negative control: random non-stress-aligned windows


In [ ]:
# =========================================================================
# Negative control: 100-draw Monte Carlo over random non-stress-aligned windows
#
# For each draw:
#   1) Sample random WINDOW_DAYS-day windows >= MIN_GAP_DAYS away from any
#      S-STAI survey, matched per participant to their number of real
#      stress episodes, restricted to the 29 analytic participants.
#   2) Z-score each metric against the same per-participant low-stress
#      baseline used in the primary pipeline.
#   3) Run PCA -> GMM (k = 2..6, BIC-selected).
#   4) Record cluster sizes and per-metric mean z-score within each cluster.
#
# Reporting strategy:
#   - We do NOT report centroid Pearson r. Cluster sizes under the null
#     are typically very unbalanced, which makes between-centroid r an
#     unstable summary.
#   - We instead report, across draws:
#       * the distribution of cluster sizes
#       * the median + IQR of per-cluster mean z-scores per metric
#     Under H0, cluster means should cluster around 0 with no coordinated
#     opposing pattern -- in contrast to Activation/Recovery in the real data.
# =========================================================================
import os

# ---- Parameters ---------------------------------------------------------
N_DRAWS         = 100
MIN_GAP_DAYS    = 5     # >= 5 days from any real S-STAI date
MIN_GAP_BETWEEN = 7     # >= 7 days between successive pseudo-events
WINDOW_DAYS_NC  = window_days   # = 5, matches the primary pipeline
K_RANGE         = range(2, 7)

z_cols_names = [
    'deep_mean_hr_z',
    'deep_RMSSD_mean_z',
    'deep_LF_HF_Ratio_mean_z',
    'Time_asleep_z',
    'deep_sleep_breathing_rate_z',
    'Total_Number_of_Steps_z',
]
raw_signals = [c.replace('_z', '') for c in z_cols_names]

# Restrict to the 29 analytic participants from the primary PCA -> GMM fit
analytic_ids = sorted(big_df_clust['Global_Deidentified'].unique())
print(f"Restricting null to {len(analytic_ids)} analytic participants.")

# Per-subject baseline lookup: (id, metric) -> {mean, std}
baseline_lookup = (
    baseline_df
    .drop_duplicates(subset=['id', 'metric'])
    .set_index(['id', 'metric'])[['mean', 'std']]
    .to_dict('index')
)

# Real S-STAI dates per subject
stress_dates_by_subj = {
    sid: np.sort(pd.to_datetime(g['Date'].dropna().unique()))
    for sid, g in df_anxiety_min_2_samples.groupby('Global_Deidentified')
}


# ---- Helper: build one null draw ---------------------------------------
def build_null_draw(rng):
    """Sample one set of random non-stress windows and return their z-matrix."""
    pseudo_rows = []

    for sid in analytic_ids:
        if sid not in stress_dates_by_subj:
            continue
        stress_dates = stress_dates_by_subj[sid]
        stress_set   = set(stress_dates)
        n_stress     = len(stress_dates)

        subj_dates = np.sort(pd.to_datetime(
            full_df.loc[full_df['Global_Deidentified'] == sid, 'Date']
            .dropna().unique()
        ))
        if len(subj_dates) == 0:
            continue

        # Days >= MIN_GAP_DAYS from every real stress date
        far = []
        for d in subj_dates:
            if d in stress_set:
                continue
            deltas = np.abs(stress_dates - d) / np.timedelta64(1, 'D')
            if deltas.min() >= MIN_GAP_DAYS:
                far.append(d)
        if not far:
            continue

        far = np.array(far)
        rng.shuffle(far)

        # Greedy selection enforcing >= MIN_GAP_BETWEEN spacing
        selected = []
        for d in far:
            if all(abs((d - s) / np.timedelta64(1, 'D')) >= MIN_GAP_BETWEEN
                   for s in selected):
                selected.append(d)
                if len(selected) >= n_stress:
                    break

        for d in selected:
            pseudo_rows.append({'Global_Deidentified': sid, 'pseudo_date': d})

    if not pseudo_rows:
        return None

    # z-score each pseudo-event's window against the subject's baseline
    records = []
    for r in pseudo_rows:
        sid    = r['Global_Deidentified']
        p_date = r['pseudo_date']
        win = full_df[
            (full_df['Global_Deidentified'] == sid)
            & (full_df['Date'] >= p_date - pd.Timedelta(days=WINDOW_DAYS_NC))
            & (full_df['Date'] <= p_date + pd.Timedelta(days=WINDOW_DAYS_NC))
        ]

        out = {'Global_Deidentified': sid}
        for metric in raw_signals:
            avg  = win[metric].mean() if len(win) > 0 else np.nan
            info = baseline_lookup.get((sid, metric))
            if info and not np.isnan(avg) and info['std'] and not np.isnan(info['std']):
                out[f'{metric}_z'] = (avg - info['mean']) / info['std']
            else:
                out[f'{metric}_z'] = np.nan
        records.append(out)

    return pd.DataFrame(records).dropna(subset=z_cols_names)


# ---- Helper: PCA -> GMM on one null draw -------------------------------
def run_null_pipeline(z_df):
    """Fit PCA -> GMM (BIC-selected k); return cluster sizes and mean z-scores."""
    if len(z_df) < 15:
        return None

    X_scaled = StandardScaler().fit_transform(z_df[z_cols_names].values)
    pca = PCA(n_components=min(6, X_scaled.shape[1]),
              random_state=RANDOM_SEED).fit(X_scaled)
    cum_var = pca.explained_variance_ratio_.cumsum()
    n_pcs = max(2, int(np.searchsorted(cum_var, 0.80)) + 1)
    X_clust = pca.transform(X_scaled)[:, :n_pcs]

    bics = {}
    for k in K_RANGE:
        gmm = GaussianMixture(n_components=k, covariance_type='full',
                              random_state=RANDOM_SEED).fit(X_clust)
        bics[k] = (gmm.bic(X_clust), gmm)

    best_k = min(bics, key=lambda k: bics[k][0])
    best_gmm = bics[best_k][1]

    z_df = z_df.copy()
    z_df['cluster'] = best_gmm.predict(X_clust)

    cluster_means = z_df.groupby('cluster')[z_cols_names].mean()
    cluster_sizes = z_df['cluster'].value_counts().sort_index()

    # Order clusters by size (largest first) so cluster_rank is comparable
    # across draws regardless of GMM's arbitrary label assignment
    size_order = cluster_sizes.sort_values(ascending=False).index.tolist()

    rows = []
    for rank, c_id in enumerate(size_order):
        for metric in z_cols_names:
            rows.append({
                'cluster_rank': rank,                         # 0 = largest
                'cluster_size': int(cluster_sizes.loc[c_id]),
                'metric':       metric,
                'mean_z':       float(cluster_means.loc[c_id, metric]),
            })

    return {
        'n_episodes': len(z_df),
        'best_k':     int(best_k),
        'rows':       rows,
        'sizes':      [int(cluster_sizes.loc[c]) for c in size_order],
    }


# ---- Run the Monte Carlo -----------------------------------------------
summary_rows = []
cluster_rows = []
rng = np.random.default_rng(123)

for draw in range(N_DRAWS):
    z_df = build_null_draw(rng)
    if z_df is None:
        continue
    res = run_null_pipeline(z_df)
    if res is None:
        continue

    summary_rows.append({
        'draw':       draw,
        'n_episodes': res['n_episodes'],
        'best_k':     res['best_k'],
        'sizes':      res['sizes'],
    })
    for r in res['rows']:
        cluster_rows.append({**r, 'draw': draw, 'best_k': res['best_k']})

    if (draw + 1) % 10 == 0:
        print(f"  completed draw {draw + 1} / {N_DRAWS}")

mc_summary  = pd.DataFrame(summary_rows)
mc_clusters = pd.DataFrame(cluster_rows)

print(f"\nCompleted {len(mc_summary)} / {N_DRAWS} null draws")
print("\nBIC-selected k distribution under the null:")
print(mc_summary['best_k'].value_counts().sort_index())


# =========================================================================
# Cluster-size distribution across draws (size-ranked: largest, 2nd largest, ...)
# =========================================================================
size_long = (
    mc_clusters[['draw', 'cluster_rank', 'cluster_size']]
    .drop_duplicates()
)
size_summary = (
    size_long.groupby('cluster_rank')['cluster_size']
    .agg(n_draws='count',
         median='median',
         iqr_low =lambda s: s.quantile(0.25),
         iqr_high=lambda s: s.quantile(0.75),
         min='min',
         max='max')
    .reset_index()
    .rename(columns={'cluster_rank': 'rank (0 = largest)'})
)
print("\nCluster sizes across draws (size-ranked):")
print(size_summary.round(2).to_string(index=False))


# =========================================================================
# Per-cluster, per-metric mean z-score: median + IQR across draws
# =========================================================================
zscore_summary = (
    mc_clusters
    .groupby(['cluster_rank', 'metric'])['mean_z']
    .agg(n_draws='count',
         null_median='median',
         null_iqr_low =lambda s: s.quantile(0.25),
         null_iqr_high=lambda s: s.quantile(0.75))
    .reset_index()
)
print("\nNull cluster mean z-scores per metric (median + IQR across draws):")
print(zscore_summary.round(3).to_string(index=False))


# =========================================================================
# Real-data comparison (mean z-score per cluster x metric)
# =========================================================================
real_means = big_df_clust.groupby('cluster')[z_cols].mean()
real_sizes = big_df_clust['cluster'].value_counts().sort_index()
# Rank real clusters the same way (largest first) so the comparison aligns
real_rank_order = real_sizes.sort_values(ascending=False).index.tolist()

real_long_rows = []
for rank, c_id in enumerate(real_rank_order):
    for metric in z_cols:
        real_long_rows.append({
            'cluster_rank':  rank,
            'cluster_size':  int(real_sizes.loc[c_id]),
            'metric':        metric,
            'real_mean_z':   float(real_means.loc[c_id, metric]),
        })
real_long = pd.DataFrame(real_long_rows)

print("\nReal-data cluster mean z-scores (largest cluster first):")
print(real_long.round(3).to_string(index=False))

# Save outputs
out_dir = str(OUTPUT_DIR)
mc_summary.to_csv(os.path.join(out_dir, 'S9_null_mc_per_draw_summary.csv'), index=False)
size_summary.to_csv(os.path.join(out_dir, 'S9_null_cluster_sizes.csv'), index=False)
zscore_summary.to_csv(os.path.join(out_dir, 'S9_null_zscore_per_cluster_metric.csv'), index=False)
real_long.to_csv(os.path.join(out_dir, 'S9_real_cluster_means.csv'), index=False)


# =========================================================================
# Visualization: null vs real, per cluster rank, per metric
# =========================================================================
pretty = {
    'deep_mean_hr_z':              'Deep-sleep HR',
    'deep_RMSSD_mean_z':           'Deep-sleep RMSSD',
    'deep_LF_HF_Ratio_mean_z':     'Deep-sleep LF/HF',
    'Time_asleep_z':               'Total sleep',
    'deep_sleep_breathing_rate_z': 'Deep-sleep BR',
    'Total_Number_of_Steps_z':     'Daily steps',
}
metrics_order = list(pretty.keys())

ranks_to_plot = sorted(real_long['cluster_rank'].unique())
fig, axes = plt.subplots(1, len(ranks_to_plot),
                         figsize=(6.5 * len(ranks_to_plot), 4),
                         sharey=True)
if len(ranks_to_plot) == 1:
    axes = [axes]

for ax, rank in zip(axes, ranks_to_plot):
    real_c = real_long[real_long['cluster_rank'] == rank].set_index('metric')
    null_c = zscore_summary[zscore_summary['cluster_rank'] == rank].set_index('metric')

    x = np.arange(len(metrics_order))
    w = 0.4

    null_med = [null_c.loc[m, 'null_median']   if m in null_c.index else np.nan for m in metrics_order]
    null_lo  = [null_c.loc[m, 'null_iqr_low']  if m in null_c.index else np.nan for m in metrics_order]
    null_hi  = [null_c.loc[m, 'null_iqr_high'] if m in null_c.index else np.nan for m in metrics_order]
    null_err = [[abs(m - lo) for m, lo in zip(null_med, null_lo)],
                [abs(hi - m) for m, hi in zip(null_med, null_hi)]]
    real_med = [real_c.loc[m, 'real_mean_z'] if m in real_c.index else np.nan for m in metrics_order]

    ax.bar(x - w/2, null_med, w, yerr=null_err, capsize=4,
           label='Null (median +/- IQR)', color='#cfe3f7', edgecolor='#1f77b4')
    ax.bar(x + w/2, real_med, w,
           label='Real data', color='#f7cfcf', edgecolor='#d62728')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([pretty[m] for m in metrics_order], rotation=25, ha='right')

    real_n = int(real_c['cluster_size'].iloc[0])
    null_n = size_summary.loc[size_summary['rank (0 = largest)'] == rank,
                              'median'].iloc[0]
    ax.set_title(f'Cluster rank {rank} '
                 f'(real n={real_n}; null median n={null_n:.0f})')
    ax.legend(frameon=False, fontsize=9)
    ax.grid(axis='y', alpha=0.25)

axes[0].set_ylabel('Mean z-score')
fig.suptitle('Null vs real: mean z-scores per cluster (size-ranked)', fontsize=12)
plt.tight_layout()
plt.show()

# =========================================================================
# Consolidated negative-control summary for manuscript
# =========================================================================
print("\n" + "="*80)
print("NEGATIVE CONTROL SUMMARY (100 Monte Carlo draws)")
print("="*80)

print("\n1. BIC MODEL SELECTION")
print("-" * 80)
k_dist = mc_summary['best_k'].value_counts().sort_index()
print(f"k = 2: {k_dist.get(2, 0)}/100 draws")
print(f"k ≥ 3: {100 - k_dist.get(2, 0)}/100 draws (fragmenting into 3–6 components)")

print("\n2. CLUSTER SIZE COMPARISON (size-ranked, largest first)")
print("-" * 80)
print("\nNull distribution (across 100 draws):")
print(size_summary[['rank (0 = largest)', 'n_draws', 'median', 'iqr_low', 'iqr_high']]
      .to_string(index=False))
print("\nReal data (29 analytic participants, 131 stress episodes):")
print(real_long[['cluster_rank', 'cluster_size']].drop_duplicates()
      .rename(columns={'cluster_rank': 'rank', 'cluster_size': 'n'})
      .to_string(index=False))

zscore_summary = (
    mc_clusters
    .groupby(['cluster_rank', 'metric'])['mean_z']
    .agg(n_draws='count',
         null_mean='mean',
         null_median='median',
         null_iqr_low =lambda s: s.quantile(0.25),
         null_iqr_high=lambda s: s.quantile(0.75))
    .reset_index()
)

print("\n3. CLUSTER MEAN Z-SCORES: NULL vs REAL")
print("-" * 80)
for rank in sorted(real_long['cluster_rank'].unique()):
    print(f"\nRank {rank}:")
    null_rank = zscore_summary[zscore_summary['cluster_rank'] == rank][
        ['metric', 'null_mean', 'null_median', 'null_iqr_low', 'null_iqr_high']
    ].rename(columns={'null_iqr_low': 'null_Q1', 'null_iqr_high': 'null_Q3'})
    real_rank = real_long[real_long['cluster_rank'] == rank][
        ['metric', 'real_mean_z']
    ].rename(columns={'real_mean_z': 'real_mean'})

    comparison = null_rank.merge(real_rank, on='metric')
    print(comparison.to_string(index=False))

### Single random-window draw (Figure S8)


In [ ]:
# =========================================================================
# Figure S8 — single random-window draw (negative control)
#
# Sanity check that the Activation/Recovery structure is specific to
# stress-aligned windows. We sample non-stress windows for each of the 29
# analytic participants, run the same PCA -> GMM -> UMAP pipeline at k = 2,
# and confirm the resulting clusters lack the coordinated opposing structure
# seen in the real data.
#
# Steps:
#   1) For each subject, sample N pseudo-events (N = number of real S-STAI
#      events for that subject), each >= MIN_GAP_DAYS away from any real
#      stress date.
#   2) Average each metric in a +/- WINDOW_DAYS window around each pseudo
#      event and z-score against the same per-participant low-stress
#      baseline used in the primary pipeline.
#   3) Standardize -> PCA (>= 80% variance) -> GMM (k = 2) -> UMAP.
# =========================================================================

# ---- Parameters ---------------------------------------------------------
MIN_GAP_DAYS = 5
WINDOW_DAYS  = 5   # = 5, as selected previously

z_cols_names = [
    'deep_mean_hr_z',
    'deep_RMSSD_mean_z',
    'deep_LF_HF_Ratio_mean_z',
    'Time_asleep_z',
    'deep_sleep_breathing_rate_z',
    'Total_Number_of_Steps_z',
]
raw_signals = [c.replace('_z', '') for c in z_cols_names]

# Restrict to the 29 analytic participants from the primary PCA -> GMM fit
analytic_ids = set(big_df_clust['Global_Deidentified'].unique())
print(f"Restricting to {len(analytic_ids)} analytic participants.")

# Per-subject baseline lookup: (id, metric) -> {mean, std}
baseline_lookup = (
    baseline_df
    .drop_duplicates(subset=['id', 'metric'])
    .set_index(['id', 'metric'])[['mean', 'std']]
    .to_dict('index')
)

# Real stress dates per subject
real_stress_dates = (
    df_anxiety_min_2_samples
    .groupby('Global_Deidentified')['Date']
    .apply(list)
    .to_dict()
)

# ---- 1. Sample random non-stress dates per subject ---------------------
pseudo_rows = []
rng = np.random.default_rng(RANDOM_SEED)

for sid in analytic_ids:
    subj_dates = pd.to_datetime(
        full_df.loc[full_df['Global_Deidentified'] == sid, 'Date']
        .dropna().unique()
    ).normalize()

    if len(subj_dates) == 0:
        continue

    # Forbid days within MIN_GAP_DAYS of any real stress date
    forbidden = set()
    for d in real_stress_dates.get(sid, []):
        d0 = pd.Timestamp(d).normalize()
        for offset in range(-MIN_GAP_DAYS, MIN_GAP_DAYS + 1):
            forbidden.add(d0 + pd.Timedelta(days=offset))

    candidates = [d for d in subj_dates if d not in forbidden]
    n_needed   = len(real_stress_dates.get(sid, []))

    if n_needed == 0 or len(candidates) < n_needed:
        continue

    chosen = rng.choice(len(candidates), size=n_needed, replace=False)
    for j in chosen:
        pseudo_rows.append({'Global_Deidentified': sid,
                            'pseudo_date': candidates[j]})

df_pseudo = pd.DataFrame(pseudo_rows)
print(f"Generated {len(df_pseudo)} random pseudo-events across "
      f"{df_pseudo['Global_Deidentified'].nunique()} subjects.")


# ---- 2. Compute z-scored physiology around each pseudo-event ------------
records = []
for _, row in df_pseudo.iterrows():
    sid    = row['Global_Deidentified']
    p_date = pd.Timestamp(row['pseudo_date'])
    start  = p_date - pd.Timedelta(days=WINDOW_DAYS)
    end    = p_date + pd.Timedelta(days=WINDOW_DAYS)

    win = full_df[
        (full_df['Global_Deidentified'] == sid)
        & (full_df['Date'] >= start)
        & (full_df['Date'] <= end)
    ]

    out = {'Global_Deidentified': sid}
    for metric in raw_signals:
        avg = win[metric].mean() if len(win) > 0 else np.nan
        info = baseline_lookup.get((sid, metric))

        if info and not np.isnan(avg) and info['std'] and not np.isnan(info['std']):
            out[f'{metric}_z'] = (avg - info['mean']) / info['std']
        else:
            out[f'{metric}_z'] = np.nan

    records.append(out)

df_pseudo_z = pd.DataFrame(records).dropna(subset=z_cols_names)
print(f"Random windows with complete features: {len(df_pseudo_z)}")

# ---- 3. Standardize, PCA, GMM (k = 2) ----------------------------------
X = df_pseudo_z[z_cols_names].values
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=min(6, X_scaled.shape[1]), random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled)
cum_var = pca.explained_variance_ratio_.cumsum()
n_pcs = max(2, int(np.searchsorted(cum_var, 0.80)) + 1)
X_clust = X_pca[:, :n_pcs]
print(f"Using {n_pcs} PCs (cumulative variance {cum_var[n_pcs-1]:.2f})")

# k = 2 to match the two-cluster radar comparison in Figure S8
gmm = GaussianMixture(n_components=2, covariance_type='full',
                      random_state=RANDOM_SEED).fit(X_clust)
df_pseudo_z['cluster'] = gmm.predict(X_clust)

cluster_counts = df_pseudo_z['cluster'].value_counts().sort_index()
print("\nRandom-window cluster counts:")
print(cluster_counts)

# Cluster profiles in the original z-space
cluster_profiles = df_pseudo_z.groupby('cluster')[z_cols_names].mean()
print("\nCluster profiles (mean z-scores per metric):")
print(cluster_profiles.round(3))

# ---- 4. UMAP visualization ---------------------------------------------
X_umap = umap.UMAP(n_neighbors=10, min_dist=0.1,
                   random_state=RANDOM_SEED).fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(X_umap[:, 0], X_umap[:, 1],
                     c=df_pseudo_z['cluster'], cmap='tab10',
                     s=40, alpha=0.7)
ax.set_title("UMAP of random non-stress windows (k = 2)")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

handles, _ = scatter.legend_elements()
ax.legend(handles,
          [f"Cluster {i} (n={cluster_counts.get(i, 0)})" for i in range(2)],
          title="Random cluster")
plt.tight_layout()
plt.show()


# ---- 5. Radar plot of cluster-mean z-scores ----------------------------
pretty_labels = [
    "Deep-sleep HR",
    "Deep-sleep RMSSD",
    "Deep-sleep LF/HF",
    "Sleep duration",
    "Deep-sleep BR",
    "Daily steps",
]

angles = np.linspace(0, 2 * np.pi, len(z_cols_names), endpoint=False)
angles = np.concatenate([angles, [angles[0]]])

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
cluster_colors = ['tab:blue', 'tab:orange']

for k in cluster_profiles.index:
    vals = np.concatenate([cluster_profiles.loc[k].values,
                           [cluster_profiles.loc[k].values[0]]])
    ax.plot(angles, vals, linewidth=2, color=cluster_colors[k], alpha=0.9,
            label=f"Cluster {k} (n={cluster_counts.get(k, 0)})")
    ax.fill(angles, vals, color=cluster_colors[k], alpha=0.2)

ax.plot(angles, [0] * len(angles), '--', color='gray',
        linewidth=1.2, alpha=0.8)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(pretty_labels, fontsize=11)
ax.tick_params(axis='x', pad=32)
ax.set_ylim(-1.5, 1.5)
ax.set_yticks([-1, -0.5, 0, 0.5, 1])
ax.legend(title="Random cluster", loc='upper center',
          bbox_to_anchor=(0.5, -0.08), frameon=False, fontsize=11)
plt.tight_layout()
plt.show()

### Distribution comparison: stress episodes vs. baseline vs. random windows

In [ ]:
# =========================================================================
# Distribution comparison across three window types
#
# For each of the 6 metrics, plot and formally test the distribution of
# z-scores across:
#   (1) Stress episodes  -- from big_df_clust (non-baseline S-STAI windows)
#   (2) Baseline windows -- +/-5-day window around each participant's lowest S-STAI
#   (3) Random survey-distal windows -- same sampling as Figure S8 negative control
#
# Tests whether stress episodes are physiologically distinct from both
# baseline and random windows, and whether baseline and random windows
# overlap (validating the survey-distal assumption in the negative control).
#
# Variables confirmed in scope by this point:
#   big_df_clust, z_cols, baseline_lookup, full_df,
#   df_anxiety_min_2_samples, signals, window_days, RANDOM_SEED,
#   analytic_ids, real_stress_dates
# =========================================================================
from scipy.stats import gaussian_kde, kruskal as kruskal_wallis

# z_cols_names == z_cols (both refer to the same 6 z-score columns)
z_cols_names = z_cols

pretty_metric_names = {
    'deep_mean_hr_z':              'Deep sleep HR',
    'deep_RMSSD_mean_z':           'Deep sleep HRV (RMSSD)',
    'deep_LF_HF_Ratio_mean_z':     'Deep sleep LF/HF',
    'Time_asleep_z':               'Sleep duration',
    'deep_sleep_breathing_rate_z': 'Deep sleep BR',
    'Total_Number_of_Steps_z':     'Daily steps',
}

# -----------------------------------------------------------------------
# Window type 1: Stress episodes (already computed in big_df_clust)
# -----------------------------------------------------------------------
stress_z = big_df_clust[z_cols_names].copy()
stress_z['window_type'] = 'Stress episodes'

# -----------------------------------------------------------------------
# Window type 2: Baseline windows
# Extract wearable data from the +/-5-day window around each participant's
# lowest S-STAI survey and z-score using the same baseline_lookup.
# -----------------------------------------------------------------------
baseline_z_rows = []

for sid in sorted(big_df_clust['Global_Deidentified'].unique()):
    subj_stress = df_anxiety_min_2_samples[
        df_anxiety_min_2_samples['Global_Deidentified'] == sid
    ].copy()
    subj_stress['Date'] = pd.to_datetime(subj_stress['Date'])

    if subj_stress.empty:
        continue

    min_idx = subj_stress['stai_stress'].idxmin()
    baseline_date = subj_stress.loc[min_idx, 'Date']

    row = {'window_type': 'Baseline windows'}
    for metric, z_col in zip(signals, z_cols_names):
        info = baseline_lookup.get((sid, metric))
        if not info or pd.isna(info['std']) or info['std'] == 0:
            row[z_col] = np.nan
            continue
        win = full_df[
            (full_df['Global_Deidentified'] == sid) &
            (full_df['Date'] >= baseline_date - pd.Timedelta(days=window_days)) &
            (full_df['Date'] <= baseline_date + pd.Timedelta(days=window_days))
        ]
        raw_val = win[metric].mean() if (metric in win.columns and not win.empty) else np.nan
        row[z_col] = (raw_val - info['mean']) / info['std'] if not pd.isna(raw_val) else np.nan
    baseline_z_rows.append(row)

baseline_z = pd.DataFrame(baseline_z_rows)

# -----------------------------------------------------------------------
# Window type 3: Random survey-distal windows (one draw, matched n)
# Reuses analytic_ids, real_stress_dates, baseline_lookup from Cell 74.
# -----------------------------------------------------------------------
rng_dist = np.random.default_rng(RANDOM_SEED + 9999)
random_z_rows = []

for sid in sorted(analytic_ids):
    if sid not in real_stress_dates:
        continue
    stress_dates_sid = [pd.Timestamp(d).normalize() for d in real_stress_dates[sid]]
    n_needed = len(stress_dates_sid)
    forbidden = set()
    for d in stress_dates_sid:
        for offset in range(-window_days, window_days + 1):
            forbidden.add(d + pd.Timedelta(days=offset))
    subj_dates = pd.to_datetime(
        full_df.loc[full_df['Global_Deidentified'] == sid, 'Date']
        .dropna().unique()
    ).normalize()
    candidates = [d for d in subj_dates if d not in forbidden]
    if len(candidates) < n_needed:
        continue
    chosen_idx = rng_dist.choice(len(candidates), size=n_needed, replace=False)
    for idx in chosen_idx:
        center = candidates[idx]
        row = {'window_type': 'Random windows'}
        win = full_df[
            (full_df['Global_Deidentified'] == sid) &
            (full_df['Date'] >= center - pd.Timedelta(days=window_days)) &
            (full_df['Date'] <= center + pd.Timedelta(days=window_days))
        ]
        for metric, z_col in zip(signals, z_cols_names):
            info = baseline_lookup.get((sid, metric))
            if not info or pd.isna(info['std']) or info['std'] == 0:
                row[z_col] = np.nan
                continue
            raw_val = win[metric].mean() if (metric in win.columns and not win.empty) else np.nan
            row[z_col] = ((raw_val - info['mean']) / info['std']
                          if not pd.isna(raw_val) else np.nan)
        random_z_rows.append(row)

random_z = pd.DataFrame(random_z_rows)

# -----------------------------------------------------------------------
# Combine and plot distributions
# -----------------------------------------------------------------------
dist_df = pd.concat([stress_z, baseline_z, random_z], ignore_index=True)

window_colors = {
    'Stress episodes':  'steelblue',
    'Baseline windows': 'forestgreen',
    'Random windows':   'darkorange',
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (z_col, metric_name) in enumerate(pretty_metric_names.items()):
    ax = axes[i]
    for wtype, color in window_colors.items():
        vals = dist_df[dist_df['window_type'] == wtype][z_col].dropna()
        if vals.empty:
            continue
        ax.hist(vals, bins=20, alpha=0.45, color=color,
                label=f"{wtype} (n={len(vals)})", density=True)
        if len(vals) > 5:
            kde_x = np.linspace(vals.min() - 0.5, vals.max() + 0.5, 200)
            kde = gaussian_kde(vals)
            ax.plot(kde_x, kde(kde_x), color=color, linewidth=2)
    ax.axvline(x=0, color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_title(metric_name, fontsize=12)
    ax.set_xlabel('Z-score (relative to low-stress baseline)')
    ax.set_ylabel('Density')
    if i == 0:
        ax.legend(fontsize=9)

plt.suptitle(
    'Physiological z-score distributions:\n'
    'Stress episodes vs. Baseline windows vs. Random windows',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()

# Formal Kruskal-Wallis test across three window types per metric
print("\nKruskal-Wallis tests across three window types per metric:")
print(f"{'Metric':<30} {'H-stat':>8} {'p-value':>10}")
print("-" * 52)
for z_col, metric_name in pretty_metric_names.items():
    groups = [
        dist_df[dist_df['window_type'] == wt][z_col].dropna().values
        for wt in ['Stress episodes', 'Baseline windows', 'Random windows']
    ]
    groups = [g for g in groups if len(g) > 1]
    if len(groups) < 2:
        continue
    h, p = kruskal_wallis(*groups)
    print(f"{metric_name:<30} {h:>8.3f} {p:>10.4f}")


### 4.6 — LLM Activation Vs. Recovery


In [ ]:
# Mixed-effects models: outcome ~ Activation + (1 | subject)
# Uses statsmodels MixedLM (random intercept per participant).


signals = [
    "deep_mean_hr",
    "deep_RMSSD_mean",
    "deep_LF_HF_Ratio_mean",
    "Time_asleep",
    "deep_sleep_breathing_rate",
    "Total_Number_of_Steps",
]

df = big_df_clust.copy()

baseline_sd_wide = (
    baseline_df
    .dropna(subset=["std"])
    .drop_duplicates(subset=["id", "metric"])
    .pivot(index="id", columns="metric", values="std")
    .rename(columns=lambda m: f"{m}_baseline_std")
    .reset_index()
    .rename(columns={"id": "Global_Deidentified"})
)

df = df.merge(baseline_sd_wide, on="Global_Deidentified", how="left")

for m in signals:
    z_col = f"{m}_z"
    sd_col = f"{m}_baseline_std"
    rawdev_col = f"{m}_rawdev"
    df[rawdev_col] = df[z_col] * df[sd_col]

# -----------------------------
# 1) Fit MixedLM per metric
# -----------------------------
df["Activation"] = (df["cluster"] == 1).astype(int)

outcome_cols = [f"{m}_rawdev" for m in signals] if run_rawdev else [f"{m}_z" for m in signals]

import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

results = []

for y in outcome_cols:
    d = df[["Global_Deidentified", "Activation", y]].dropna().copy()

    n_samples = int(len(d))
    n_subj = int(d["Global_Deidentified"].nunique())

    # Need at least 2 subjects and some variability
    if n_subj < 2 or d["Activation"].nunique() < 2:
        results.append({
            "outcome": y,
            "n_subjects": n_subj,
            "n_samples": n_samples,
            "beta_activation": None,
            "se": None,
            "ci95_low": None,
            "ci95_high": None,
            "p_value": None,
            "converged": False,
            "note": "insufficient data/variation",
        })
        continue

    # Random intercept model: y ~ Activation + (1 | subject)
    try:
        md = smf.mixedlm(f"{y} ~ Activation", d, groups=d["Global_Deidentified"], re_formula="1")
        m = md.fit(reml=True, method="lbfgs", maxiter=200, disp=False)

        beta = float(m.params.get("Activation", float("nan")))
        se = float(m.bse.get("Activation", float("nan")))
        p = float(m.pvalues.get("Activation", float("nan")))

        ci_low = beta - 1.96 * se
        ci_high = beta + 1.96 * se

        results.append({
            "outcome": y,
            "n_subjects": n_subj,
            "n_samples": n_samples,
            "beta_activation": beta,     # Activation minus Recovery
            "se": se,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "p_value": p,
            "converged": bool(getattr(m, "converged", True)),
            "note": "",
        })

    except Exception as e:
        results.append({
            "outcome": y,
            "n_subjects": n_subj,
            "n_samples": n_samples,
            "beta_activation": None,
            "se": None,
            "ci95_low": None,
            "ci95_high": None,
            "p_value": None,
            "converged": False,
            "note": f"fit error: {type(e).__name__}",
        })

res_df = pd.DataFrame(results)

# -----------------------------
# 2) Multiple-testing correction across metrics (FDR)
# -----------------------------
mask = res_df["p_value"].notna()
if mask.sum() > 0:
    rej, p_fdr, _, _ = multipletests(res_df.loc[mask, "p_value"].values, alpha=0.05, method="fdr_bh")
    res_df.loc[mask, "p_value_fdr"] = p_fdr
    res_df.loc[mask, "sig_fdr_0p05"] = rej
else:
    res_df["p_value_fdr"] = None
    res_df["sig_fdr_0p05"] = None

# Nice label
label_map = {
    "deep_mean_hr_z": "Deep sleep HR (z)",
    "deep_RMSSD_mean_z": "Deep sleep HRV RMSSD (z)",
    "deep_LF_HF_Ratio_mean_z": "Deep sleep LF/HF (z)",
    "Time_asleep_z": "Sleep duration (z)",
    "deep_sleep_breathing_rate_z": "Deep sleep BR (z)",
    "Total_Number_of_Steps_z": "Daily steps (z)",
    "deep_mean_hr_rawdev": "Deep sleep HR (raw dev)",
    "deep_RMSSD_mean_rawdev": "Deep sleep HRV RMSSD (raw dev)",
    "deep_LF_HF_Ratio_mean_rawdev": "Deep sleep LF/HF (raw dev)",
    "Time_asleep_rawdev": "Sleep duration (raw dev)",
    "deep_sleep_breathing_rate_rawdev": "Deep sleep BR (raw dev)",
    "Total_Number_of_Steps_rawdev": "Daily steps (raw dev)",
}

res_df["metric"] = res_df["outcome"].map(lambda s: label_map.get(s, s))

# Round for reporting
for c in ["beta_activation", "se", "ci95_low", "ci95_high", "p_value", "p_value_fdr"]:
    if c in res_df.columns:
        res_df[c] = pd.to_numeric(res_df[c], errors="coerce").round(4)

res_df = res_df[[
    "metric", "n_subjects", "n_samples",
    "beta_activation", "ci95_low", "ci95_high",
    "p_value", "p_value_fdr", "sig_fdr_0p05",
    "converged", "note"
]].sort_values("metric")

res_df

### Recovery vs. baseline windows — mixed-effects comparison

In [ ]:
# =========================================================================
# Recovery vs. Baseline mixed-effects comparison
#
# Tests whether Recovery-mode episodes (cluster == 0) are physiologically
# distinguishable from each participant's low-stress baseline windows.
#
# Model: Z_ij = b0 + u_i0 + (b1 + u_i1) * is_recovery_ij + e_ij
#   is_recovery = 1 for Recovery episodes, 0 for baseline windows
#
# Interpretation:
#   b ~ 0, q > 0.05: Recovery is physiologically indistinguishable from
#     the low-stress baseline (consistent with Recovery = normal physiology).
#   b != 0, q < 0.05: Recovery is a genuine stress-associated physiological
#     state distinct from baseline (stronger biological claim).
#
# Variables confirmed in scope by this point:
#   big_df_clust, z_cols / z_cols_names, signals, baseline_lookup,
#   full_df, df_anxiety_min_2_samples, window_days, smf, multipletests
#   pretty_metric_names (defined in distribution comparison cell above)
# =========================================================================

# Recovery episodes (cluster == 0)
recovery_episodes = big_df_clust[big_df_clust['cluster'] == 0].copy()
recovery_episodes['is_recovery'] = 1

# Build baseline window rows using the same z-scoring as the primary pipeline
baseline_lmm_rows = []

for sid in sorted(recovery_episodes['Global_Deidentified'].unique()):
    subj_stress = df_anxiety_min_2_samples[
        df_anxiety_min_2_samples['Global_Deidentified'] == sid
    ].copy()
    subj_stress['Date'] = pd.to_datetime(subj_stress['Date'])

    if subj_stress.empty:
        continue

    min_idx = subj_stress['stai_stress'].idxmin()
    baseline_date = subj_stress.loc[min_idx, 'Date']

    win = full_df[
        (full_df['Global_Deidentified'] == sid) &
        (full_df['Date'] >= baseline_date - pd.Timedelta(days=window_days)) &
        (full_df['Date'] <= baseline_date + pd.Timedelta(days=window_days))
    ]

    if win.empty:
        continue

    row = {'Global_Deidentified': sid, 'is_recovery': 0}
    for metric, z_col in zip(signals, z_cols_names):
        info = baseline_lookup.get((sid, metric))
        if not info or pd.isna(info['std']) or info['std'] == 0:
            row[z_col] = np.nan
            continue
        raw_val = win[metric].mean() if (metric in win.columns) else np.nan
        row[z_col] = ((raw_val - info['mean']) / info['std']
                      if not pd.isna(raw_val) else np.nan)
    baseline_lmm_rows.append(row)

baseline_lmm_df = pd.DataFrame(baseline_lmm_rows)

# Combine Recovery episodes and baseline windows
shared_cols = z_cols_names + ['Global_Deidentified', 'is_recovery']
lmm_input = pd.concat([
    recovery_episodes[shared_cols],
    baseline_lmm_df[shared_cols]
], ignore_index=True)

# Run mixed-effects model per metric
print("=" * 85)
print("Recovery episodes vs. Baseline windows -- mixed-effects models")
print("(b > 0: Recovery above baseline; b < 0: Recovery below baseline)")
print("=" * 85)
print(f"{'Metric':<35} {'N_obs':>6} {'Beta':>8} {'95% CI':>22} {'p-value':>10}")
print("-" * 85)

results_rb = []
for z_col, metric_name in pretty_metric_names.items():
    df_m = lmm_input[['Global_Deidentified', 'is_recovery', z_col]].dropna().copy()
    df_m = df_m.rename(columns={z_col: 'outcome'})

    if df_m['Global_Deidentified'].nunique() < 3:
        print(f"{metric_name:<35} -- too few subjects --")
        continue

    try:
        model = smf.mixedlm(
            "outcome ~ is_recovery",
            df_m,
            groups=df_m["Global_Deidentified"],
            re_formula="~is_recovery"
        ).fit(reml=True, method='lbfgs', disp=False)
    except Exception:
        # Fall back to random-intercept-only if random slope fails to converge
        try:
            model = smf.mixedlm(
                "outcome ~ is_recovery",
                df_m,
                groups=df_m["Global_Deidentified"],
                re_formula="1"
            ).fit(reml=True, method='lbfgs', disp=False)
        except Exception as e:
            print(f"{metric_name:<35} model failed: {e}")
            continue

    beta = float(model.fe_params.get('is_recovery', np.nan))
    ci   = (model.conf_int().loc['is_recovery'].values
            if 'is_recovery' in model.conf_int().index else [np.nan, np.nan])
    pval = float(model.pvalues.get('is_recovery', np.nan))
    n_obs = int(model.nobs)

    results_rb.append({
        'metric':  metric_name,
        'n_obs':   n_obs,
        'beta':    beta,
        'ci_low':  ci[0],
        'ci_high': ci[1],
        'p_value': pval,
    })
    print(f"{metric_name:<35} {n_obs:>6} {beta:>8.3f} "
          f"[{ci[0]:>7.3f}, {ci[1]:>7.3f}] {pval:>10.4f}")

# Benjamini-Hochberg FDR correction
results_rb_df = pd.DataFrame(results_rb)
if not results_rb_df.empty and results_rb_df['p_value'].notna().any():
    valid = results_rb_df['p_value'].notna()
    reject, q_vals, _, _ = multipletests(
        results_rb_df.loc[valid, 'p_value'], method='fdr_bh'
    )
    results_rb_df.loc[valid, 'q_value']     = q_vals
    results_rb_df.loc[valid, 'significant'] = reject

    print("\nAfter Benjamini-Hochberg FDR correction:")
    print(results_rb_df[['metric', 'beta', 'ci_low', 'ci_high',
                          'p_value', 'q_value', 'significant']]
          .round(4).to_string(index=False))

# Forest plot
if not results_rb_df.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    y_pos = range(len(results_rb_df))
    for i, row in results_rb_df.iterrows():
        sig = row.get('significant', False)
        color = 'royalblue' if sig else 'gray'
        ax.plot([row['ci_low'], row['ci_high']], [i, i], color=color, linewidth=2)
        ax.scatter(row['beta'], i, color=color, zorder=5, s=60)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(results_rb_df['metric'].tolist(), fontsize=11)
    ax.set_xlabel('Beta coefficient (Recovery vs. Baseline, z-scored)', fontsize=11)
    ax.set_title(
        'Recovery episodes vs. low-stress baseline windows\n'
        '(mixed-effects, BH-FDR corrected; blue = q < 0.05)',
        fontsize=11
    )
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

print("\nInterpretation:")
print("  b ~ 0, q > 0.05: Recovery is physiologically indistinguishable from")
print("  the low-stress baseline (consistent with Recovery = normal physiology).")
print("  b != 0, q < 0.05: Recovery is a genuine stress-associated physiological")
print("  state distinct from baseline (stronger biological claim for the paper).")


In [ ]:

# --------------------------------------------------------------------------
# 1) MAP LABELS + ORDER
# --------------------------------------------------------------------------
label_mapping = {
    "Deep sleep HR (z)": "Deep Sleep\n HR",
    "Deep sleep HRV RMSSD (z)": "Deep Sleep \nHRV (RMSSD)",
    "Sleep duration (z)": "Total Sleep \nTime",
    "Daily steps (z)": "Daily Steps",
    "Deep sleep LF/HF (z)": "Deep Sleep \nHRV (LF/HF Ratio)",
    "Deep sleep BR (z)": "Deep Sleep \nResp. Rate",
}

metric_order = [
    "Daily steps (z)",
    "Deep sleep HR (z)",
    "Deep sleep HRV RMSSD (z)",
    "Sleep duration (z)",
    "Deep sleep LF/HF (z)",
    "Deep sleep BR (z)",
]

# --------------------------------------------------------------------------
# 2) PREPARE THE DATA
# --------------------------------------------------------------------------
plot_df = res_df.copy()

plot_df = plot_df[plot_df["metric"].isin(metric_order)].copy()
plot_df["metric"] = pd.Categorical(plot_df["metric"], categories=metric_order, ordered=True)
plot_df = plot_df.sort_values("metric", ascending=False).reset_index(drop=True)

# Safe fallback: convert to string (avoids categorical assignment issues)
plot_df["Nice_Label"] = plot_df["metric"].map(label_mapping)
plot_df["Nice_Label"] = plot_df["Nice_Label"].where(plot_df["Nice_Label"].notna(), plot_df["metric"].astype(str))

plot_df["Beta"] = pd.to_numeric(plot_df["beta_activation"], errors="coerce")
plot_df["CI_lower"] = pd.to_numeric(plot_df["ci95_low"], errors="coerce")
plot_df["CI_upper"] = pd.to_numeric(plot_df["ci95_high"], errors="coerce")
plot_df["p_value"] = pd.to_numeric(plot_df["p_value"], errors="coerce")

# Prefer FDR if available; else use raw p
if "p_value_fdr" in plot_df.columns:
    plot_df["p_fdr"] = pd.to_numeric(plot_df["p_value_fdr"], errors="coerce")
else:
    plot_df["p_fdr"] = plot_df["p_value"]

plot_df["is_sig"] = plot_df["p_fdr"].notna() & (plot_df["p_fdr"] < 0.05)

# --------------------------------------------------------------------------
# 3) CREATE THE PLOT (same style)
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))

y_pos = list(range(len(plot_df)))

xerr_left = (plot_df["Beta"] - plot_df["CI_lower"]).values
xerr_right = (plot_df["CI_upper"] - plot_df["Beta"]).values

ax.errorbar(
    x=plot_df["Beta"].values,
    y=y_pos,
    xerr=[xerr_left, xerr_right],
    fmt="o",
    color="dimgrey",
    ecolor="dimgrey",
    capsize=4,
    elinewidth=2,
    markeredgewidth=2,
    markersize=8,
    linestyle="none",
)

# Overlay significant points in black
sig_idx = plot_df.index[plot_df["is_sig"]].tolist()
if len(sig_idx) > 0:
    ax.scatter(
        plot_df.loc[sig_idx, "Beta"].values,
        sig_idx,
        s=8**2,
        color="black",
        edgecolors="black",
        linewidths=2,
        zorder=3,
    )

# --------------------------------------------------------------------------
# 4) STYLING
# --------------------------------------------------------------------------
ax.axvline(x=0, color="#E74C3C", linestyle="--", linewidth=1.5, alpha=0.8)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["Nice_Label"].values, fontsize=12)

ax.set_xlabel("Standardarized Beta Coefficient (Z score)", fontsize=12, fontweight="bold")

ax.grid(axis="x", linestyle=":", alpha=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

# --------------------------------------------------------------------------
# 5) P-values on the right (q if available; else p)
# --------------------------------------------------------------------------
finite_ci = pd.concat([plot_df["CI_lower"], plot_df["CI_upper"]], axis=0).dropna()
text_x_pos = float(finite_ci.max()) + 0.06 if len(finite_ci) else 0.08

for i, (p_raw, q_val, is_sig) in enumerate(zip(plot_df["p_value"], plot_df["p_fdr"], plot_df["is_sig"])):

    if pd.isna(q_val) and pd.isna(p_raw):
        txt = "p=N/A"
        color = "grey"
    else:
        if pd.isna(q_val) or ("p_value_fdr" not in plot_df.columns):
            # no FDR available
            txt = f"p={p_raw:.3f}" if not pd.isna(p_raw) else "p=N/A"
        else:
            # FDR available
            if q_val < 0.001:
                txt = "q<0.001"
            else:
                txt = f"q={q_val:.3f}"
            # if not pd.isna(p_raw):
            #     txt += f" (p={p_raw:.3f})"

        color = "black"  # keep your style (no grey text)

    ax.text(text_x_pos, i, txt, va="center", fontsize=10, color=color)

plt.tight_layout()
plt.show()